# IAAA Brain CT Triage Challenge — Master Notebook

Full pipeline: MIL baseline (Sections 8-13) → pivot to segmentation
(Section 14) → segmentation v1/v2 (Sections 15-20) → IntermediatesModel +
MLS regression-to-mean investigation (Section 21) → end-to-end hybrid
validation (Section 22) → packaging & deployment (Sections 23-25).

**Reorganization pass complete: 25/25 sections restructured.**

**Official leaderboard: f1_macro=0.8297 (restored and confirmed genuine).**

**Resolved during this pass:**
- ~~Section 21.7~~ — fair comparison: no evidence v2's weighted MLS loss
  improves on v1 (p=0.220 MAE, p=0.749 AUC). v1 remains deployed.
- ~~Section 23.3~~ — deployment choice (v1 for IntermediatesModel)
  confirmed correct; comment corrected to cite the real justification.
- ~~Section 25.5~~ — a diagnostic canary submission (f1_macro=0.218) proved
  the evaluator has no caching; a freshly-verified true-v2 resubmission
  then scored f1_macro=0.8297, matching v1 almost exactly. **The
  connected-component filter is a genuine no-op on this hidden test set**
  -- the two earlier 0.8204 scores were stale/buggy code, as first
  suspected. v2 (segmentation filter) remains deployed going forward.

**Still open:**
- **Section 22** — `load_inference_slice`-equivalent DICOM-loading logic
  duplicated across three call sites; consolidation into
  `src/preprocessing.py` proposed but not implemented.


# IAAA Brain CT Triage Challenge

Main project notebook — organized and cleaned. Each section is marked with a numbered Markdown header. See the project README for the full pipeline overview (MIL baseline → segmentation-based hybrid pipeline).


## 1. Setup: Mount Drive & Install Dependencies

**Objective:** mount Google Drive (where checkpoints, logs, and the extracted
competition data live across Colab sessions) and install the DICOM-handling
packages this project needs beyond Colab's defaults.

**Why these specific packages:**
- `pydicom` — reads DICOM files (pixel data + metadata headers).
- `gdcm`, `pylibjpeg`, `pylibjpeg-libjpeg` — decoders for compressed DICOM
  transfer syntaxes (e.g. JPEG-Lossless, JPEG2000). Some CT scanners export
  compressed pixel data; without these, `pydicom` can read the header but
  raises on `.pixel_array` for those series. Installing them upfront avoids
  a silent subset of series failing deep into a training run.

**Bug fixed here:** the original notebook called `drive.mount(...)` twice —
once as an orphan cell before this section's header, then again inside the
main setup cell. Harmless (idempotent) but redundant; merged into one call.


In [ ]:
# Cell 1: environment setup -- mount Drive, install DICOM-handling deps, import core libraries.
from google.colab import drive
drive.mount("/content/drive")

!pip install -q gdcm pylibjpeg pylibjpeg-libjpeg torch
!pip install -q pydicom

import os
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydicom

print("pydicom version:", pydicom.__version__)


In [ ]:
# pydicom logs a WARNING-level message per file for several benign header
# quirks that are common in real-world clinical DICOM exports (e.g. missing
# optional tags). At dataset scale (thousands of files) this floods the
# Colab output; suppress below WARNING while keeping ERROR-level visible.
import logging

logging.getLogger("pydicom").setLevel(logging.ERROR)


## 2. Extract Competition Data

**Objective:** unpack the competition zip (mounted from Drive) onto Colab's
local disk, and resolve the exact paths to the DICOM series, annotations,
and metadata table.

**Why extract to local disk instead of reading straight from the mounted
Drive path:** Colab's Drive mount is a FUSE filesystem over the network --
fine for a handful of large sequential reads, but slow for thousands of
small, randomly-accessed files (exactly what per-slice DICOM training does).
Extracting once to `/content/data` (local SSD) up front avoids paying that
network round-trip cost on every file access during training.


In [ ]:
# Cell 2: Extract the competition data zip from Google Drive
# into the local Colab disk for fast I/O.
ZIP_PATH = "/content/drive/MyDrive/iaaa-contest-bct.zip"
LOCAL_DATA_ROOT = "/content/data"

if not os.path.isfile(ZIP_PATH):
    raise FileNotFoundError(f"Zip file not found at: {ZIP_PATH}")

os.makedirs(LOCAL_DATA_ROOT, exist_ok=True)

print(f"Extracting {ZIP_PATH} -> {LOCAL_DATA_ROOT} ...")
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    total_files = len(zf.namelist())
    print(f"Zip contains {total_files} entries.")
    zf.extractall(LOCAL_DATA_ROOT)

print("Extraction finished.")

In [ ]:
# The extraction root itself is /content/data; the real data lives one
# level below it, under "Data/" (capital D, matches the zip's internal layout).
DATA_ROOT = os.path.join(LOCAL_DATA_ROOT, "Data")

if not os.path.isdir(DATA_ROOT):
    # Defensive fallback: list what's actually there so we never guess blindly again.
    print("DATA_ROOT not found. Contents of LOCAL_DATA_ROOT:")
    print(os.listdir(LOCAL_DATA_ROOT))
    raise FileNotFoundError(f"Expected DATA_ROOT not found: {DATA_ROOT}")

dicom_dir = os.path.join(DATA_ROOT, "training")
annotations_dir = os.path.join(DATA_ROOT, "annotations")
metadata_path = os.path.join(DATA_ROOT, "training_df.pkl")

for label, path in [
    ("training (DICOM) dir", dicom_dir),
    ("annotations dir", annotations_dir),
    ("training_df.pkl", metadata_path),
]:
    exists = os.path.exists(path)
    print(f"{label}: {'FOUND' if exists else 'MISSING'} -> {path}")

series_ids = sorted(os.listdir(dicom_dir))
print(f"\nNumber of series folders under 'training/': {len(series_ids)}")
print("First 5 series IDs:", series_ids[:5])

## 3. Data Understanding: Load & Inspect Metadata Schema

**Objective:** before writing any modeling code, understand exactly what the
metadata table contains -- its granularity (per-slice, not per-series),
which column is the real join key to the DICOM folders on disk, and which
columns need type coercion before they can be trusted numerically.

**Why this matters for the project's four goals:** getting this wrong
silently (e.g. assuming per-series metadata when it's actually per-slice,
or trusting a column's dtype without checking it) is exactly the kind of
mistake that produces a model that trains "successfully" but learns the
wrong thing -- hard to catch later, expensive to diagnose after a full
training run.


In [ ]:
# Cell 4: Load the real metadata file and inspect its schema.
metadata_df = pd.read_pickle(metadata_path)

print("Shape:", metadata_df.shape)
print("\nColumn names and dtypes:")
print(metadata_df.dtypes)

print("\nFirst 3 rows:")
metadata_df.head(3)

In [ ]:
# Cell 5: Sanity-check that metadata rows can be joined to actual series folders.
# We don't yet know the exact ID column name, so first list all columns,
# then try the most likely candidates.
print("All columns:", metadata_df.columns.tolist())

likely_id_columns = [c for c in metadata_df.columns if "id" in c.lower() or "series" in c.lower()]
print("\nColumns that look like an identifier:", likely_id_columns)

series_ids_set = set(series_ids)
for col in likely_id_columns:
    overlap = metadata_df[col].astype(str).isin(series_ids_set).sum()
    print(f"Column '{col}': {overlap} / {len(metadata_df)} rows match a folder name in training/")

In [ ]:
# Investigate whether metadata is truly per-slice, and how
# triage_class/Area columns behave within a single series.
# (pydicom logging was already silenced in Section 1 -- removed the
# duplicate call that was here in the original notebook.)

# Pick one series and inspect all its rows.
sample_series_id = metadata_df["dicom_series.id"].iloc[0]
sample_rows = metadata_df[metadata_df["dicom_series.id"] == sample_series_id]

print(f"Series {sample_series_id}: {len(sample_rows)} rows (slices) in metadata")
print("\nUnique triage_class values within this series:", sample_rows["triage_class"].unique())
print("Unique AnyICH values within this series:", sample_rows["AnyICH"].unique())

print("\nArea columns summary for this series:")
area_cols = [c for c in metadata_df.columns if c.endswith("_Area")]
print(sample_rows[area_cols].describe())

print("\nMidlineShiftMM summary for this series:")
print(sample_rows["MidlineShiftMM"].describe())
print("Non-zero MidlineShiftMM count:", (sample_rows["MidlineShiftMM"] > 0).sum())

# Confirm row count matches actual number of DICOM files on disk for this series.
actual_dicom_count = len(os.listdir(os.path.join(dicom_dir, str(sample_series_id))))
print(f"\nRows in metadata for this series: {len(sample_rows)}")
print(f"Actual .dcm files on disk for this series: {actual_dicom_count}")

In [ ]:
# Cell 7: SliceThickness has dtype "object" (not float) — inspect why
# before trusting it in volume calculations.
print("Unique SliceThickness values (sample):")
print(metadata_df["dicom_series.SliceThickness"].unique()[:20])
print("\nAny non-numeric-looking values?")
non_numeric = metadata_df["dicom_series.SliceThickness"].apply(
    lambda x: not str(x).replace(".", "", 1).replace("-", "", 1).isdigit()
)
print(f"{non_numeric.sum()} / {len(metadata_df)} rows look non-numeric")
print(metadata_df.loc[non_numeric, "dicom_series.SliceThickness"].unique()[:10])

**Explaining the `SliceThickness` dtype finding above:** the column is
`dtype=object` even though every value looks numeric. This is a pydicom
quirk, not a data-quality problem: DICOM's Decimal String (DS) value
representation is returned by pydicom as `pydicom.valuerep.DSfloat` -- a
`float` subclass that preserves the original decimal-string formatting.
pandas stores a column of these as `object` dtype because it doesn't
recognize the subclass as a uniform native float type, even though every
element behaves like one.

**Practical consequence:** arithmetic on individual values works fine
(`DSfloat` behaves like `float`), but vectorized pandas operations that
require a native numeric dtype (e.g. `.mean()` on the whole column,
`.astype(np.float32)` for a tensor) can behave unexpectedly on an `object`
column. The codebase already handles this correctly downstream (see
`IntermediatesSliceDataset` in Section 21, which does an explicit
`float(row["dicom_series.SliceThickness"])` per row) -- documenting it here
so the reason is on record rather than being a rediscovered mystery later.


## 4. Official Triage Rule (verbatim from competition docs)

**Objective:** encode the competition's ground-truth decision rule -- the
function that maps seven imaging-derived "intermediates" (five hemorrhage
subtype volumes, skull-fracture probability, midline shift) to the final
triage class (0=Non-Urgent, 1=Urgent, 2=Critical). This is the single most
important piece of domain logic in the whole project: every model this
notebook trains exists to estimate these seven numbers accurately, because
this fixed rule -- not a learned classifier -- makes the final decision.

**Clinical background on the thresholds below** (for context on *why* these
specific numbers, not to imply they were derived from patient outcomes in
this project -- they come from the competition's own documentation):
- **EDH (Epidural Hematoma) ≥ 30 mL** and **SDH (Subdural Hematoma) ≥ 70 mL**
  as Critical/surgical thresholds echo the widely-cited Brain Trauma
  Foundation heuristic ("the 30cc rule") used in emergency neurosurgery to
  flag hematomas likely to need evacuation.
- **MLS (midline shift) ≥ 5mm** as a Critical threshold reflects the point
  at which shift is strongly associated with clinically significant raised
  intracranial pressure (see Section 21's markdown for the anatomy/physics
  of MLS itself).
- **Total hemorrhage volume ≥ 60 mL** is a general mass-effect threshold
  independent of *which* hemorrhage subtype is present.
- The **combination rules** (e.g. any ICH present *and* MLS ≥ 3mm *and*
  total volume ≥ 40mL → Critical) capture that moderate findings in
  multiple dimensions together can be as dangerous as one large finding
  alone -- the reason the model can't just threshold volume in isolation.

**Reading the rule's structure:** it's a staged decision list, checked in
order -- first every way to reach Critical (2), then every remaining way to
reach Urgent (1), and Non-Urgent (0) is the fall-through default. This
ordering matters: e.g. a series with `MLS_mm=6` (above the 5mm Critical MLS
threshold) is only Critical if hemorrhage *or* fracture is also present --
isolated midline shift with no other finding is treated as Urgent (1), not
Critical, by the `MLS_mm >= MLS_CRITICAL and not (has_ich or fracture_present)`
branch.


In [ ]:
# Cell 8: Official triage rule, copied verbatim from the competition's HTML documentation.
from typing import Any, Dict, Mapping

TRIAGE_REQUIRED_KEYS = {
    "V_EDH", "V_SDH", "V_IPH", "V_SAH", "V_IVH", "fracture_prob", "MLS_mm",
}


def validate_intermediates(intermediates: Mapping[str, Any]) -> Dict[str, float]:
    """Validate and normalize intermediate values for a single series."""
    missing = TRIAGE_REQUIRED_KEYS - intermediates.keys()
    extra = intermediates.keys() - TRIAGE_REQUIRED_KEYS
    if missing:
        raise ValueError(f"Missing keys in intermediates: {sorted(missing)}.")
    if extra:
        raise ValueError(f"Unexpected keys in intermediates: {sorted(extra)}.")
    cleaned: Dict[str, float] = {}
    for key in TRIAGE_REQUIRED_KEYS:
        cleaned[key] = float(intermediates[key])
    return cleaned


def triage_from_intermediates(intermediates: Mapping[str, Any]) -> int:
    """Compute triage class (0/1/2) from intermediate imaging primitives.

    Verbatim port of the official rule from the competition documentation.
    """
    vals = validate_intermediates(intermediates)

    V_EDH = max(0.0, vals["V_EDH"])
    V_SDH = max(0.0, vals["V_SDH"])
    V_IPH = max(0.0, vals["V_IPH"])
    V_SAH = max(0.0, vals["V_SAH"])
    V_IVH = max(0.0, vals["V_IVH"])
    MLS_mm = max(0.0, vals["MLS_mm"])
    fracture_prob = float(vals["fracture_prob"])

    total_vol = V_EDH + V_SDH + V_IPH + V_SAH + V_IVH

    EPS_VOLUME = 0.1
    EPS_MLS = 1.0
    MLS_CRITICAL = 5.0
    MLS_URGENT_LOW = 3.0
    EDH_CRIT = 30.0
    SDH_CRIT = 70.0
    IPH_CRIT = 70.0
    TOTAL_VOL_CRIT = 60.0
    COMBO_MLS = 3.0
    COMBO_VOL = 40.0
    FRAC_VOL_CRIT = 15.0
    FRACTURE_PRESENCE_THRESHOLD = 0.5

    has_ich = total_vol >= EPS_VOLUME
    mls_present = MLS_mm >= EPS_MLS
    fracture_present = fracture_prob >= FRACTURE_PRESENCE_THRESHOLD

    if MLS_mm >= MLS_CRITICAL and (has_ich or fracture_present):
        return 2
    if V_EDH >= EDH_CRIT:
        return 2
    if V_SDH >= SDH_CRIT:
        return 2
    if V_IPH >= IPH_CRIT:
        return 2
    if total_vol >= TOTAL_VOL_CRIT:
        return 2
    if has_ich and MLS_mm >= COMBO_MLS and total_vol >= COMBO_VOL:
        return 2
    if fracture_present and total_vol >= FRAC_VOL_CRIT:
        return 2

    if MLS_mm >= MLS_CRITICAL and not (has_ich or fracture_present):
        return 1
    if has_ich:
        return 1
    if MLS_URGENT_LOW <= MLS_mm < MLS_CRITICAL:
        return 1
    if fracture_present and total_vol < FRAC_VOL_CRIT:
        return 1
    if total_vol >= EPS_VOLUME and mls_present:
        return 1

    return 0

## 5. Aggregate Slice-Level Metadata into Series-Level Intermediates, and Validate

**Objective:** the metadata table is per-slice; the triage rule (Section 4)
needs per-series intermediates. This section builds the aggregation
(pixel-area sums → physical volumes in mL, using `PixelSpacing × SliceThickness`
for the voxel volume) and then checks the *implementation* of the rule
itself: does re-deriving `triage_class` from the raw metadata, through our
own code, reproduce the `triage_class` already present in the data?

**Why 100% accuracy below is expected, not impressive:** this check is
tautological by construction, not a model evaluation. `MidlineShiftMM`,
the hemorrhage `_Area` columns, and `triage_class` are all ground-truth
columns already present in the metadata -- there is no prediction involved
yet. A mismatch here would mean a *bug in our rule implementation or unit
conversion*, not a modeling problem. Getting 1.0/1.0 confirms Section 4's
code is a correct, bug-free port of the rule; it says nothing about any
model's real performance, which only appears once a model estimates these
intermediates from *images* instead of reading them from ground truth
(Sections 21 onward).


In [ ]:
# Cell 9: Aggregate per-slice metadata into series-level intermediates,
# then validate against the known triage_class for our sample series.
AREA_TO_VOLUME_COLUMNS = {
    "V_EDH": "EpiduralHemorrhage_Area",
    "V_SDH": "SubduralHemorrhage_Area",
    "V_IPH": "IntraparenchymalHemorrhage_Area",
    "V_SAH": "SubarachnoidHemorrhage_Area",
    "V_IVH": "IntraventricularHemorrhage_Area",
}


def aggregate_series_intermediates(series_rows: pd.DataFrame) -> dict:
    """Aggregate per-slice metadata rows of a single series into the
    official intermediate schema expected by triage_from_intermediates.

    Args:
        series_rows: All metadata rows (slices) belonging to one series.

    Returns:
        Dict with keys V_EDH, V_SDH, V_IPH, V_SAH, V_IVH, fracture_prob, MLS_mm.
    """
    pixel_spacing0 = series_rows["dicom_series.PixelSpacing0"].iloc[0]
    pixel_spacing1 = series_rows["dicom_series.PixelSpacing1"].iloc[0]
    slice_thickness_mm = float(series_rows["dicom_series.SliceThickness"].iloc[0])

    voxel_volume_mm3 = pixel_spacing0 * pixel_spacing1 * slice_thickness_mm

    intermediates = {}
    for target_key, area_col in AREA_TO_VOLUME_COLUMNS.items():
        total_pixels = series_rows[area_col].sum()
        volume_mm3 = total_pixels * voxel_volume_mm3
        intermediates[target_key] = volume_mm3 / 1000.0  # mm^3 -> mL

    intermediates["fracture_prob"] = float(series_rows["SkullFracture"].any())
    intermediates["MLS_mm"] = float(series_rows["MidlineShiftMM"].max())

    return intermediates


# Test on our known sample series (271208, known triage_class = 2)
sample_intermediates = aggregate_series_intermediates(sample_rows)
predicted_triage = triage_from_intermediates(sample_intermediates)
actual_triage = sample_rows["triage_class"].iloc[0]

print("Computed intermediates:", sample_intermediates)
print(f"\nPredicted triage_class: {predicted_triage}")
print(f"Actual triage_class:    {actual_triage}")
print(f"Match: {predicted_triage == actual_triage}")

In [ ]:
# Full validation -- aggregate every series and compare our computed
# triage against the ground-truth triage_class.
#
# NOTE (corrected from the original comment here, which called QWK "the
# competition's actual metric" -- that was wrong and later corrected
# elsewhere in the project once confirmed against the competition docs:
# the official metric is f1_macro, NOT QWK. Both are reported below so
# this section is internally consistent with that correction.)
from sklearn.metrics import cohen_kappa_score, accuracy_score, f1_score

results = []
errors = []

for series_id, series_rows in metadata_df.groupby("dicom_series.id"):
    try:
        intermediates = aggregate_series_intermediates(series_rows)
        predicted = triage_from_intermediates(intermediates)
        actual = int(series_rows["triage_class"].iloc[0])
        results.append({
            "series_id": series_id,
            "predicted_triage": predicted,
            "actual_triage": actual,
            **intermediates,
        })
    except Exception as exc:
        errors.append({"series_id": series_id, "error": str(exc)})

results_df = pd.DataFrame(results)

print(f"Successfully processed: {len(results_df)} / {metadata_df['dicom_series.id'].nunique()} series")
if errors:
    print(f"\nErrors encountered: {len(errors)}")
    for e in errors[:5]:
        print(e)

accuracy = accuracy_score(results_df["actual_triage"], results_df["predicted_triage"])
f1_macro = f1_score(results_df["actual_triage"], results_df["predicted_triage"], average="macro")
qwk = cohen_kappa_score(results_df["actual_triage"], results_df["predicted_triage"], weights="quadratic")

print(f"\nAccuracy: {accuracy:.4f}")
print(f"f1_macro (OFFICIAL competition metric): {f1_macro:.4f}")
print(f"Quadratic Weighted Kappa (reference only, NOT the official metric): {qwk:.4f}")

print("\nConfusion breakdown (actual vs predicted):")
print(pd.crosstab(results_df["actual_triage"], results_df["predicted_triage"],
                   rownames=["actual"], colnames=["predicted"]))

## 6. Patient-Grouped, Class-Stratified Cross-Validation Folds

**Objective:** build the 5-fold split used by every model trained in this
project (MIL, segmentation, IntermediatesModel), with two constraints
enforced simultaneously.

**Why grouping by patient matters (not just series):** 18 of the 338 series
belong to patients with more than one series in the dataset. If two series
from the same patient land in different folds, validation performance is
inflated by leakage -- the model can partially "recognize" that patient's
anatomy/pathology pattern from training, rather than genuinely generalizing.
`StratifiedGroupKFold`'s `groups=patient_id` argument guarantees a patient's
series all stay together in one fold.

**Why stratification matters:** `triage_class` is imbalanced
(~49% Non-Urgent / 18% Urgent / 33% Critical overall). Without stratifying,
a plain `GroupKFold` could by chance starve one fold of the minority
"Urgent" class, making that fold's metrics noisy and non-comparable to the
others.

**A limitation to keep in mind for later sections:** even with
stratification, the class balance per fold below still varies more than
you might expect (Urgent ranges from 13.0% in fold 4 to 26.4% in fold 0).
This is a direct consequence of grouping by patient on a small dataset (338
series, only 60 total Urgent) -- stratification can only balance folds
*within* what the patient-grouping constraint allows, and with 18 series
tied up in multi-series patients, perfect 3-way balance across 5 folds
isn't achievable. **This is worth remembering**: it's the same root cause
behind the high per-fold variance found later in Section 21.7 (MLS MAE and
fracture AUC deltas swinging widely fold-to-fold) -- small, unevenly-balanced
folds amplify noise in any per-fold metric.


In [ ]:
# Cell 11: Build the series-level targets table with patient-grouped,
# class-stratified cross-validation folds.
#
# Why StratifiedGroupKFold:
#   - group=patient_id: some patients have multiple series (we found 15 earlier
#     via the DICOM PatientID field); a patient must never appear in both
#     train and validation for the same fold, or the split leaks information.
#   - stratify=triage_class: the class distribution is imbalanced
#     (~49% / 18% / 33%), so a plain GroupKFold could accidentally starve
#     one fold of the "Urgent" class.
from sklearn.model_selection import StratifiedGroupKFold

N_SPLITS = 5
RANDOM_STATE = 42

# --- Step 1: build one row per series with the 7 official intermediate targets,
# plus patient_id (for grouping) and triage_class (for stratification / QC). ---
series_records = []
for series_id, series_rows in metadata_df.groupby("dicom_series.id"):
    intermediates = aggregate_series_intermediates(series_rows)
    series_records.append({
        "series_id": series_id,
        "patient_id": series_rows["dicom_series.PatientID"].iloc[0],
        "triage_class": int(series_rows["triage_class"].iloc[0]),
        **intermediates,
    })

series_targets_df = pd.DataFrame(series_records)

print(f"Built series_targets_df: {series_targets_df.shape[0]} rows, "
      f"{series_targets_df.shape[1]} columns")
print(f"Unique patients: {series_targets_df['patient_id'].nunique()}")
print(f"Series with a repeated patient_id: "
      f"{series_targets_df.shape[0] - series_targets_df['patient_id'].nunique()}")

# --- Step 2: assign folds. ---
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

series_targets_df["fold"] = -1
X_dummy = series_targets_df.index.values.reshape(-1, 1)
y = series_targets_df["triage_class"].values
groups = series_targets_df["patient_id"].values

for fold_idx, (_, val_idx) in enumerate(sgkf.split(X_dummy, y, groups)):
    series_targets_df.loc[val_idx, "fold"] = fold_idx

assert (series_targets_df["fold"] == -1).sum() == 0, "Some rows were not assigned a fold."

# --- Step 3: verify no patient leakage across folds. ---
leakage_found = False
for fold_idx in range(N_SPLITS):
    patients_in_fold = set(series_targets_df.loc[series_targets_df["fold"] == fold_idx, "patient_id"])
    patients_outside_fold = set(series_targets_df.loc[series_targets_df["fold"] != fold_idx, "patient_id"])
    overlap = patients_in_fold & patients_outside_fold
    if overlap:
        leakage_found = True
        print(f"LEAKAGE in fold {fold_idx}: {len(overlap)} patients also appear outside this fold")

print(f"\nPatient leakage across folds: {'FOUND -- FIX BEFORE PROCEEDING' if leakage_found else 'none (clean split)'}")

# --- Step 4: sanity-check class balance per fold. ---
print("\nClass distribution per fold (%):")
class_balance = (
    series_targets_df.groupby("fold")["triage_class"]
    .value_counts(normalize=True)
    .unstack()
    .mul(100)
    .round(1)
)
print(class_balance)

print("\nRow counts per fold:")
print(series_targets_df["fold"].value_counts().sort_index())

# --- Step 5: save to Drive. ---
OUTPUT_CSV_PATH = "/content/drive/MyDrive/iaaa-brain-ct-triage/data/processed/series_targets_with_folds.csv"
os.makedirs(os.path.dirname(OUTPUT_CSV_PATH), exist_ok=True)
series_targets_df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"\nSaved: {OUTPUT_CSV_PATH}")
series_targets_df.head()

## 7. `src/` Package Setup

From this point on, reusable logic (preprocessing, dataset, model, ...)
lives in `src/*.py` files on Drive rather than directly in the notebook.
`load_src.py` loads these files straight from disk (robust against
Colab/Drive module caching).

**Architecture rationale (why not just `import src.model` normally):**
Colab sessions can persist Python's module cache across a disconnect/
reconnect, or across re-running cells after editing a `.py` file on Drive
with `%%writefile`. A normal `import` can silently keep serving a stale,
cached version of a module after the underlying file changed -- a classic
source of "I fixed the bug but it's still happening" confusion. `load_src.py`
sidesteps this by loading modules directly from disk by path, deleting any
stale `sys.modules` entry first, so every cell run is guaranteed to reflect
the current file content on Drive.

**Notice the assertion-based verification at the end of the next cell:**
every module is checked for the specific class/function it's expected to
expose, immediately after loading. This turns a silent "wrong file loaded"
failure (which would otherwise only surface confusingly, deep inside a
training loop) into an immediate, clear `AssertionError` right where the
import happens -- fail fast, fail loud.


In [ ]:
import os
!pip install segmentation-models-pytorch --quiet
init_path = "/content/drive/MyDrive/iaaa-brain-ct-triage/src/__init__.py"
os.makedirs(os.path.dirname(init_path), exist_ok=True)
with open(init_path, "w") as f:
    pass
print("Ensured:", init_path)

In [ ]:
# Verify src/ files are correctly placed and load them robustly
# (bypassing sys.path / bytecode-cache flakiness) using load_src.py.
import os

PROJECT_ROOT = "/content/drive/MyDrive/iaaa-brain-ct-triage"
SRC_DIR = os.path.join(PROJECT_ROOT, "src")

print("src/ contents:")
for fname in sorted(os.listdir(SRC_DIR)):
    print(" ", fname)

loader_path = os.path.join(PROJECT_ROOT, "load_src.py")
print(f"\nload_src.py at project root: {'FOUND' if os.path.isfile(loader_path) else 'MISSING'}")

import sys
sys.path.append(PROJECT_ROOT)

if "load_src" in sys.modules:
    del sys.modules["load_src"]

from load_src import load_src_package
modules = load_src_package()
print("\nLoaded modules:", sorted(modules.keys()))

# --- Extract each module as a plain variable for convenience ---
preprocessing = modules["preprocessing"]
augmentation = modules["augmentation"]
dataset = modules["dataset"]
model = modules["model"]
losses = modules["losses"]
triage = modules["triage"]
metrics = modules["metrics"]
# Renamed from "checkpoint" -> "ckpt_utils": "checkpoint" is a very common name
# for a plain dict/torch.load() result in ad-hoc diagnostic cells, and any such
# cell run in this same notebook session would silently shadow the module
# (exactly what happened earlier). "ckpt_utils" is unlikely to collide.
ckpt_utils = modules["checkpoint"]
train_utils = modules["train_utils"]
rle = modules["rle"]
seg_dataset = modules["seg_dataset"]
seg_model = modules["seg_model"]
seg_losses = modules["seg_losses"]
volume_aggregation = modules["volume_aggregation"]

# --- Verify each module has the expected contents ---
import inspect
sig = inspect.signature(dataset.BrainCTSeriesDataset.__init__)
assert "augment" in sig.parameters and "augmentor" in sig.parameters, "Loaded an outdated dataset.py."
assert hasattr(model, "BrainCTMILModel"), "model.py loaded but BrainCTMILModel not found."
assert hasattr(losses, "BrainCTLoss"), "losses.py loaded but BrainCTLoss not found."
assert hasattr(triage, "triage_from_intermediates"), "triage.py loaded but function not found."
assert hasattr(metrics, "compute_metrics"), "metrics.py loaded but compute_metrics not found."
assert hasattr(ckpt_utils, "save_checkpoint") and hasattr(ckpt_utils, "load_checkpoint"), \
    "checkpoint.py loaded but functions not found."
assert hasattr(train_utils, "build_dataloaders") and hasattr(train_utils, "train_one_epoch"), \
    "train_utils.py loaded but functions not found."
assert hasattr(rle, "decode_rle_mask"), "rle.py loaded but function not found."
assert hasattr(seg_dataset, "SegmentationSliceDataset"), "seg_dataset.py loaded but class not found."
assert hasattr(seg_model, "HemorrhageSegModel"), "seg_model.py loaded but class not found."
assert hasattr(seg_losses, "SegmentationLoss"), "seg_losses.py loaded but class not found."
assert hasattr(volume_aggregation, "predict_series_volumes"), "volume_aggregation.py loaded but function not found."

print("\nPASS: all src/ modules loaded and verified successfully.")

## 8. MIL Pipeline — Dataset & Augmentation Sanity Checks

**What is Multiple Instance Learning (MIL), and why use it here?**

A CT *series* is a stack of ~20-60 2D slices, but only a **series-level**
label exists (the 7 intermediate targets) -- there is no per-slice label
saying "this specific slice shows the hemorrhage." MIL treats the whole
series as a **bag** of instances (slices): the model processes every slice
through a shared CNN backbone, then an **attention-pooling** layer learns
*by itself*, from the series-level loss alone, which slices matter most and
combines their features into one series-level prediction. This is the
standard approach when you have coarse (series/patient-level) labels but
fine-grained (slice-level) input structure -- common in histopathology
whole-slide imaging and, here, in CT.

**Why sanity-check the dataset before training:** these checks catch
architecture/data-pipeline bugs (e.g. augmentation params drifting between
slices of the same series, which would make the model see a physically
inconsistent "series") for the cost of a few seconds, instead of discovering
them after a multi-hour GPU training run produces confusingly bad results.

### 8.1 Augmentation consistency & visual check


In [ ]:
# Cell 13: Verify SeriesAugmentor integration in BrainCTSeriesDataset,
# using the live series_targets_df (Cell 11) and dicom_dir (Cell 3)
# already available in this notebook session.
import matplotlib.pyplot as plt
import torch

BrainCTSeriesDataset = dataset.BrainCTSeriesDataset
SeriesAugmentor = augmentation.SeriesAugmentor

TEST_SERIES_ID = 447

row = series_targets_df[series_targets_df["series_id"] == TEST_SERIES_ID].reset_index(drop=True)
assert len(row) == 1, f"Expected exactly one row for series {TEST_SERIES_ID}, got {len(row)}"
print(f"Series {TEST_SERIES_ID} is in fold {int(row['fold'].iloc[0])}")

augmentor = SeriesAugmentor()

ds_plain = BrainCTSeriesDataset(row, dicom_dir, is_train=True, augment=False)
ds_aug = BrainCTSeriesDataset(row, dicom_dir, is_train=True, augment=True, augmentor=augmentor)

# --- Check: sample_series_params() called exactly once per series ---
call_count = {"n": 0}
original_sampler = augmentor.sample_series_params


def counting_sampler():
    call_count["n"] += 1
    return original_sampler()


augmentor.sample_series_params = counting_sampler

sample_plain = ds_plain[0]
sample_aug = ds_aug[0]

n_slices_in_series = int(sample_plain["mask"].sum().item())
print(f"\n{n_slices_in_series} slices loaded.")
print(f"sample_series_params() call count: {call_count['n']} (expected: 1)")
assert call_count["n"] == 1, (
    f"Expected exactly 1 call, got {call_count['n']}. "
    f"Augmentation parameters are NOT consistent across slices."
)
print("PASS: augmentation parameters sampled once per series.\n")

# --- Check: targets identical between augmented and non-augmented samples ---
targets_match = torch.allclose(sample_plain["targets"], sample_aug["targets"])
print(f"targets (plain):     {sample_plain['targets'].tolist()}")
print(f"targets (augmented): {sample_aug['targets'].tolist()}")
assert targets_match, "Targets changed after augmentation."
print("PASS: targets unaffected by augmentation.\n")

# --- Check + visual: pixel-level difference on 3 sample slices ---
slice_indices = [0, n_slices_in_series // 2, n_slices_in_series - 1]

fig, axes = plt.subplots(2, len(slice_indices), figsize=(4 * len(slice_indices), 8))
for col, slice_idx in enumerate(slice_indices):
    plain_img = sample_plain["images"][slice_idx].permute(1, 2, 0).numpy()
    aug_img = sample_aug["images"][slice_idx].permute(1, 2, 0).numpy()

    mean_abs_diff = abs(plain_img[..., 0] - aug_img[..., 0]).mean()
    print(f"Slice {slice_idx}: mean abs pixel diff (brain window) = {mean_abs_diff:.4f}")
    assert mean_abs_diff > 0, f"Slice {slice_idx} shows no change after augmentation."

    axes[0, col].imshow(plain_img[..., 0], cmap="gray")
    axes[0, col].set_title(f"Plain - slice {slice_idx}")
    axes[0, col].axis("off")

    axes[1, col].imshow(aug_img[..., 0], cmap="gray")
    axes[1, col].set_title(f"Augmented - slice {slice_idx}")
    axes[1, col].axis("off")

plt.tight_layout()
plt.show()

print("\nPASS: augmented images differ from plain images (pixel diff > 0).")
print("\nVisual check: confirm the flip/rotation looks consistent across all 3 augmented columns.")

### 8.2 DataLoader throughput benchmark (choosing `num_workers`)

In [ ]:
# Cell 14: Benchmark DataLoader throughput with different num_workers settings,
# for both the augmented (training) and plain (validation) data paths.
import time

from torch.utils.data import DataLoader



BrainCTSeriesDataset = dataset.BrainCTSeriesDataset
SeriesAugmentor = augmentation.SeriesAugmentor

BATCH_SIZE = 4
N_BATCHES_TO_TIME = 10  # time a fixed number of batches, not the full epoch, for a quick estimate
NUM_WORKERS_OPTIONS = [0, 2, 4]

# Use fold 0's training split (all folds != 0) as a representative sample.
train_df = series_targets_df[series_targets_df["fold"] != 0].reset_index(drop=True)
print(f"Benchmarking on {len(train_df)} series (all folds except fold 0)")

augmentor = SeriesAugmentor()


def time_loader(ds, num_workers, n_batches, batch_size):
    loader = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
    )
    it = iter(loader)
    start = time.time()
    n_done = 0
    for _ in range(n_batches):
        try:
            next(it)
        except StopIteration:
            break
        n_done += 1
    elapsed = time.time() - start
    del it, loader
    return elapsed, n_done


results = []
for augment in [False, True]:
    ds = BrainCTSeriesDataset(
        train_df, dicom_dir, is_train=True,
        augment=augment,
        augmentor=augmentor if augment else None,
    )
    for num_workers in NUM_WORKERS_OPTIONS:
        elapsed, n_done = time_loader(ds, num_workers, N_BATCHES_TO_TIME, BATCH_SIZE)
        if n_done == 0:
            print(f"augment={augment}, num_workers={num_workers}: no batches completed, skipping")
            continue
        sec_per_batch = elapsed / n_done
        sec_per_series = sec_per_batch / BATCH_SIZE
        est_epoch_sec = sec_per_series * len(train_df)
        results.append({
            "augment": augment,
            "num_workers": num_workers,
            "sec_per_batch": sec_per_batch,
            "sec_per_series": sec_per_series,
            "estimated_epoch_sec": est_epoch_sec,
        })
        print(f"augment={augment}, num_workers={num_workers}: "
              f"{sec_per_batch:.3f} s/batch, {sec_per_series:.3f} s/series, "
              f"est. epoch time = {est_epoch_sec:.1f} s ({est_epoch_sec/60:.1f} min)")

results_df = pd.DataFrame(results)
print("\nSummary:")
results_df

## 9. MIL Model — Sanity Checks (forward pass, gradients, single-batch overfit)

**Objective:** verify the model architecture is wired correctly *before*
spending GPU time on real training. Four independent things are checked
below: output shapes, attention masking correctness, non-negativity of
regression outputs (enforced by Softplus), and gradient flow through the
intended trainable/frozen layer split.


In [ ]:
# Cell 15: Sanity-check BrainCTMILModel with one real batch from the DataLoader.
# Checks:
#   1. Forward pass runs without error on a real (non-dummy) batch.
#   2. Output shapes are correct.
#   3. Attention weights are exactly 0 for padded slices.
#   4. Gradients flow correctly (loss.backward() succeeds, backbone params get grads).
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

BrainCTSeriesDataset = dataset.BrainCTSeriesDataset
SeriesAugmentor = augmentation.SeriesAugmentor
BrainCTMILModel = model.BrainCTMILModel
TARGET_COLUMNS = dataset.TARGET_COLUMNS

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# --- Build a small real batch ---
train_df = series_targets_df[series_targets_df["fold"] != 0].reset_index(drop=True)
augmentor = SeriesAugmentor()
ds = BrainCTSeriesDataset(train_df, dicom_dir, is_train=True, augment=True, augmentor=augmentor)
loader = DataLoader(ds, batch_size=2, shuffle=True, num_workers=0)

batch = next(iter(loader))
images = batch["images"].to(DEVICE)       # (B, T, 3, H, W)
mask = batch["mask"].to(DEVICE)           # (B, T)
targets = batch["targets"].to(DEVICE)     # (B, 7), order = TARGET_COLUMNS

print(f"\nBatch shapes: images={tuple(images.shape)}, mask={tuple(mask.shape)}, targets={tuple(targets.shape)}")
print(f"Real slice counts in this batch: {mask.sum(dim=1).tolist()}")

# --- Forward pass ---
net = BrainCTMILModel(pretrained=True).to(DEVICE)
output = net(images, mask)

print("\nOutput keys:", list(output.keys()))
for key in ["V_EDH", "V_SDH", "V_IPH", "V_SAH", "V_IVH", "MLS_mm", "fracture_logit"]:
    val = output[key]
    assert val.shape == (images.shape[0],), f"Unexpected shape for {key}: {val.shape}"
print("PASS: all per-target output shapes are (batch_size,).")

# --- Check: attention weights are exactly 0 on padded slices ---
attn = output["attention_weights"]  # (B, T)
assert attn.shape == mask.shape
padded_weights = attn[mask == 0]
max_padded_weight = padded_weights.max().item() if padded_weights.numel() > 0 else 0.0
print(f"\nMax attention weight on a padded slice: {max_padded_weight:.8f} (expected: 0.0)")
assert max_padded_weight == 0.0, "Padded slices are receiving nonzero attention weight!"

attn_sums = attn.sum(dim=1)
print(f"Attention weights sum per series: {attn_sums.tolist()} (expected: ~1.0 each)")
assert torch.allclose(attn_sums, torch.ones_like(attn_sums), atol=1e-5)
print("PASS: attention correctly ignores padded slices and sums to 1 over real slices.")

# --- Check: non-negativity of regression outputs (Softplus should guarantee this) ---
for key in ["V_EDH", "V_SDH", "V_IPH", "V_SAH", "V_IVH", "MLS_mm"]:
    assert (output[key] >= 0).all(), f"{key} produced a negative value!"
print("PASS: all volume/MLS outputs are non-negative.")

# --- Check: intermediate tensor assembly matches TARGET_COLUMNS order ---
pred_intermediates = net.to_intermediate_tensor(output)
print(f"\nPredicted intermediates shape: {tuple(pred_intermediates.shape)} (expected: (B, 7))")
print("Order check (TARGET_COLUMNS):", TARGET_COLUMNS)
print("Sample prediction:", pred_intermediates[0].tolist())
print("Sample ground truth:", targets[0].tolist())

# --- Check: backward pass works and backbone receives gradients ---
loss_fn = nn.MSELoss()
loss = loss_fn(pred_intermediates, targets)
print(f"\nDummy MSE loss (untrained model, expect large): {loss.item():.2f}")

net.zero_grad()
loss.backward()

frozen_param = next(net.backbone.conv1.parameters())
assert frozen_param.grad is None, (
    "conv1 was expected to be frozen (freeze_early_backbone=True) but received a gradient anyway."
)
print("PASS: frozen early backbone layers (conv1/bn1/layer1/layer2) correctly received NO gradient.")

trainable_backbone_param = next(net.backbone.layer4.parameters())
assert trainable_backbone_param.grad is not None, "layer4 (should be trainable) did not receive gradients!"
assert trainable_backbone_param.grad.abs().sum().item() > 0, "layer4 gradients are all zero!"
print("PASS: gradients flow through attention pooling into the trainable backbone layers (layer3/layer4).")

print("\nAll sanity checks passed. Model architecture is wired correctly.")

## 10. MIL Training — v1 Baseline (ResNet18 + plain Attention Pooling)

**Note:** the full 5-fold v1 training loop has already been run to completion; its checkpoints live at `checkpoints/fold_{0..4}/best_model.pt` on Drive. If retraining from scratch is needed, use the v3 training cell (Section 13) as a template (set `CHECKPOINT_ROOT`/`LOG_PATH` and `use_weighted_sampling=False` to reproduce v1 exactly).

**Final v1 result:** `Mean CV QWK ≈ 0.4602` (per fold: 0.458, 0.518, 0.492, 0.400, 0.434).


In [ ]:
# Quick fix: recompute the CV summary directly from training_log.csv on disk,
# instead of relying on any in-memory variable (avoids the stale log_df bug).
import pandas as pd

LOG_PATH = "/content/drive/MyDrive/iaaa-brain-ct-triage/logs/training_log.csv"
log_df = pd.read_csv(LOG_PATH)

print(f"Total logged epochs across all folds: {len(log_df)}")
print(f"Folds present in log: {sorted(log_df['fold'].unique())}")

summary = log_df.groupby("fold")["val_qwk"].max()
print("\nPer-fold best QWK:")
print(summary)
print(f"\nMean CV QWK: {summary.mean():.4f}")

## 11. Building & Testing `submission.py` (v1 baseline)

**Objective:** package the trained model into the standalone script the
competition evaluator actually runs, and verify it end-to-end on real data
*before* submitting -- catching packaging bugs locally instead of burning
one of a limited number of official submission attempts to discover them.

**Why the submission CSV contains raw intermediates, not a `triage_class`
column:** `submission.py` outputs `series_id` + the 7 predicted
intermediates (`V_EDH`...`MLS_mm`, `fracture_prob`) -- not a final 0/1/2
class. The competition's own evaluator applies the official
`triage_from_intermediates` rule (Section 4) to score submissions. This
means participants are being scored on how well they estimate the
underlying imaging biomarkers, with the triage decision itself fixed and
identical for everyone -- which is exactly why this project's whole
strategy centers on intermediate-prediction accuracy rather than training
an end-to-end classifier.

**Why `submission.py` must be self-contained (no `from src import ...`):**
the competition's execution sandbox does not have this project's Drive-mounted
`src/` package available -- only the files actually included in the
submission package. A version that still imports from `src` would work
perfectly here (where `src/` happens to be on the path) and then fail
opaquely on the evaluator's side. Cell 11.4 below checks for this
specifically.

### 11.1 Inventory existing checkpoints


In [ ]:
# Diagnostic: check contents of all model/checkpoint-related directories.
import os

PROJECT_ROOT = "/content/drive/MyDrive/iaaa-brain-ct-triage"

paths_to_check = {
    "checkpoints/ (v1, per-fold)": os.path.join(PROJECT_ROOT, "checkpoints"),
    "checkpoints_v2/ (v2, per-fold)": os.path.join(PROJECT_ROOT, "checkpoints_v2"),
    "submission/models/ (for competition upload)": os.path.join(PROJECT_ROOT, "submission", "models"),
    "submission/ (root)": os.path.join(PROJECT_ROOT, "submission"),
}

for label, path in paths_to_check.items():
    print(f"\n{'=' * 60}\n{label}\npath: {path}\n{'=' * 60}")
    if not os.path.exists(path):
        print("  DOES NOT EXIST")
        continue

    for root, dirs, files in os.walk(path):
        depth = root.replace(path, "").count(os.sep)
        indent = "  " * (depth + 1)
        rel = os.path.relpath(root, path)
        print(f"{indent}{rel}/" if rel != "." else f"  ./")
        for f in files:
            fpath = os.path.join(root, f)
            size_mb = os.path.getsize(fpath) / (1024 * 1024)
            print(f"{indent}  {f}  ({size_mb:.1f} MB)")
        if not files and not dirs:
            print(f"{indent}  (empty)")

### 11.2 Collect the 5 best fold checkpoints into `submission/models/`

In [ ]:
# Cell 18: Gather the 5 best fold checkpoints into a clean, self-contained
# models/ folder for the submission package.
import os
import shutil

CHECKPOINT_ROOT = "/content/drive/MyDrive/iaaa-brain-ct-triage/checkpoints"
SUBMISSION_MODELS_DIR = "/content/drive/MyDrive/iaaa-brain-ct-triage/submission/models"
os.makedirs(SUBMISSION_MODELS_DIR, exist_ok=True)

N_FOLDS = 5
for fold in range(N_FOLDS):
    src_path = os.path.join(CHECKPOINT_ROOT, f"fold_{fold}", "best_model.pt")
    dst_path = os.path.join(SUBMISSION_MODELS_DIR, f"fold_{fold}_best.pt")
    assert os.path.isfile(src_path), f"Missing checkpoint: {src_path}"
    shutil.copy(src_path, dst_path)
    size_mb = os.path.getsize(dst_path) / (1024 * 1024)
    print(f"Copied fold {fold}: {dst_path} ({size_mb:.1f} MB)")

print(f"\nAll {N_FOLDS} fold checkpoints copied to {SUBMISSION_MODELS_DIR}")

### 11.3 Assemble the full submission package and smoke-test on 5 real series

In [ ]:
# Cell 19: Assemble the full submission package and smoke-test it end-to-end
# on a small subset of series BEFORE trusting it for the real evaluation.
import os
import shutil
import subprocess

PROJECT_ROOT = "/content/drive/MyDrive/iaaa-brain-ct-triage"
SUBMISSION_DIR = os.path.join(PROJECT_ROOT, "submission")
SUBMISSION_SRC_DIR = os.path.join(SUBMISSION_DIR, "src")

os.makedirs(SUBMISSION_SRC_DIR, exist_ok=True)

# --- Step 1: copy the minimal set of src/ files needed for INFERENCE only. ---
inference_files = ["__init__.py", "preprocessing.py", "augmentation.py", "dataset.py", "model.py"]
for fname in inference_files:
    src_path = os.path.join(PROJECT_ROOT, "src", fname)
    dst_path = os.path.join(SUBMISSION_SRC_DIR, fname)
    shutil.copy(src_path, dst_path)
    print(f"Copied {fname}")

# --- Step 2: copy submission.py into the submission root. ---
# (Assumes you've already uploaded submission.py to PROJECT_ROOT; adjust path if elsewhere.)
submission_py_src = os.path.join(PROJECT_ROOT, "submission.py")
submission_py_dst = os.path.join(SUBMISSION_DIR, "submission.py")
shutil.copy(submission_py_src, submission_py_dst)
print("Copied submission.py")

# --- Step 3: models/ should already exist from Cell 18. Verify. ---
models_dir = os.path.join(SUBMISSION_DIR, "models")
model_files = sorted(os.listdir(models_dir)) if os.path.isdir(models_dir) else []
print(f"\nmodels/ contents: {model_files}")
assert len(model_files) == 5, f"Expected 5 model files, found {len(model_files)}. Run Cell 18 first."

# --- Step 4: build a small test data-dir (symlinks to 5 real series) for a fast smoke test. ---
SMOKE_TEST_DIR = "/content/smoke_test_data"
if os.path.exists(SMOKE_TEST_DIR):
    shutil.rmtree(SMOKE_TEST_DIR)
os.makedirs(SMOKE_TEST_DIR)

sample_series_ids = series_targets_df["series_id"].sample(5, random_state=0).tolist()
for sid in sample_series_ids:
    src_series_dir = os.path.join(dicom_dir, str(sid))
    dst_series_dir = os.path.join(SMOKE_TEST_DIR, str(sid))
    os.symlink(src_series_dir, dst_series_dir)

print(f"\nSmoke-test data-dir built with series: {sample_series_ids}")

# --- Step 5: actually run submission.py as a subprocess, exactly as the official
# evaluator would invoke it. ---
output_csv_path = "/content/smoke_test_submission.csv"
result = subprocess.run(
    [
        "python", submission_py_dst,
        "--data-dir", SMOKE_TEST_DIR,
        "--predictions-file-path", output_csv_path,
    ],
    capture_output=True, text=True,
)

print("\n--- submission.py stdout ---")
print(result.stdout)
if result.returncode != 0:
    print("\n--- submission.py stderr ---")
    print(result.stderr)
    raise RuntimeError(f"submission.py exited with code {result.returncode}")

# --- Step 6: validate the output CSV. ---
import pandas as pd
from src import dataset as dataset_module  # for TARGET_COLUMNS, just to double-check schema

out_df = pd.read_csv(output_csv_path)
print("\nOutput CSV:")
print(out_df)

expected_columns = ["series_id"] + dataset_module.TARGET_COLUMNS
assert list(out_df.columns) == expected_columns, (
    f"Column mismatch!\nExpected: {expected_columns}\nGot:      {list(out_df.columns)}"
)
assert len(out_df) == len(sample_series_ids), "Row count doesn't match number of input series."
assert set(out_df["series_id"].astype(str)) == set(str(s) for s in sample_series_ids), \
    "series_id values don't match the input series."
assert (out_df["fracture_prob"] >= 0).all() and (out_df["fracture_prob"] <= 1).all(), \
    "fracture_prob out of [0, 1] range."
volume_cols = ["V_EDH", "V_SDH", "V_IPH", "V_SAH", "V_IVH", "MLS_mm"]
assert (out_df[volume_cols] >= 0).all().all(), "Found negative volume/MLS values."

print("\nPASS: submission.py ran end-to-end and produced a valid, correctly-schema'd CSV.")

### 11.4 Verify `submission.py` is truly standalone (no `src` dependency), and clean up

In [ ]:
# Verify submission.py is really the self-contained version (no "from src" imports),
# and clean up the leftover src/ folder inside submission/ that's no longer needed.
import os
import shutil

SUBMISSION_PY_PATH = "/content/drive/MyDrive/iaaa-brain-ct-triage/submission/submission.py"
SUBMISSION_SRC_DIR = "/content/drive/MyDrive/iaaa-brain-ct-triage/submission/src"

with open(SUBMISSION_PY_PATH) as f:
    content = f.read()

n_lines = content.count("\n")
has_src_import = "from src" in content
has_standalone_markers = "class InferenceSeriesDataset" in content and "class BrainCTMILModel" in content

print(f"submission.py: {n_lines} lines, {len(content)} bytes")
print(f"Contains 'from src' import: {has_src_import}  (expected: False)")
print(f"Contains standalone class definitions: {has_standalone_markers}  (expected: True)")

if has_src_import:
    print("\n*** This is STILL the OLD src-dependent version. Re-upload the standalone file. ***")
elif not has_standalone_markers:
    print("\n*** This file doesn't look like the expected standalone submission.py at all. ***")
else:
    print("\nPASS: this is the correct self-contained submission.py.")

    # Clean up the now-unnecessary src/ folder inside submission/.
    if os.path.isdir(SUBMISSION_SRC_DIR):
        shutil.rmtree(SUBMISSION_SRC_DIR)
        print(f"Removed leftover folder: {SUBMISSION_SRC_DIR}")

### 11.5 Regenerate a fresh submission CSV by running `submission.py` exactly as the evaluator would

In [ ]:
# Regenerate a fresh submission CSV, exactly as the official evaluator would
# produce it, by running your actual submission.py as a subprocess.
import os
import shutil
import subprocess

PROJECT_ROOT = "/content/drive/MyDrive/iaaa-brain-ct-triage"
SUBMISSION_PY = os.path.join(PROJECT_ROOT, "submission", "submission.py")

# Build a small test data-dir with a few real series (symlinks -- fast, no copying).
TEST_DATA_DIR = "/content/fresh_test_data"
if os.path.exists(TEST_DATA_DIR):
    shutil.rmtree(TEST_DATA_DIR)
os.makedirs(TEST_DATA_DIR)

sample_series_ids = series_targets_df["series_id"].sample(5, random_state=1).tolist()
for sid in sample_series_ids:
    os.symlink(os.path.join(dicom_dir, str(sid)), os.path.join(TEST_DATA_DIR, str(sid)))

OUTPUT_CSV_PATH = "/content/fresh_submission.csv"

result = subprocess.run(
    [
        "python", SUBMISSION_PY,
        "--data-dir", TEST_DATA_DIR,
        "--predictions-file-path", OUTPUT_CSV_PATH,
    ],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
else:
    print(f"\nFresh submission CSV generated at: {OUTPUT_CSV_PATH}")
    print("Use this path as YOUR_CSV_PATH in the comparison script.")

### 11.6 Byte-level comparison against the official sample CSV

**Why byte-level, not just "does it load in pandas":** subtle formatting
differences (a BOM, `\r\n` vs `\n` line endings, `series_id` written as
`2069` vs `2069.0`, trailing whitespace) can all load fine locally in
pandas while still causing a strict evaluator-side parser to reject the
file or silently misparse a column. Comparing raw bytes catches exactly
the class of bug that "it looks right when I print it" would miss.


In [ ]:
# Compare your actual generated submission CSV against the OFFICIAL sample
# (from the competition's own template repo) at the raw-byte level, to catch
# anything invisible when just eyeballing the file (BOM, encoding, line endings,
# extra whitespace, wrong dtype formatting for series_id, etc.).
import subprocess

# 1. Fetch the official sample directly (this runs in YOUR Colab environment,
#    which has normal internet access, unlike some restricted tools).
subprocess.run([
    "wget", "-q",
    "https://raw.githubusercontent.com/osint2030/iAAA-3rd-Contest/main/sample.bct.submission.csv",
    "-O", "/content/official_sample.csv",
], check=True)

with open("/content/official_sample.csv", "rb") as f:
    official_header = f.readline()

print("Official sample header (raw bytes):")
print(repr(official_header))

# 2. Point this at YOUR actual generated submission CSV (adjust path as needed --
#    e.g. the smoke-test output, or a fresh run of submission.py).
YOUR_CSV_PATH = "/content/fresh_submission.csv"  # <-- change if needed

with open(YOUR_CSV_PATH, "rb") as f:
    your_header = f.readline()

print("\nYour submission header (raw bytes):")
print(repr(your_header))

print(f"\nHeaders match exactly (byte-for-byte): {official_header.strip() == your_header.strip()}")

# 3. Also print a couple of full rows from your file, to sanity-check series_id
#    formatting (e.g. plain int vs float-looking "2069.0" vs string).
print("\nFirst 3 rows of your submission:")
with open(YOUR_CSV_PATH) as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i >= 3:
            break

### Official Result on the Competition System (v1)

`accuracy=0.3676`, `f1_macro=0.3427` — **the platform's displayed `Score` is actually `f1_macro`, not QWK** (unlike the original documentation, which described QWK as the primary metric).


## 12. Diagnostic: Internal (CV) Performance vs. Hidden Test Set Result

**Objective:** the official result above (`f1_macro=0.3427`) is much lower
than hoped. Before concluding anything about *why*, first answer a more
basic question: **is this a generalization problem** (the model does well
on data it has seen validation-style, but poorly on genuinely new hidden
data -- pointing at overfitting or a train/test distribution mismatch), **or
is it simply the real performance ceiling of this architecture** (the model
does equally poorly even on its own held-out folds, meaning there's nothing
to "fix" about generalization -- the architecture itself needs to change)?

Getting this diagnosis right matters a lot for what to do next: chasing
regularization/overfitting fixes for a problem that's actually an
architecture ceiling would waste significant time.

**A reproducibility detail worth noticing in the cell below:** it
re-defines the v1 model architecture (`BrainCTMILModelV1`) directly in this
cell, rather than importing `BrainCTMILModel` from the `model` module. This
is deliberate: by the time this diagnostic was run, `src/model.py` on Drive
had already been updated for later experiments (e.g. added Dropout), so
importing it here would silently mismatch the actual v1 checkpoint weights
and fail (or worse, load partially/incorrectly). Pinning an exact,
self-contained copy of the v1 architecture guarantees this diagnostic
reflects the real v1 model, immune to later edits elsewhere in the
codebase -- the same "fail fast/fail loud instead of silently wrong"
principle already seen in Section 7's assertions.


In [ ]:
# Compute accuracy and f1_macro (matching the competition's reported metrics,
# not just QWK) on our OWN internal validation folds using the v1 best models.
# This tells us whether the low external accuracy/f1_macro is consistent with
# our own validation data (=> no real train/test mismatch, just a metric-
# definition difference from QWK) or whether it's genuinely much worse on the
# hidden test set (=> a real generalization gap worth investigating further).
#
# NOTE: this defines the v1 model architecture LOCALLY (not via the notebook's
# `model` module, which now points to v2/Dropout) since the checkpoints/ folder
# holds v1 weights and the two must match exactly for load_state_dict to work.
import torch
import torch.nn as nn
import torchvision.models as tv_models
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


class AttentionMILPoolingV1(nn.Module):
    def __init__(self, feature_dim, hidden_dim=128):
        super().__init__()
        self.attention = nn.Sequential(nn.Linear(feature_dim, hidden_dim), nn.Tanh(), nn.Linear(hidden_dim, 1))

    def forward(self, features, mask):
        attn_logits = self.attention(features).squeeze(-1)
        attn_logits = attn_logits.masked_fill(mask == 0, float("-inf"))
        attn_weights = torch.softmax(attn_logits, dim=1)
        pooled = torch.sum(features * attn_weights.unsqueeze(-1), dim=1)
        return pooled, attn_weights


class BrainCTMILModelV1(nn.Module):
    def __init__(self, pretrained=False, attention_hidden=128):
        super().__init__()
        backbone = tv_models.resnet18(weights=tv_models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        self.feature_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.register_buffer("imagenet_mean", torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1))
        self.register_buffer("imagenet_std", torch.tensor(IMAGENET_STD).view(1, 3, 1, 1))
        self.pooling = AttentionMILPoolingV1(self.feature_dim, attention_hidden)
        self.regression_head = nn.Sequential(nn.Linear(self.feature_dim, 128), nn.ReLU(inplace=True), nn.Linear(128, 6))
        self.softplus = nn.Softplus()
        self.fracture_head = nn.Sequential(nn.Linear(self.feature_dim, 64), nn.ReLU(inplace=True), nn.Linear(64, 1))

    def forward(self, images, mask):
        B, T, C, H, W = images.shape
        flat_images = images.view(B * T, C, H, W)
        flat_images = (flat_images - self.imagenet_mean) / self.imagenet_std
        flat_features = self.backbone(flat_images)
        features = flat_features.view(B, T, self.feature_dim)
        pooled, attn_weights = self.pooling(features, mask)
        reg_out = self.softplus(self.regression_head(pooled))
        fracture_logit = self.fracture_head(pooled).squeeze(-1)
        return {
            "V_EDH": reg_out[:, 0], "V_SDH": reg_out[:, 1], "V_IPH": reg_out[:, 2],
            "V_SAH": reg_out[:, 3], "V_IVH": reg_out[:, 4], "MLS_mm": reg_out[:, 5],
            "fracture_logit": fracture_logit, "attention_weights": attn_weights,
        }

    def to_intermediate_tensor(self, output):
        fracture_prob = torch.sigmoid(output["fracture_logit"])
        return torch.stack(
            [output["V_EDH"], output["V_SDH"], output["V_IPH"], output["V_SAH"],
             output["V_IVH"], fracture_prob, output["MLS_mm"]], dim=1,
        )


BrainCTMILModel = BrainCTMILModelV1  # use the v1 class for the rest of this script
BrainCTSeriesDataset = dataset.BrainCTSeriesDataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CHECKPOINT_ROOT = "/content/drive/MyDrive/iaaa-brain-ct-triage/checkpoints"

all_fold_results = []

for fold in range(5):
    val_df = series_targets_df[series_targets_df["fold"] == fold].reset_index(drop=True)
    val_ds = BrainCTSeriesDataset(val_df, dicom_dir, is_train=True, augment=False)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=2)

    net = BrainCTMILModel(pretrained=False).to(DEVICE)
    ckpt_path = f"{CHECKPOINT_ROOT}/fold_{fold}/best_model.pt"
    checkpoint = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    net.load_state_dict(checkpoint["model_state"])
    net.eval()

    all_series_ids, all_preds = [], []
    with torch.no_grad():
        for batch in val_loader:
            images = batch["images"].to(DEVICE)
            mask = batch["mask"].to(DEVICE)
            output = net(images, mask)
            pred_intermediates = net.to_intermediate_tensor(output)
            all_preds.append(pred_intermediates.cpu())
            all_series_ids.extend(batch["series_id"].tolist())

    pred_tensor = torch.cat(all_preds, dim=0)
    pred_df = pd.DataFrame(pred_tensor.numpy(), columns=dataset.TARGET_COLUMNS)
    pred_df["series_id"] = all_series_ids
    true_df = val_df.set_index("series_id").loc[pred_df["series_id"]].reset_index()

    pred_triage = [triage.triage_from_intermediates(row.to_dict()) for _, row in pred_df[dataset.TARGET_COLUMNS].iterrows()]
    true_triage = [triage.triage_from_intermediates(row.to_dict()) for _, row in true_df[dataset.TARGET_COLUMNS].iterrows()]

    fold_accuracy = accuracy_score(true_triage, pred_triage)
    fold_f1_macro = f1_score(true_triage, pred_triage, average="macro")
    fold_qwk = cohen_kappa_score(true_triage, pred_triage, weights="quadratic")

    print(f"Fold {fold}: accuracy={fold_accuracy:.4f}  f1_macro={fold_f1_macro:.4f}  qwk={fold_qwk:.4f}")
    all_fold_results.append({"fold": fold, "accuracy": fold_accuracy, "f1_macro": fold_f1_macro, "qwk": fold_qwk})

results_df = pd.DataFrame(all_fold_results)
print("\nMean across folds:")
print(results_df[["accuracy", "f1_macro", "qwk"]].mean())

print(f"\nFor comparison, the competition's hidden-test scores were:")
print(f"  accuracy=0.3676, f1_macro=0.3427")

**Result:** internal accuracy/f1_macro (`0.361`/`0.333`) match the official platform result (`0.368`/`0.343`) almost exactly — meaning **there is no real overfitting or train/test mismatch**; this is simply the real ceiling of the MIL v1 architecture.


**What this diagnostic justified:** since the gap between internal CV and the hidden test result is essentially zero, there was no train/test mismatch to chase. This is what justified the architectural pivot in Sections 13-14 (gated attention, then abandoning MIL entirely for a segmentation-based approach) rather than spending time on regularization or more data augmentation, which would not have addressed a ceiling that was already being hit on the training distribution itself.

## 13. Model Improvement (v3): Gated Attention Pooling + Weighted Sampling

Two cheap, complementary improvements on top of the v2 architecture (which already added Dropout + freezing the backbone's early layers):
- **Gated Attention Pooling** (Ilse et al. 2018): `tanh(Vh) * sigmoid(Uh)` instead of plain `tanh(Vh)`.
- **Weighted Random Sampling**: the minority Urgent class (~18% of the data) is seen as often as the other classes each epoch.

First tested with a quick single-fold cycle (`FOLD_TO_RUN=0`), then run in full across all 5 folds.


**The intuition behind gated attention (why `tanh(Vh) * sigmoid(Uh)` beats
plain `tanh(Vh)`):** plain `tanh` attention scores a slice using one
non-linear transform of its features -- monotonic and fairly rigid. Adding a
second, independently-learned `sigmoid` "gate" lets the network learn to
*selectively suppress or pass through* specific feature dimensions before
scoring, rather than committing to one fixed non-linearity for all of them.
Concretely: `sigmoid(Uh)` acts like a soft feature mask (values in [0,1]
per dimension), multiplied element-wise into the `tanh(Vh)` score
contribution -- so the attention mechanism can learn "ignore texture-related
features when scoring this slice's relevance, but weight intensity-related
features heavily," instead of treating the whole feature vector uniformly.
This added flexibility is why it consistently outperforms plain attention
in the original MIL paper's ablations, at negligible extra compute cost.

**Why weighted sampling instead of e.g. oversampling by duplicating rows:**
`WeightedRandomSampler` assigns each *series* a sampling probability
inversely proportional to its class frequency, so the minority Urgent class
is seen roughly as often as the others across an epoch -- without
physically duplicating any series in the dataset (which would waste memory
and, worse, make the model see byte-identical augmented crops of the same
series more than once per epoch if not careful with the RNG). It directly
addresses the class-imbalance problem this project keeps running into
(also the motivation for the sqrt-inverse-frequency weighting later applied
to MLS regression in Section 21 -- same underlying idea, different loss
mechanism).

In [ ]:
# Cell 17 (v3): Full 5-fold cross-validation with gated attention pooling +
# weighted sampling for the Urgent minority class. Primary criterion for
# checkpointing/early-stopping is now f1_macro (confirmed to be the platform's
# displayed "Score"), with qwk and accuracy logged alongside for reference.
#
# Safe to re-run after a disconnect: resumes from saved checkpoints.
import os
import time

import pandas as pd
import torch

N_FOLDS = 5
N_EPOCHS = 15
BATCH_SIZE = 4
NUM_WORKERS = 2
PATIENCE = 5

CHECKPOINT_ROOT = "/content/drive/MyDrive/iaaa-brain-ct-triage/checkpoints_v3"
LOG_PATH = "/content/drive/MyDrive/iaaa-brain-ct-triage/logs/training_log_v3.csv"
os.makedirs(os.path.dirname(LOG_PATH), exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

log_rows = pd.read_csv(LOG_PATH).to_dict("records") if os.path.isfile(LOG_PATH) else []


def append_log(fold, epoch, train_loss, val_loss, metrics_dict):
    log_rows.append({
        "fold": fold, "epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
        "qwk": metrics_dict["qwk"], "accuracy": metrics_dict["accuracy"],
        "f1_macro": metrics_dict["f1_macro"],
        "timestamp": pd.Timestamp.now().isoformat(),
    })
    pd.DataFrame(log_rows).to_csv(LOG_PATH, index=False)


def epochs_since_best(history, key="f1_macro"):
    if not history:
        return 0
    vals = [h[key] for h in history]
    best_val = max(vals)
    last_best_idx = max(i for i, v in enumerate(vals) if v == best_val)
    return (len(history) - 1) - last_best_idx


for fold in range(N_FOLDS):
    print(f"\n{'=' * 60}\nFOLD {fold}\n{'=' * 60}")

    fold_dir = os.path.join(CHECKPOINT_ROOT, f"fold_{fold}")
    os.makedirs(fold_dir, exist_ok=True)
    last_ckpt_path = os.path.join(fold_dir, "last_checkpoint.pt")
    best_ckpt_path = os.path.join(fold_dir, "best_model.pt")
    done_marker_path = os.path.join(fold_dir, "DONE")

    if os.path.isfile(done_marker_path):
        print(f"Fold {fold} already marked DONE (finished in a previous run). Skipping.")
        continue

    train_loader, val_loader, train_df, val_df = train_utils.build_dataloaders(
        fold, series_targets_df, dicom_dir, dataset, augmentation,
        batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, use_weighted_sampling=True,
    )
    net, loss_fn, optimizer = train_utils.build_model_and_optimizer(model, losses, DEVICE)

    start_epoch = 0
    best_metric = -1.0
    history = []

    if os.path.isfile(last_ckpt_path):
        _, epoch_ckpt, best_metric, history = ckpt_utils.load_checkpoint(
            last_ckpt_path, net, optimizer, DEVICE
        )
        start_epoch = epoch_ckpt + 1
        print(f"Resuming fold {fold} from epoch {start_epoch} (best f1_macro so far: {best_metric:.4f})")
        if start_epoch >= N_EPOCHS:
            print(f"Fold {fold} already completed ({start_epoch}/{N_EPOCHS} epochs). Skipping.")
            continue

    for epoch in range(start_epoch, N_EPOCHS):
        t0 = time.time()
        train_loss = train_utils.train_one_epoch(net, train_loader, loss_fn, optimizer, DEVICE)
        val_loss, metrics_dict, pred_df, true_df = train_utils.validate_one_epoch(
            net, val_loader, loss_fn, val_df, DEVICE, metrics, dataset
        )
        elapsed = time.time() - t0

        print(f"  epoch {epoch:2d}/{N_EPOCHS - 1}: train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
              f"qwk={metrics_dict['qwk']:.4f}  accuracy={metrics_dict['accuracy']:.4f}  "
              f"f1_macro={metrics_dict['f1_macro']:.4f}  ({elapsed:.0f}s)")

        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, **metrics_dict})
        append_log(fold, epoch, train_loss, val_loss, metrics_dict)

        ckpt_utils.save_checkpoint(last_ckpt_path, fold, epoch, net, optimizer, best_metric, history)

        if metrics_dict["f1_macro"] > best_metric:
            best_metric = metrics_dict["f1_macro"]
            ckpt_utils.save_checkpoint(best_ckpt_path, fold, epoch, net, optimizer, best_metric, history)
            print(f"    New best f1_macro for fold {fold}: {best_metric:.4f} -- saved best_model.pt")

        since_best = epochs_since_best(history, key="f1_macro")
        if since_best >= PATIENCE:
            print(f"  Early stopping: f1_macro has not improved for {since_best} epochs (patience={PATIENCE}).")
            break

    with open(done_marker_path, "w") as f:
        f.write(f"best_f1_macro={best_metric:.4f}\n")

    print(f"Fold {fold} finished. Best f1_macro: {best_metric:.4f}")

print("\nAll folds complete.\n")
final_log_df = pd.DataFrame(log_rows)
summary = final_log_df.groupby("fold")[["qwk", "accuracy", "f1_macro"]].max()
print("Per-fold best metrics:")
print(summary)
print(f"\nMean across folds:\n{summary.mean()}")

**v3 result (fold 0, quick cycle):** `f1_macro` rose from `0.341` (v1) to `0.381` (~12% relative improvement). The full 5-fold infrastructure is ready and runnable.


## 14. Pivot: Segmentation-Based Approach

**Why:** top leaderboard scores are around `0.90+`, while our best result (the MIL architecture) was only `0.34-0.42`. Hypothesis: `triage_from_intermediates` is governed by strict, fine-grained volume thresholds (`EDH_CRIT=30mL`, etc.), and an approximate CNN-feature regression isn't precise enough at that scale. The real pixel-level annotations we already have make it possible to train a **segmentation** model that computes volume directly and precisely (`pixel_count × spacing`) from the predicted mask — exactly the same method the ground truth itself was built with.


### 14.1 Discovering and Validating the RLE Annotation Format

**Background: what is RLE (run-length encoding), and why use it for masks?**
A segmentation mask is a 2D array where most pixels usually belong to one
class (background). Storing every pixel explicitly wastes space; RLE instead
stores alternating **runs**: "how many pixels of value X, then how many of
value Y, then how many of X again, ...", scanning the flattened array in
order. This is small for masks with large contiguous regions (exactly the
case for anatomical structures), but the exact *convention* varies a lot
between tools (does it start with background or foreground? one class only,
or multiple encoded together? row-major or column-major flattening?) --
which is precisely why the cells below don't just assume a format and
decode blindly, but test competing hypotheses against a pixel count we
already know is correct from `training_df`.

**Methodology note:** this is a real example of hypothesis-driven debugging
applied to a data format problem, not just a model-training problem --
inspect the raw structure first (14.1a), form and test a specific decoding
hypothesis against a known-correct value (14.1b), and only trust the result
once it's been validated both numerically (exact pixel-count match) and
visually (mask overlaid on the actual anatomy, 14.1c).

#### 14.1a Inspect raw annotation JSON structure


In [ ]:
# Inspect the raw structure of ONE real annotation JSON file before writing
# any decode logic -- we need to see the actual key names and RLE encoding
# format with our own eyes, not guess from the documentation's prose description.
import glob
import json
import os

ANNOTATIONS_ROOT = annotations_dir  # already defined in Section 2

# Find any one series folder with at least one annotation file.
series_dirs = sorted(glob.glob(os.path.join(ANNOTATIONS_ROOT, "*")))
print(f"Found {len(series_dirs)} series folders under annotations/")

sample_series_dir = series_dirs[0]
json_files = sorted(glob.glob(os.path.join(sample_series_dir, "*.json")))
print(f"Sample series: {sample_series_dir}")
print(f"Number of annotation JSON files in it: {len(json_files)}")

sample_json_path = json_files[0]
print(f"\nInspecting: {sample_json_path}\n")

with open(sample_json_path) as f:
    data = json.load(f)

print("Top-level keys:", list(data.keys()))
print()

for key, value in data.items():
    if isinstance(value, list):
        print(f"'{key}': list, length={len(value)}")
        print(f"  First few elements: {value[:5]}")
    elif isinstance(value, dict):
        print(f"'{key}': dict, keys={list(value.keys())}")
        # Print a couple of sample entries
        for i, (k, v) in enumerate(value.items()):
            print(f"  '{k}': {v}")
            if i >= 4:
                break
    else:
        print(f"'{key}': {type(value).__name__} = {value}")
    print()

In [ ]:
# Find a slice that actually HAS hemorrhage annotated (non-zero area in
# training_df), then inspect its full segmentation_rle structure -- the
# previous sample happened to be an empty/background-only slice.
import json
import os

import pandas as pd

# metadata_df (loaded in Section 3) IS the slice-level training_df -- same file, no need to reload.
training_df = metadata_df
ANNOTATIONS_ROOT = annotations_dir

# Adjust these column names if your training_df uses different ones --
# print the columns first to check.
area_cols = [c for c in training_df.columns if "Area" in c]
print("\nArea-related columns in training_df:", area_cols)

# Find a row with a large, non-zero area in any hemorrhage column.
nonzero_row = None
for col in area_cols:
    candidates = training_df[training_df[col] > 1000]  # a decently large lesion
    if len(candidates) > 0:
        nonzero_row = candidates.iloc[0]
        print(f"\nUsing a slice with large '{col}' = {nonzero_row[col]}")
        break

print("\nFull row:")
print(nonzero_row)

# Now locate the matching annotation JSON. Adjust column names below to match
# whatever identifies the series and the specific slice (SOPInstanceUID) in
# your training_df -- print nonzero_row's index/columns above to check exact names.

#### 14.1b Test competing RLE decoding hypotheses against a known ground-truth pixel count

In [ ]:
# Deep-inspect a slice we KNOW has real hemorrhage (IVH area = 2230 px,
# per training_df), to determine the exact RLE decoding convention.
import json
import os

import numpy as np
import pandas as pd

# Reuse metadata_df (already loaded) and annotations_dir (already defined) instead of reloading.
training_df = metadata_df
ANNOTATIONS_ROOT = annotations_dir
row = training_df.iloc[15]  # the IVH=2230 example from before

json_path = os.path.join(ANNOTATIONS_ROOT, row["RelativeAnnotationPath"])
print(f"Loading: {json_path}")

with open(json_path) as f:
    data = json.load(f)

print("\nclass_map (full):")
for entry in data["class_map"]:
    print(f"  {entry}")

seg = data["segmentation_rle"]
shape = seg["shape"]
counts = seg["counts"]
h, w = shape
total_pixels = h * w

print(f"\nshape: {shape} (total pixels: {total_pixels})")
print(f"counts: length={len(counts)}, sum={sum(counts)}")
print(f"counts (first 20): {counts[:20]}")
print(f"counts (last 20): {counts[-20:]}")

# Known ground truth: IVH area should be 2230 pixels (class value 1, per class_map).
KNOWN_IVH_AREA = 2230

# --- Hypothesis A: standard binary/COCO-style alternating RLE, starting with
# background (even-indexed runs = value 0, odd-indexed = value 1), single class only. ---
def decode_alternating_binary(counts, h, w):
    flat = np.zeros(h * w, dtype=np.uint8)
    pos = 0
    current_value = 0
    for run_length in counts:
        if current_value == 1:
            flat[pos:pos + run_length] = 1
        pos += run_length
        current_value = 1 - current_value
    return flat.reshape(h, w)

mask_a = decode_alternating_binary(counts, h, w)
print(f"\nHypothesis A (alternating binary, starts at 0): "
      f"nonzero pixel count = {int(mask_a.sum())} (expected {KNOWN_IVH_AREA})")

# --- Hypothesis B: same but starts with value 1 (i.e. first run IS foreground). ---
def decode_alternating_binary_start1(counts, h, w):
    flat = np.zeros(h * w, dtype=np.uint8)
    pos = 0
    current_value = 1
    for run_length in counts:
        if current_value == 1:
            flat[pos:pos + run_length] = 1
        pos += run_length
        current_value = 1 - current_value
    return flat.reshape(h, w)

mask_b = decode_alternating_binary_start1(counts, h, w)
print(f"Hypothesis B (alternating binary, starts at 1): "
      f"nonzero pixel count = {int(mask_b.sum())} (expected {KNOWN_IVH_AREA})")

In [ ]:
# Decode using the (class_value, run_length) pair format, and validate
# against the known ground-truth IVH area (2230 px) for this exact slice.
import json
import os

import numpy as np
import pandas as pd

# Reuse metadata_df (already loaded) and annotations_dir (already defined) instead of reloading.
training_df = metadata_df
ANNOTATIONS_ROOT = annotations_dir
row = training_df.iloc[15]

json_path = os.path.join(ANNOTATIONS_ROOT, row["RelativeAnnotationPath"])
with open(json_path) as f:
    data = json.load(f)

seg = data["segmentation_rle"]
shape = seg["shape"]
counts = seg["counts"]
h, w = shape


def decode_rle_mask(counts, shape):
    """Decode a (class_value, run_length) pair-encoded RLE mask."""
    h, w = shape
    flat = np.zeros(h * w, dtype=np.uint8)
    pos = 0
    for i in range(0, len(counts), 2):
        value = counts[i]
        run_length = counts[i + 1]
        flat[pos:pos + run_length] = value
        pos += run_length
    assert pos == h * w, f"Decoded {pos} pixels, expected {h * w} -- RLE doesn't cover the full image."
    return flat.reshape(h, w)


mask = decode_rle_mask(counts, shape)

print("Unique values in decoded mask:", np.unique(mask))
for value in np.unique(mask):
    count = int((mask == value).sum())
    class_name = next(e["name"] for e in data["class_map"] if e["value"] == value)
    print(f"  value={value} ({class_name}): {count} pixels")

KNOWN_IVH_AREA = 2230
ivh_pixel_count = int((mask == 1).sum())
print(f"\nDecoded IVH (value=1) pixel count: {ivh_pixel_count}")
print(f"Known ground-truth IVH area: {KNOWN_IVH_AREA}")
print(f"MATCH: {ivh_pixel_count == KNOWN_IVH_AREA}")

In [ ]:
# Broader validation: decode RLE masks for several slices covering all 5
# hemorrhage subtypes, and compare decoded pixel counts to the known
# ground-truth *_Area columns in training_df.
import json
import os

import numpy as np
import pandas as pd

# Reuse metadata_df (already loaded) and annotations_dir (already defined) instead of reloading.
training_df = metadata_df
ANNOTATIONS_ROOT = annotations_dir

CLASS_AREA_COLUMNS = {
    1: "IntraventricularHemorrhage_Area",
    2: "IntraparenchymalHemorrhage_Area",
    3: "SubduralHemorrhage_Area",
    4: "EpiduralHemorrhage_Area",
    5: "SubarachnoidHemorrhage_Area",
}


def decode_rle_mask(counts, shape):
    h, w = shape
    flat = np.zeros(h * w, dtype=np.uint8)
    pos = 0
    for i in range(0, len(counts), 2):
        value = counts[i]
        run_length = counts[i + 1]
        flat[pos:pos + run_length] = value
        pos += run_length
    assert pos == h * w, f"Decoded {pos} pixels, expected {h * w}."
    return flat.reshape(h, w)


results = []
for class_value, area_col in CLASS_AREA_COLUMNS.items():
    candidates = training_df[training_df[area_col] > 500].head(4)
    for _, row in candidates.iterrows():
        json_path = os.path.join(ANNOTATIONS_ROOT, row["RelativeAnnotationPath"])
        if not os.path.isfile(json_path):
            continue
        with open(json_path) as f:
            data = json.load(f)

        seg = data["segmentation_rle"]
        mask = decode_rle_mask(seg["counts"], seg["shape"])
        decoded_area = int((mask == class_value).sum())
        true_area = row[area_col]
        pct_error = abs(decoded_area - true_area) / true_area * 100

        results.append({
            "series_id": row["dicom_series.id"],
            "class": area_col.replace("_Area", ""),
            "true_area": true_area,
            "decoded_area": decoded_area,
            "pct_error": pct_error,
        })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
print(f"\nMean absolute %% error: {results_df['pct_error'].mean():.3f}%")
print(f"Max %% error: {results_df['pct_error'].max():.3f}%")

#### 14.1c Visual sanity check: overlay the decoded mask on the real DICOM slice

In [ ]:
# Visual sanity check: load the actual DICOM slice for our known IVH example,
# overlay the decoded mask on top of it, and visually confirm:
#   1. The DICOM file is found at dicom_dir/{series_id}/{SOPInstanceUID}.dcm
#      (same naming convention as the annotation JSON).
#   2. The mask aligns spatially with the real anatomy (not flipped/shifted).
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydicom

# Reuse existing variables instead of reloading/redefining paths.
DICOM_ROOT = dicom_dir
ANNOTATIONS_ROOT = annotations_dir
training_df = metadata_df
row = training_df.iloc[15]  # our known IVH=2230 example

series_id = row["dicom_series.id"]
sop_uid = row["dicom_series.SOPInstanceUID"]

dcm_path = os.path.join(DICOM_ROOT, str(series_id), f"{sop_uid}.dcm")
print(f"Looking for DICOM at: {dcm_path}")
print(f"Exists: {os.path.isfile(dcm_path)}")

if not os.path.isfile(dcm_path):
    # Fall back to searching the whole series folder for a file containing the SOPInstanceUID.
    import glob
    candidates = glob.glob(os.path.join(DICOM_ROOT, str(series_id), f"*{sop_uid}*"))
    print(f"Fallback search found: {candidates}")
    dcm_path = candidates[0] if candidates else None

assert dcm_path is not None, "Could not locate the DICOM file for this slice."

dcm = pydicom.dcmread(dcm_path)
pixel_array = dcm.pixel_array.astype(np.float32)
slope = float(getattr(dcm, "RescaleSlope", 1.0))
intercept = float(getattr(dcm, "RescaleIntercept", 0.0))
hu_image = pixel_array * slope + intercept

# Brain window for display.
low, high = 40 - 80 / 2, 40 + 80 / 2
display_image = np.clip(hu_image, low, high)
display_image = (display_image - low) / (high - low)

# Decode the mask (reusing the validated logic inline, in case rle.py isn't uploaded yet).
import json

json_path = os.path.join(ANNOTATIONS_ROOT, row["RelativeAnnotationPath"])
with open(json_path) as f:
    ann = json.load(f)
seg = ann["segmentation_rle"]
h, w = seg["shape"]
flat = np.zeros(h * w, dtype=np.uint8)
pos = 0
for i in range(0, len(seg["counts"]), 2):
    value, run_length = seg["counts"][i], seg["counts"][i + 1]
    flat[pos:pos + run_length] = value
    pos += run_length
mask = flat.reshape(h, w)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(display_image, cmap="gray")
axes[0].set_title("CT slice (brain window)")
axes[0].axis("off")

axes[1].imshow(mask, cmap="tab10", vmin=0, vmax=5)
axes[1].set_title("Decoded mask (class labels)")
axes[1].axis("off")

axes[2].imshow(display_image, cmap="gray")
overlay = np.ma.masked_where(mask == 0, mask)
axes[2].imshow(overlay, cmap="autumn", alpha=0.6, vmin=0, vmax=5)
axes[2].set_title("Overlay (mask on CT)")
axes[2].axis("off")

plt.tight_layout()
plt.savefig("/content/rle_visual_check.png", dpi=120)
plt.show()

print("\nVisual check: does the highlighted region in the overlay look like it's INSIDE the skull,")
print("in a plausible location for intraventricular hemorrhage (near the ventricles)?")

**Result:** the RLE format is a list of `(class_value, run_length)` pairs (not standard COCO RLE). Mean decode error vs. ground truth: `~1.16%` (stable and small across all 5 classes — a sign of a minor difference in area-computation method, not a bug). Visual verification also confirmed image/mask alignment.


### 14.2 Checking Annotation Coverage (how many series actually have pixel-level annotations?)


In [ ]:
# Diagnose: how many series actually have annotation folders/files on disk,
# vs. how many are referenced in training_df (slice-level metadata).
import os

ANNOTATIONS_ROOT = annotations_dir

series_in_training_df = set(metadata_df["dicom_series.id"].unique())
print(f"Unique series referenced in training_df: {len(series_in_training_df)}")

series_with_annotation_folder = set(
    int(d) if d.isdigit() else d
    for d in os.listdir(ANNOTATIONS_ROOT)
    if os.path.isdir(os.path.join(ANNOTATIONS_ROOT, d))
)
print(f"Series with an annotations/ folder actually on disk: {len(series_with_annotation_folder)}")

missing = series_in_training_df - series_with_annotation_folder
print(f"\nSeries in training_df but MISSING annotation folder: {len(missing)}")

# Also check at the file level: for series that DO have a folder, are all
# individual JSON files present (not just the folder)?
present_series = series_in_training_df & series_with_annotation_folder
missing_files = 0
checked = 0
for series_id in list(present_series)[:50]:  # sample check, to keep this fast
    rows = metadata_df[metadata_df["dicom_series.id"] == series_id]
    for _, row in rows.iterrows():
        checked += 1
        json_path = os.path.join(ANNOTATIONS_ROOT, row["RelativeAnnotationPath"])
        if not os.path.isfile(json_path):
            missing_files += 1

print(f"\nSample check on {len(present_series) if len(present_series) < 50 else 50} series "
      f"with folders present: {checked} slices checked, {missing_files} missing files")

# Total slice-level impact.
n_slices_total = len(metadata_df)
n_slices_in_present_series = len(metadata_df[metadata_df["dicom_series.id"].isin(present_series)])
print(f"\nTotal slices in training_df: {n_slices_total}")
print(f"Slices belonging to series WITH annotation folder present: {n_slices_in_present_series} "
      f"({n_slices_in_present_series/n_slices_total*100:.1f}%)")

In [ ]:
# Check whether the 140 "missing" series actually exist inside the original
# zip archive (=> just a local extraction issue, fixable) or are genuinely
# never included in the archive at all (=> a real dataset limitation).
import zipfile
import glob

# Adjust this to wherever your original competition zip file is.
zip_candidates = glob.glob("/content/*.zip") + glob.glob("/content/drive/MyDrive/**/*.zip", recursive=True)
print("Found zip files:", zip_candidates)

if not zip_candidates:
    print("\nNo zip file found automatically -- please tell me the exact path to your "
          "original competition zip (e.g. iaaa-contest-bct.zip) so I can adjust this.")
else:
    zip_path = zip_candidates[0]
    print(f"\nInspecting: {zip_path}")

    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()

    annotation_entries = [n for n in names if "/annotations/" in n or n.startswith("annotations/")]
    print(f"Total annotation-related entries in zip: {len(annotation_entries)}")

    # Extract the series_id from each annotation path entry.
    series_ids_in_zip = set()
    for n in annotation_entries:
        parts = n.split("/")
        # find the segment right after "annotations"
        if "annotations" in parts:
            idx = parts.index("annotations")
            if idx + 1 < len(parts) and parts[idx + 1]:
                series_ids_in_zip.add(parts[idx + 1])

    print(f"Unique series_id folders referenced inside the zip's annotations/: {len(series_ids_in_zip)}")

    missing_series_str = {str(s) for s in missing}  # `missing` from the previous diagnostic cell
    recoverable = missing_series_str & series_ids_in_zip
    truly_absent = missing_series_str - series_ids_in_zip

    print(f"\nOf the 140 missing series:")
    print(f"  Present in zip (recoverable by re-extracting): {len(recoverable)}")
    print(f"  Genuinely absent from the zip entirely: {len(truly_absent)}")

**Result:** only `198` of `338` series have real pixel-level annotations — a genuine data limitation (confirmed by inspecting the original zip itself, not an extraction bug). That leaves 5176 slices (68.9%) usable for segmentation training.


**Why this ~69% coverage matters downstream:** this is a hard ceiling on segmentation training data (confirmed against the original zip, not an artifact of local extraction) -- worth remembering whenever segmentation performance is discussed later (Sections 15-19), since it isn't fixable by better code, only by a fundamentally different data source.

## 15. Segmentation Dataset & Model

**Objective:** build and sanity-check the pieces of the segmentation
pipeline -- a slice-level dataset restricted to the ~69% of slices with
real pixel annotations (Section 14.2), and a U-Net model that predicts a
6-class mask (background + 5 hemorrhage subtypes: EDH, SDH, IPH, SAH, IVH)
per pixel.

**Why U-Net specifically, for this problem:** hemorrhage regions can be
small and irregularly shaped, and their *exact boundary* matters directly
for the downstream volume calculation (Section 4's rule thresholds are in
mL, computed from pixel count × physical spacing) -- not just "hemorrhage
present somewhere in this slice." U-Net's encoder-decoder structure with
skip connections is built for exactly this: the encoder path captures
broad context (is this plausibly brain tissue with a hemorrhage pattern at
all?), while the skip connections feed fine spatial detail from early
encoder layers directly into the decoder, preserving the boundary precision
a classification-only CNN (like the earlier MIL model) architecturally
cannot recover once it pools that detail away.

**Why `segmentation_models_pytorch` (SMP) with a ResNet18 encoder:** SMP
provides well-tested U-Net (and other) decoder implementations pluggable
onto any torchvision-style backbone, with ImageNet-pretrained weights
available for the encoder. Given only ~5,176 annotated slices (Section
14.2's hard data ceiling), starting from ImageNet features rather than
random initialization matters a lot -- the encoder doesn't have to relearn
basic edge/texture detectors from scratch on a comparatively small medical
dataset.

**Why a combined Dice + Cross-Entropy loss (seen in the sanity check
below):** Dice loss directly optimizes region overlap, which is robust to
the severe foreground/background class imbalance in these masks (a
hemorrhage might occupy a few percent of a slice's pixels) -- but Dice's
gradient can be unstable or uninformative early in training when
predicted masks are still near-empty. Cross-entropy gives a stable,
well-behaved pixel-wise gradient from the very first step, even when
Dice alone would barely move. Combining both is a well-established
approach in medical segmentation literature for exactly this reason: CE
provides stable early training, Dice provides accurate late-training
region-overlap optimization.


In [ ]:
!pip install segmentation-models-pytorch --quiet

In [ ]:
# Cell: Build the slice-level DataFrame (with fold assignment merged in from
# series_targets_df), then sanity-test SegmentationSliceDataset end-to-end
# with a real batch, and visualize a few (image, mask) pairs.
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

# Reuse dicom_dir / annotations_dir (defined in Section 2).
DICOM_ROOT = dicom_dir
ANNOTATIONS_ROOT = annotations_dir

build_slice_df_with_folds = seg_dataset.build_slice_df_with_folds
SegmentationSliceDataset = seg_dataset.SegmentationSliceDataset

slice_df = build_slice_df_with_folds(metadata_df, series_targets_df)
slice_df = seg_dataset.filter_to_available_annotations(slice_df, ANNOTATIONS_ROOT)

# Build a small dataset/loader on fold 0's slices for a quick test.
sample_slice_df = slice_df[slice_df["fold"] == 0].reset_index(drop=True)
ds = SegmentationSliceDataset(sample_slice_df, DICOM_ROOT, ANNOTATIONS_ROOT)
loader = DataLoader(ds, batch_size=4, shuffle=True, num_workers=0)

batch = next(iter(loader))
print(f"\nBatch shapes: image={tuple(batch['image'].shape)}, mask={tuple(batch['mask'].shape)}")
print(f"Mask value range: {batch['mask'].min().item()} to {batch['mask'].max().item()} (expected 0-5)")
print(f"Unique mask values in this batch: {torch.unique(batch['mask']).tolist()}")

# Visualize.
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    img = batch["image"][i].permute(1, 2, 0).numpy()
    mask = batch["mask"][i].numpy()

    axes[0, i].imshow(img[..., 0], cmap="gray")
    axes[0, i].set_title(f"Slice {i} (series {batch['series_id'][i].item()})")
    axes[0, i].axis("off")

    axes[1, i].imshow(img[..., 0], cmap="gray")
    import numpy as np
    overlay = np.ma.masked_where(mask == 0, mask)
    axes[1, i].imshow(overlay, cmap="autumn", alpha=0.6, vmin=0, vmax=5)
    axes[1, i].axis("off")

plt.tight_layout()
plt.savefig("/content/seg_dataset_check.png", dpi=120)
plt.show()

print("\nPASS (pending visual check): if any of these 4 slices has hemorrhage, "
      "confirm the overlay looks anatomically plausible.")

In [ ]:
# Cell: sanity-check HemorrhageSegModel + SegmentationLoss with a real batch,
# including a single-batch overfit test (same philosophy as the MIL model's
# Cell 16 -- catch wiring bugs in ~30 seconds instead of after a full run).
import torch

HemorrhageSegModel = seg_model.HemorrhageSegModel
SegmentationLoss = seg_losses.SegmentationLoss

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

net = HemorrhageSegModel(pretrained=True).to(DEVICE)
loss_fn = SegmentationLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=1e-4)

images = batch["image"].to(DEVICE)
masks = batch["mask"].to(DEVICE)

print(f"images: {tuple(images.shape)}, masks: {tuple(masks.shape)}")

logits = net(images)
print(f"logits: {tuple(logits.shape)} (expected (B, 6, H, W))")
assert logits.shape[1] == 6

loss_history = []
for step in range(40):
    optimizer.zero_grad()
    logits = net(images)
    loss_dict = loss_fn(logits, masks)
    loss_dict["total"].backward()
    optimizer.step()
    loss_history.append(loss_dict["total"].item())
    if step % 10 == 0 or step == 39:
        print(f"step {step:2d}: total={loss_dict['total'].item():.4f}  "
              f"dice={loss_dict['dice'].item():.4f}  ce={loss_dict['ce'].item():.4f}")

print(f"\nReduction: {(1 - loss_history[-1] / loss_history[0]) * 100:.1f}%")
assert loss_history[-1] < loss_history[0] * 0.7, "Loss did not drop enough -- check wiring."
print("PASS: segmentation model + loss + optimizer pipeline is wired correctly.")

## 16. Next Steps (TODO, as of the segmentation pivot)

1. Write the full segmentation training loop (similar to Section 13, but slice-level, across 5 filtered folds).
2. Validate: compare volumes computed from the predicted mask (`volume_aggregation.predict_series_volumes`) against ground truth, both at the pixel-Dice/IoU level and the final mL-volume level.
3. Build the final hybrid pipeline: segmentation for the 5 hemorrhage volumes + the MIL (v3) model for `MLS_mm` and `fracture_prob`.
4. Build and test a new hybrid `submission.py` (similar to Section 11, this time with two models).
5. Submit to the competition system and compare the new `Score` against the current baseline (`f1_macro=0.343`).


### 16.1 Extended sanity check before committing to full training

**Why go further than Section 15's 40-step overfit test:** Section 12's
lesson (a whole MIL architecture change was needed, not a quick fix) made
clear how costly it is to discover a pipeline problem only after a full
multi-hour training run. Before committing GPU time to 5-fold segmentation
training, this cell pushes the overfit test further (150 steps, higher LR)
to build real confidence the loss keeps dropping toward near-zero on a
single batch, rather than plateauing early -- the difference between "just
needs more steps" and "there's a real bug" is exactly what this extra check
is designed to catch cheaply.


In [ ]:
# Extended diagnostic: (1) check what's actually in this batch, (2) run a
# longer overfit test with a higher LR to see if loss continues toward
# near-zero (confirming the pipeline is fine, just needed more steps) or
# plateaus early (which would indicate a real problem).
import torch

print("Mask class composition in this batch:")
unique, counts = torch.unique(masks, return_counts=True)
for u, c in zip(unique.tolist(), counts.tolist()):
    pct = c / masks.numel() * 100
    print(f"  class {u}: {c} pixels ({pct:.2f}%)")

# Re-run the overfit test with more steps and a slightly higher LR.
net2 = seg_model.HemorrhageSegModel(pretrained=True).to(DEVICE)
loss_fn2 = seg_losses.SegmentationLoss()
optimizer2 = torch.optim.Adam(net2.parameters(), lr=3e-4)

loss_history2 = []
for step in range(150):
    optimizer2.zero_grad()
    logits = net2(images)
    loss_dict = loss_fn2(logits, masks)
    loss_dict["total"].backward()
    optimizer2.step()
    loss_history2.append(loss_dict["total"].item())
    if step % 25 == 0 or step == 149:
        print(f"step {step:3d}: total={loss_dict['total'].item():.4f}  "
              f"dice={loss_dict['dice'].item():.4f}  ce={loss_dict['ce'].item():.4f}")

print(f"\nReduction over 150 steps: {(1 - loss_history2[-1] / loss_history2[0]) * 100:.1f}%")

# Check per-pixel accuracy on this batch now, for an intuitive sanity check.
net2.eval()
with torch.no_grad():
    final_logits = net2(images)
    pred_mask = final_logits.argmax(dim=1)
    pixel_accuracy = (pred_mask == masks).float().mean().item()
print(f"Final per-pixel accuracy on this batch: {pixel_accuracy:.4f}")

## 17. Segmentation v1 Training (5-fold, slice-level)

*(Missing header, fixed here — the original notebook jumped straight from
Section 16 into this training loop with no section heading at all.)*

**Objective:** train the U-Net hemorrhage segmentation model across the 5
CV folds, at the slice level, using only the 198 series with real pixel
annotations (Section 14.2). Checkpointed and resumable, mirroring the MIL
v3 training loop's pattern (Section 13) -- log every epoch to CSV, save the
best checkpoint per fold by `val_dice`, stop early on `PATIENCE` epochs
without improvement.

**Why `mean_dice_score` excludes the background class (class 0):**
background dominates almost every slice by pixel count; including it in the
mean would make the metric close to 1.0 regardless of hemorrhage
segmentation quality, masking the thing actually being optimized for.
Averaging Dice only over the 5 hemorrhage classes keeps the metric
sensitive to what clinically matters.


In [ ]:
# Segmentation training: full 5-fold cross-validation at the SLICE level,
# using the 198 series with real pixel annotations. Checkpointed/resumable
# like the MIL training loop (Cell 17 v3).
import os
import time

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

N_FOLDS = 5
N_EPOCHS = 12
BATCH_SIZE = 8
NUM_WORKERS = 2
PATIENCE = 4

CHECKPOINT_ROOT = "/content/drive/MyDrive/iaaa-brain-ct-triage/checkpoints_seg"
LOG_PATH = "/content/drive/MyDrive/iaaa-brain-ct-triage/logs/training_log_seg.csv"
os.makedirs(os.path.dirname(LOG_PATH), exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Build the filtered slice-level DataFrame (198 series with real annotations).
full_slice_df = seg_dataset.build_slice_df_with_folds(metadata_df, series_targets_df)
full_slice_df = seg_dataset.filter_to_available_annotations(full_slice_df, annotations_dir)

log_rows = pd.read_csv(LOG_PATH).to_dict("records") if os.path.isfile(LOG_PATH) else []


def append_log(fold, epoch, train_loss, val_loss, val_dice, val_pixel_acc):
    log_rows.append({
        "fold": fold, "epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
        "val_dice": val_dice, "val_pixel_acc": val_pixel_acc,
        "timestamp": pd.Timestamp.now().isoformat(),
    })
    pd.DataFrame(log_rows).to_csv(LOG_PATH, index=False)


def epochs_since_best(history, key="val_dice"):
    if not history:
        return 0
    vals = [h[key] for h in history]
    best_val = max(vals)
    last_best_idx = max(i for i, v in enumerate(vals) if v == best_val)
    return (len(history) - 1) - last_best_idx


@torch.no_grad()
def mean_dice_score(logits, target_mask, n_classes=6, eps=1e-6):
    """Mean Dice score across the 5 hemorrhage classes (excludes BG=0, since
    background dominates and isn't the metric of clinical interest)."""
    pred = logits.argmax(dim=1)
    dice_scores = []
    for c in range(1, n_classes):
        pred_c = (pred == c)
        target_c = (target_mask == c)
        if target_c.sum() == 0 and pred_c.sum() == 0:
            continue  # class absent from both -- skip, not informative
        intersection = (pred_c & target_c).float().sum()
        dice = (2 * intersection + eps) / (pred_c.float().sum() + target_c.float().sum() + eps)
        dice_scores.append(dice.item())
    return float(np.mean(dice_scores)) if dice_scores else 1.0


for fold in range(N_FOLDS):
    print(f"\n{'=' * 60}\nSEGMENTATION FOLD {fold}\n{'=' * 60}")

    fold_dir = os.path.join(CHECKPOINT_ROOT, f"fold_{fold}")
    os.makedirs(fold_dir, exist_ok=True)
    last_ckpt_path = os.path.join(fold_dir, "last_checkpoint.pt")
    best_ckpt_path = os.path.join(fold_dir, "best_model.pt")
    done_marker_path = os.path.join(fold_dir, "DONE")

    if os.path.isfile(done_marker_path):
        print(f"Fold {fold} already marked DONE. Skipping.")
        continue

    train_slice_df = full_slice_df[full_slice_df["fold"] != fold].reset_index(drop=True)
    val_slice_df = full_slice_df[full_slice_df["fold"] == fold].reset_index(drop=True)

    train_ds = seg_dataset.SegmentationSliceDataset(train_slice_df, dicom_dir, annotations_dir)
    val_ds = seg_dataset.SegmentationSliceDataset(val_slice_df, dicom_dir, annotations_dir)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)

    net = seg_model.HemorrhageSegModel(pretrained=True).to(DEVICE)
    loss_fn = seg_losses.SegmentationLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=3e-4, weight_decay=1e-5)

    start_epoch = 0
    best_metric = -1.0
    history = []

    if os.path.isfile(last_ckpt_path):
        ckpt = torch.load(last_ckpt_path, map_location=DEVICE, weights_only=False)
        net.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        start_epoch = ckpt["epoch"] + 1
        best_metric = ckpt["best_metric"]
        history = ckpt["history"]
        print(f"Resuming fold {fold} from epoch {start_epoch} (best val_dice so far: {best_metric:.4f})")
        if start_epoch >= N_EPOCHS:
            print(f"Fold {fold} already completed. Skipping.")
            continue

    for epoch in range(start_epoch, N_EPOCHS):
        t0 = time.time()

        # --- train ---
        net.train()
        train_loss_total, n_train_batches = 0.0, 0
        for batch in train_loader:
            images = batch["image"].to(DEVICE)
            masks = batch["mask"].to(DEVICE)
            optimizer.zero_grad()
            logits = net(images)
            loss_dict = loss_fn(logits, masks)
            loss_dict["total"].backward()
            optimizer.step()
            train_loss_total += loss_dict["total"].item()
            n_train_batches += 1
        train_loss = train_loss_total / max(n_train_batches, 1)

        # --- validate ---
        net.eval()
        val_loss_total, n_val_batches = 0.0, 0
        dice_scores, pixel_accs = [], []
        with torch.no_grad():
            for batch in val_loader:
                images = batch["image"].to(DEVICE)
                masks = batch["mask"].to(DEVICE)
                logits = net(images)
                loss_dict = loss_fn(logits, masks)
                val_loss_total += loss_dict["total"].item()
                n_val_batches += 1
                dice_scores.append(mean_dice_score(logits, masks))
                pred = logits.argmax(dim=1)
                pixel_accs.append((pred == masks).float().mean().item())
        val_loss = val_loss_total / max(n_val_batches, 1)
        val_dice = float(np.mean(dice_scores))
        val_pixel_acc = float(np.mean(pixel_accs))

        elapsed = time.time() - t0
        print(f"  epoch {epoch:2d}/{N_EPOCHS - 1}: train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
              f"val_dice={val_dice:.4f}  val_pixel_acc={val_pixel_acc:.4f}  ({elapsed:.0f}s)")

        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                         "val_dice": val_dice, "val_pixel_acc": val_pixel_acc})
        append_log(fold, epoch, train_loss, val_loss, val_dice, val_pixel_acc)

        torch.save({"epoch": epoch, "model_state": net.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "best_metric": best_metric, "history": history}, last_ckpt_path)

        if val_dice > best_metric:
            best_metric = val_dice
            torch.save({"epoch": epoch, "model_state": net.state_dict(),
                        "optimizer_state": optimizer.state_dict(),
                        "best_metric": best_metric, "history": history}, best_ckpt_path)
            print(f"    New best val_dice for fold {fold}: {best_metric:.4f} -- saved best_model.pt")

        if epochs_since_best(history) >= PATIENCE:
            print(f"  Early stopping: val_dice has not improved for {PATIENCE} epochs.")
            break

    with open(done_marker_path, "w") as f:
        f.write(f"best_val_dice={best_metric:.4f}\n")
    print(f"Fold {fold} finished. Best val_dice: {best_metric:.4f}")

print("\nAll segmentation folds complete.\n")
final_log_df = pd.DataFrame(log_rows)
summary = final_log_df.groupby("fold")[["val_dice", "val_pixel_acc"]].max()
print(summary)
print(f"\nMean val_dice across folds: {summary['val_dice'].mean():.4f}")

### v1 Segmentation Training Result

`mean val_dice ≈ 0.392` across the 5 folds. Log analysis showed the best `val_dice` occurred at epoch 0 or 1 in most folds (early stopping on a noisy, small-validation-set metric), and a one-off loss spike in fold 2 (a temporary training instability, likely gradient-related). This motivates the volume-level validation below, followed by a v2 training pass that fixes these issues directly.


### 17.2 Volume-Level Validation & Root-Cause Diagnosis

*(Also missing a header in the original notebook — six consecutive cells
of dense, important diagnostic work with no heading at all, directly
between Section 17's training loop and Section 18's fix. Numbered as a
17.x continuation, not a new top-level section, so Sections 18-20's
existing numbers don't shift.)*

**Objective:** pixel-level Dice (Section 17) doesn't directly answer the
question that actually matters -- does the *aggregated mL volume*, fed
through the real triage rule (Section 4), produce the correct triage class?
This subsection converts predicted masks to volumes with the project's real
`volume_aggregation.predict_series_volumes` function (isolated, no guessed
formulas), runs the full rule on them, and diagnoses what's wrong when it
isn't accurate yet.


In [ ]:
# Cell: FINAL volume-level validation, built strictly from the real,
# validated project functions (no guessed formulas, no guessed columns).
import pandas as pd
import numpy as np
from collections import defaultdict

CHECKPOINT_ROOT = "/content/drive/MyDrive/iaaa-brain-ct-triage/checkpoints_seg"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_SIZE = 224

all_rows = []

for fold in range(5):
    print(f"Evaluating fold {fold}...")
    best_ckpt_path = f"{CHECKPOINT_ROOT}/fold_{fold}/best_model.pt"
    ckpt = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)

    net = seg_model.HemorrhageSegModel(pretrained=False).to(DEVICE)
    net.load_state_dict(ckpt["model_state"])
    net.eval()

    val_slice_df = full_slice_df[full_slice_df["fold"] == fold].reset_index(drop=True)
    val_ds = seg_dataset.SegmentationSliceDataset(
        val_slice_df, dicom_dir, annotations_dir, image_size=IMAGE_SIZE
    )

    # Group this fold's slices by series_id, building the list-of-dicts
    # shape that volume_aggregation.predict_series_volumes expects.
    series_slices = defaultdict(list)
    for i in range(len(val_ds)):
        item = val_ds[i]
        series_slices[item["series_id"]].append(item)

    for series_id, slices in series_slices.items():
        # --- Predicted volumes from the segmentation model ---
        pred_volumes = volume_aggregation.predict_series_volumes(
            net, slices, DEVICE, image_size=IMAGE_SIZE
        )  # {"V_IVH": ..., "V_IPH": ..., "V_SDH": ..., "V_EDH": ..., "V_SAH": ...}

        # --- Ground-truth intermediates (volumes + REAL MLS/fracture) from
        # the original, already-validated per-slice metadata. ---
        series_meta_rows = metadata_df[metadata_df["dicom_series.id"] == series_id]
        gt_intermediates = aggregate_series_intermediates(series_meta_rows)

        # --- Isolation: predicted volumes + GROUND-TRUTH MLS/fracture. ---
        isolated_intermediates = {
            **pred_volumes,
            "MLS_mm": gt_intermediates["MLS_mm"],
            "fracture_prob": gt_intermediates["fracture_prob"],
        }

        pred_class = triage_from_intermediates(isolated_intermediates)
        true_class = triage_from_intermediates(gt_intermediates)

        row = {"series_id": series_id, "fold": fold,
               "true_triage": true_class, "pred_triage_from_seg": pred_class}
        for key in ["V_EDH", "V_SDH", "V_IPH", "V_SAH", "V_IVH"]:
            row[f"{key}_pred"] = pred_volumes[key]
            row[f"{key}_gt"] = gt_intermediates[key]
        all_rows.append(row)

results_df = pd.DataFrame(all_rows)

print(f"\nTotal series evaluated: {len(results_df)}")

print("\n=== Per-subtype volume MAE (mL) ===")
for key in ["V_EDH", "V_SDH", "V_IPH", "V_SAH", "V_IVH"]:
    mae = (results_df[f"{key}_pred"] - results_df[f"{key}_gt"]).abs().mean()
    print(f"  {key}: {mae:.3f} mL")

accuracy = (results_df["pred_triage_from_seg"] == results_df["true_triage"]).mean()
print(f"\n=== Isolated triage accuracy (predicted volumes + GT MLS/fracture) ===")
print(f"Accuracy: {accuracy:.4f}  ({(results_df['pred_triage_from_seg'] == results_df['true_triage']).sum()}/{len(results_df)})")

print("\nConfusion matrix (rows=true, cols=predicted):")
print(pd.crosstab(results_df["true_triage"], results_df["pred_triage_from_seg"]))

In [ ]:
# 1. Apples-to-apples metric: f1_macro on this isolated experiment.
from sklearn.metrics import f1_score, classification_report

f1_macro_isolated = f1_score(results_df["true_triage"], results_df["pred_triage_from_seg"], average="macro")
print(f"f1_macro (isolated, seg-only): {f1_macro_isolated:.4f}")
print(classification_report(results_df["true_triage"], results_df["pred_triage_from_seg"],
                              target_names=["Non-Urgent", "Urgent", "Critical"]))

# 2. Drill into the Non-Urgent -> Urgent over-triage cases specifically:
# which subtype(s) are driving the false-positive volume that crosses the threshold?
over_triage_mask = (results_df["true_triage"] == 0) & (results_df["pred_triage_from_seg"] == 1)
over_triage_cases = results_df[over_triage_mask]

print(f"\n{len(over_triage_cases)} over-triage (Non-Urgent -> Urgent) cases. "
      f"Predicted vs GT volume per subtype:")
for key in ["V_EDH", "V_SDH", "V_IPH", "V_SAH", "V_IVH"]:
    pred_mean = over_triage_cases[f"{key}_pred"].mean()
    gt_mean = over_triage_cases[f"{key}_gt"].mean()
    print(f"  {key}: pred_mean={pred_mean:.2f} mL   gt_mean={gt_mean:.2f} mL")

**Result (v1, isolated — predicted volumes + ground-truth MLS/fracture):** `accuracy=0.7929`, `f1_macro=0.6669` — roughly double the best MIL result (`f1_macro=0.343`), confirming the segmentation hypothesis. The confusion matrix showed **zero** Critical→Non-Urgent or Non-Urgent→Critical errors (the clinically dangerous under-triage direction never occurs), but a systematic over-triage pattern: 18/25 true Non-Urgent series were pushed to Urgent, traced to small false-positive volume noise (mostly in the IPH class) crossing the rule's very small `EPS_VOLUME=0.1mL` threshold.


**Testing a hypothesis before acting on it:** the over-triage pattern above
looks, at first glance, like it could be fixed with a simple minimum-volume
noise-floor cutoff. Before implementing that, the cell below checks whether
the data actually supports it -- comparing predicted-volume distributions on
truly-healthy (GT=0) series against genuinely-positive (GT>0) series, per
hemorrhage class.


In [ ]:
# Step 1: visualize the separation between FP-noise and true-positive volumes,
# per hemorrhage class, BEFORE picking any threshold.
for key in ["V_EDH", "V_SDH", "V_IPH", "V_SAH", "V_IVH"]:
    gt_col, pred_col = f"{key}_gt", f"{key}_pred"

    is_gt_zero = results_df[gt_col] < 0.01  # essentially zero ground truth
    is_gt_nonzero = ~is_gt_zero

    fp_noise = results_df.loc[is_gt_zero, pred_col]
    tp_signal = results_df.loc[is_gt_nonzero, pred_col]

    print(f"\n--- {key} ---")
    print(f"  GT=0 series (n={len(fp_noise)}): "
          f"pred mean={fp_noise.mean():.3f}, median={fp_noise.median():.3f}, "
          f"95th pct={fp_noise.quantile(0.95):.3f}, max={fp_noise.max():.3f}")
    print(f"  GT>0 series (n={len(tp_signal)}): "
          f"pred mean={tp_signal.mean():.3f}, median={tp_signal.median():.3f}, "
          f"5th pct={tp_signal.quantile(0.05):.3f}, min-nonzero-gt-vol={results_df.loc[is_gt_nonzero, gt_col].min():.3f}")

**Result: the noise-floor hypothesis is rejected.** For 4 of the 5 hemorrhage classes, the *median* predicted volume is ~0 even on series with real, non-zero ground-truth hemorrhage — meaning the model under-predicts genuine cases at least as much as it adds noise on healthy ones. A simple minimum-volume cutoff would only make recall worse. Cross-referencing this with the v1 training log points to the real root cause: **premature early stopping under severe class imbalance** (background ≈99.4% of pixels), which never let the rare hemorrhage classes converge.


**Following the root cause to its source:** premature early stopping under
class imbalance means the *loss function itself* never gave the rare
hemorrhage classes enough gradient signal relative to background before
`val_dice` (also background-dominated in aggregate, despite excluding class
0 from the *reported* metric -- see Section 17's note) triggered early
stopping. The fix has to happen at the loss level, not the stopping
criterion. Before guessing weight values, the cell below measures the real
per-class pixel frequency directly, to compute principled inverse-frequency
weights rather than hand-picked ones -- the same "measure, don't guess"
discipline seen in the MLS-binning diagnostic (Section 21.4).


In [ ]:
# Diagnostic: compute real per-class pixel frequency across the training masks,
# to derive principled (not guessed) inverse-frequency CE weights.
import numpy as np
from collections import Counter

class_pixel_counts = Counter()
sample_ds = seg_dataset.SegmentationSliceDataset(
    full_slice_df[full_slice_df["fold"] != 0].reset_index(drop=True),  # any fold's train split
    dicom_dir, annotations_dir
)

# Sampling a subset for speed is fine here -- we just need the ratio, not an exact count.
import random
random.seed(42)
sample_indices = random.sample(range(len(sample_ds)), min(500, len(sample_ds)))

for idx in sample_indices:
    mask = sample_ds[idx]["mask"].numpy()
    values, counts = np.unique(mask, return_counts=True)
    for v, c in zip(values, counts):
        class_pixel_counts[int(v)] += int(c)

total_pixels = sum(class_pixel_counts.values())
print("Class pixel frequency (sampled):")
class_names = ["Background", "IVH", "IPH", "SDH", "EDH", "SAH"]  # per CLASS_VALUE_TO_TARGET_KEY mapping
for c in range(6):
    count = class_pixel_counts.get(c, 0)
    freq = count / total_pixels
    print(f"  {c} ({class_names[c]}): {count} pixels, {freq*100:.4f}%")

## 18. Segmentation v2: Class-Balanced Loss + Training-Stability Fixes

Six targeted changes address the issues found above:

1. `DiceLoss` restricted to the 5 foreground (hemorrhage) classes — excluding background, which otherwise dilutes the signal since it's trivially easy to segment.
2. `CrossEntropyLoss` weighted by **square-root inverse pixel frequency** per class (computed just above). A plain inverse-frequency weight is too extreme here (~7000:1 ratio between the rarest and most common class) and risks destabilizing training; the square root tames this while preserving the relative ordering between classes.
3. Longer training budget (`N_EPOCHS`: 12 → 25) and higher patience (`PATIENCE`: 4 → 8), so rare classes have enough time to converge.
4. A 3-epoch moving-average smoothing of `val_dice` for early-stopping and best-checkpoint selection, so a single noisy epoch can no longer trigger an early stop or get saved as "best".
5. Gradient clipping (`max_norm=1.0`) to prevent the kind of loss spike seen in v1's fold 2.
6. A cosine-annealing learning-rate schedule for a more stable final-epoch fine-tuning phase.

Checkpoints are written to a new `checkpoints_seg_v2/` root so the v1 results remain intact for comparison.


In [ ]:
# Cell: Segmentation training v2 -- addresses (1) class-imbalanced loss,
# (2) noisy early-stopping, (3) gradient instability, (4) fixed LR.
# Writes to a NEW checkpoint root so v1 results are preserved for comparison.
import os
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp
from torch.utils.data import DataLoader

N_FOLDS = 5
N_EPOCHS = 25          # was 12 -- give rare classes enough time to converge
BATCH_SIZE = 8
NUM_WORKERS = 2
PATIENCE = 8            # was 4 -- avoid stopping on noisy single-epoch dips
EMA_WINDOW = 3           # smooth val_dice over the last 3 epochs before comparing
GRAD_CLIP_MAX_NORM = 1.0

# NEW checkpoint root -- keeps v1 results intact for before/after comparison.
CHECKPOINT_ROOT = "/content/drive/MyDrive/iaaa-brain-ct-triage/checkpoints_seg_v2"
LOG_PATH = "/content/drive/MyDrive/iaaa-brain-ct-triage/logs/training_log_seg_v2.csv"
os.makedirs(os.path.dirname(LOG_PATH), exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

full_slice_df = seg_dataset.build_slice_df_with_folds(metadata_df, series_targets_df)
full_slice_df = seg_dataset.filter_to_available_annotations(full_slice_df, annotations_dir)

log_rows = pd.read_csv(LOG_PATH).to_dict("records") if os.path.isfile(LOG_PATH) else []


def append_log(fold, epoch, train_loss, val_loss, val_dice, val_pixel_acc, lr):
    log_rows.append({
        "fold": fold, "epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
        "val_dice": val_dice, "val_pixel_acc": val_pixel_acc, "lr": lr,
        "timestamp": pd.Timestamp.now().isoformat(),
    })
    pd.DataFrame(log_rows).to_csv(LOG_PATH, index=False)


def smoothed_metric(history, key="val_dice", window=EMA_WINDOW):
    """Simple moving average over the last `window` epochs, so early-stopping
    and best-checkpoint selection react to a trend rather than single-epoch noise."""
    vals = [h[key] for h in history]
    if len(vals) < window:
        return float(np.mean(vals))
    return float(np.mean(vals[-window:]))


def epochs_since_best(history, key="val_dice", window=EMA_WINDOW):
    if len(history) < window:
        return 0
    smoothed_series = [
        float(np.mean([h[key] for h in history[max(0, i - window + 1):i + 1]]))
        for i in range(len(history))
    ]
    best_val = max(smoothed_series)
    last_best_idx = max(i for i, v in enumerate(smoothed_series) if v == best_val)
    return (len(history) - 1) - last_best_idx


@torch.no_grad()
def mean_dice_score(logits, target_mask, n_classes=6, eps=1e-6):
    """Unchanged from v1 -- excludes background, matches the definition
    used throughout this project's validation."""
    pred = logits.argmax(dim=1)
    dice_scores = []
    for c in range(1, n_classes):
        pred_c = (pred == c)
        target_c = (target_mask == c)
        if target_c.sum() == 0 and pred_c.sum() == 0:
            continue
        intersection = (pred_c & target_c).float().sum()
        dice = (2 * intersection + eps) / (pred_c.float().sum() + target_c.float().sum() + eps)
        dice_scores.append(dice.item())
    return float(np.mean(dice_scores)) if dice_scores else 1.0


# --- Compute sqrt-inverse-frequency CE weights ONCE, from the full training
# pool (all folds combined), matching the diagnostic script's methodology. ---
def compute_class_weights(slice_df, dicom_dir, annotations_dir, n_classes=6, sample_size=800, seed=42):
    import random
    random.seed(seed)
    ds = seg_dataset.SegmentationSliceDataset(slice_df, dicom_dir, annotations_dir)
    sample_indices = random.sample(range(len(ds)), min(sample_size, len(ds)))

    class_pixel_counts = np.zeros(n_classes, dtype=np.int64)
    for idx in sample_indices:
        mask = ds[idx]["mask"].numpy()
        values, counts = np.unique(mask, return_counts=True)
        for v, c in zip(values, counts):
            class_pixel_counts[int(v)] += int(c)

    total_pixels = class_pixel_counts.sum()
    raw_weights = total_pixels / (n_classes * class_pixel_counts.astype(np.float64))
    sqrt_weights = np.sqrt(raw_weights)
    return torch.tensor(sqrt_weights, dtype=torch.float32)


print("Computing class weights from training pool...")
class_weights = compute_class_weights(full_slice_df, dicom_dir, annotations_dir)
print(f"Class weights (sqrt inverse-frequency): {class_weights.tolist()}")


class SegmentationLossV2(nn.Module):
    """v2: Dice restricted to foreground classes (1-5), CE weighted by
    sqrt-inverse pixel frequency -- both changes target the SAME problem
    (rare-class under-prediction) from the two different loss terms."""

    def __init__(self, class_weights: torch.Tensor, dice_weight: float = 0.5, ce_weight: float = 0.5):
        super().__init__()
        self.dice_weight = dice_weight
        self.ce_weight = ce_weight
        self.dice_loss = smp.losses.DiceLoss(mode="multiclass", from_logits=True, classes=[1, 2, 3, 4, 5])
        self.ce_loss = nn.CrossEntropyLoss(weight=class_weights)

    def forward(self, logits, target_mask):
        dice = self.dice_loss(logits, target_mask)
        ce = self.ce_loss(logits, target_mask)
        total = self.dice_weight * dice + self.ce_weight * ce
        return {"total": total, "dice": dice.detach(), "ce": ce.detach()}


for fold in range(N_FOLDS):
    print(f"\n{'=' * 60}\nSEGMENTATION FOLD {fold} (v2)\n{'=' * 60}")

    fold_dir = os.path.join(CHECKPOINT_ROOT, f"fold_{fold}")
    os.makedirs(fold_dir, exist_ok=True)
    last_ckpt_path = os.path.join(fold_dir, "last_checkpoint.pt")
    best_ckpt_path = os.path.join(fold_dir, "best_model.pt")
    done_marker_path = os.path.join(fold_dir, "DONE")

    if os.path.isfile(done_marker_path):
        print(f"Fold {fold} already marked DONE. Skipping.")
        continue

    train_slice_df = full_slice_df[full_slice_df["fold"] != fold].reset_index(drop=True)
    val_slice_df = full_slice_df[full_slice_df["fold"] == fold].reset_index(drop=True)

    train_ds = seg_dataset.SegmentationSliceDataset(train_slice_df, dicom_dir, annotations_dir)
    val_ds = seg_dataset.SegmentationSliceDataset(val_slice_df, dicom_dir, annotations_dir)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)

    net = seg_model.HemorrhageSegModel(pretrained=True).to(DEVICE)
    loss_fn = SegmentationLossV2(class_weights.to(DEVICE))
    optimizer = torch.optim.Adam(net.parameters(), lr=3e-4, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS, eta_min=1e-6)

    start_epoch = 0
    best_metric = -1.0
    history = []

    if os.path.isfile(last_ckpt_path):
        ckpt = torch.load(last_ckpt_path, map_location=DEVICE, weights_only=False)
        net.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])
        start_epoch = ckpt["epoch"] + 1
        best_metric = ckpt["best_metric"]
        history = ckpt["history"]
        print(f"Resuming fold {fold} from epoch {start_epoch} (best smoothed val_dice so far: {best_metric:.4f})")
        if start_epoch >= N_EPOCHS:
            print(f"Fold {fold} already completed. Skipping.")
            continue

    for epoch in range(start_epoch, N_EPOCHS):
        t0 = time.time()

        # --- train ---
        net.train()
        train_loss_total, n_train_batches = 0.0, 0
        for batch in train_loader:
            images = batch["image"].to(DEVICE)
            masks = batch["mask"].to(DEVICE)
            optimizer.zero_grad()
            logits = net(images)
            loss_dict = loss_fn(logits, masks)
            loss_dict["total"].backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=GRAD_CLIP_MAX_NORM)
            optimizer.step()
            train_loss_total += loss_dict["total"].item()
            n_train_batches += 1
        train_loss = train_loss_total / max(n_train_batches, 1)
        scheduler.step()

        # --- validate ---
        net.eval()
        val_loss_total, n_val_batches = 0.0, 0
        dice_scores, pixel_accs = [], []
        with torch.no_grad():
            for batch in val_loader:
                images = batch["image"].to(DEVICE)
                masks = batch["mask"].to(DEVICE)
                logits = net(images)
                loss_dict = loss_fn(logits, masks)
                val_loss_total += loss_dict["total"].item()
                n_val_batches += 1
                dice_scores.append(mean_dice_score(logits, masks))
                pred = logits.argmax(dim=1)
                pixel_accs.append((pred == masks).float().mean().item())
        val_loss = val_loss_total / max(n_val_batches, 1)
        val_dice = float(np.mean(dice_scores))
        val_pixel_acc = float(np.mean(pixel_accs))
        current_lr = optimizer.param_groups[0]["lr"]

        elapsed = time.time() - t0
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                         "val_dice": val_dice, "val_pixel_acc": val_pixel_acc})
        smoothed_dice = smoothed_metric(history)

        print(f"  epoch {epoch:2d}/{N_EPOCHS - 1}: train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
              f"val_dice={val_dice:.4f}  smoothed_dice={smoothed_dice:.4f}  lr={current_lr:.2e}  ({elapsed:.0f}s)")

        append_log(fold, epoch, train_loss, val_loss, val_dice, val_pixel_acc, current_lr)

        torch.save({"epoch": epoch, "model_state": net.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "scheduler_state": scheduler.state_dict(),
                    "best_metric": best_metric, "history": history}, last_ckpt_path)

        if smoothed_dice > best_metric:
            best_metric = smoothed_dice
            torch.save({"epoch": epoch, "model_state": net.state_dict(),
                        "optimizer_state": optimizer.state_dict(),
                        "scheduler_state": scheduler.state_dict(),
                        "best_metric": best_metric, "history": history}, best_ckpt_path)
            print(f"    New best smoothed val_dice for fold {fold}: {best_metric:.4f} -- saved best_model.pt")

        if epochs_since_best(history) >= PATIENCE:
            print(f"  Early stopping: smoothed val_dice has not improved for {PATIENCE} epochs.")
            break

    with open(done_marker_path, "w") as f:
        f.write(f"best_smoothed_val_dice={best_metric:.4f}\n")
    print(f"Fold {fold} finished. Best smoothed val_dice: {best_metric:.4f}")

print("\nAll segmentation v2 folds complete.\n")
final_log_df = pd.DataFrame(log_rows)
summary = final_log_df.groupby("fold")[["val_dice", "val_pixel_acc"]].max()
print(summary)
print(f"\nMean val_dice across folds: {summary['val_dice'].mean():.4f}")

### v2 Segmentation Training Result

`mean val_dice ≈ 0.506` across the 5 folds (up from `0.392` in v1, a ~29% relative improvement), with visibly more stable, monotonic training curves and no repeat of the v1 fold-2 loss spike.

**One known edge case:** in fold 1, the smoothing mechanism from fix #4 above saved an under-trained epoch-0 checkpoint as "best", because a genuinely better epoch (epoch 8, raw `val_dice=0.383`) appeared right before early stopping triggered and never had a chance to become the 3-epoch smoothed maximum. The cell below promotes the correct `last_checkpoint.pt` for that fold before evaluation. This is a real trade-off of the smoothing fix (it trades away some responsiveness to genuine, late-appearing improvements in exchange for ignoring noise) and is worth revisiting if this pattern recurs in future retraining runs.


**Worth calling out for the portfolio:** this edge case is a smaller, single-fold instance of the exact same class of problem diagnosed at scale in Section 21.7 (a checkpoint-selection mechanism interacting badly with a specific training run). Documenting it transparently here, and fixing it by hand with a clear trail (cell below) rather than quietly re-running until it looks better, is exactly the kind of honesty a real engineering log should have -- and precedent for building the more general retrospective-reselection fix later.

In [ ]:
# Quick fix: for fold 1 specifically, the smoothing mechanism saved an
# undertrained epoch-0 checkpoint as "best" while a genuinely better model
# (epoch 8, raw val_dice=0.383) sits in last_checkpoint.pt, never promoted
# because it appeared right before early stopping triggered.
import shutil

fold1_dir = f"{CHECKPOINT_ROOT}/fold_1"
last_ckpt = torch.load(f"{fold1_dir}/last_checkpoint.pt", map_location=DEVICE, weights_only=False)
print(f"last_checkpoint.pt for fold 1: epoch={last_ckpt['epoch']}, "
      f"raw val_dice at that epoch={last_ckpt['history'][-1]['val_dice']:.4f}")

# If this confirms epoch 8 / dice 0.383, promote it to be the "best" used downstream.
shutil.copy(f"{fold1_dir}/last_checkpoint.pt", f"{fold1_dir}/best_model.pt")
print("Promoted last_checkpoint.pt to best_model.pt for fold 1.")

In [ ]:
# Re-run the SAME volume-level validation pipeline, now against v2 checkpoints.
CHECKPOINT_ROOT_V2 = "/content/drive/MyDrive/iaaa-brain-ct-triage/checkpoints_seg_v2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_SIZE = 224

all_rows_v2 = []

for fold in range(5):
    print(f"Evaluating fold {fold} (v2)...")
    best_ckpt_path = f"{CHECKPOINT_ROOT_V2}/fold_{fold}/best_model.pt"
    ckpt = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)

    net = seg_model.HemorrhageSegModel(pretrained=False).to(DEVICE)
    net.load_state_dict(ckpt["model_state"])
    net.eval()

    val_slice_df = full_slice_df[full_slice_df["fold"] == fold].reset_index(drop=True)
    val_ds = seg_dataset.SegmentationSliceDataset(
        val_slice_df, dicom_dir, annotations_dir, image_size=IMAGE_SIZE
    )

    series_slices = defaultdict(list)
    for i in range(len(val_ds)):
        item = val_ds[i]
        series_slices[item["series_id"]].append(item)

    for series_id, slices in series_slices.items():
        pred_volumes = volume_aggregation.predict_series_volumes(
            net, slices, DEVICE, image_size=IMAGE_SIZE
        )
        series_meta_rows = metadata_df[metadata_df["dicom_series.id"] == series_id]
        gt_intermediates = aggregate_series_intermediates(series_meta_rows)

        isolated_intermediates = {
            **pred_volumes,
            "MLS_mm": gt_intermediates["MLS_mm"],
            "fracture_prob": gt_intermediates["fracture_prob"],
        }

        pred_class = triage_from_intermediates(isolated_intermediates)
        true_class = triage_from_intermediates(gt_intermediates)

        row = {"series_id": series_id, "fold": fold,
               "true_triage": true_class, "pred_triage_from_seg": pred_class}
        for key in ["V_EDH", "V_SDH", "V_IPH", "V_SAH", "V_IVH"]:
            row[f"{key}_pred"] = pred_volumes[key]
            row[f"{key}_gt"] = gt_intermediates[key]
        all_rows_v2.append(row)

results_df_v2 = pd.DataFrame(all_rows_v2)

print(f"\nTotal series evaluated: {len(results_df_v2)}")

print("\n=== Per-subtype volume MAE (mL) — v2 ===")
for key in ["V_EDH", "V_SDH", "V_IPH", "V_SAH", "V_IVH"]:
    mae = (results_df_v2[f"{key}_pred"] - results_df_v2[f"{key}_gt"]).abs().mean()
    mae_v1 = (results_df[f"{key}_pred"] - results_df[f"{key}_gt"]).abs().mean()
    delta = mae - mae_v1
    arrow = "↓ better" if delta < 0 else "↑ worse"
    print(f"  {key}: v1={mae_v1:.3f}mL  v2={mae:.3f}mL  ({arrow}, Δ={delta:+.3f})")

accuracy_v2 = (results_df_v2["pred_triage_from_seg"] == results_df_v2["true_triage"]).mean()
f1_macro_v2 = f1_score(results_df_v2["true_triage"], results_df_v2["pred_triage_from_seg"], average="macro")

print(f"\n=== Isolated triage accuracy/f1_macro — v2 ===")
print(f"Accuracy: {accuracy_v2:.4f}  (v1 was 0.7929)")
print(f"f1_macro: {f1_macro_v2:.4f}  (v1 was 0.6669)")

print("\nPer-class report (v2):")
print(classification_report(results_df_v2["true_triage"], results_df_v2["pred_triage_from_seg"],
                              target_names=["Non-Urgent", "Urgent", "Critical"]))

print("\nConfusion matrix (v2, rows=true, cols=predicted):")
print(pd.crosstab(results_df_v2["true_triage"], results_df_v2["pred_triage_from_seg"]))

# Re-run the same over-triage drill-down to see if the IPH false-positive noise improved.
over_triage_mask_v2 = (results_df_v2["true_triage"] == 0) & (results_df_v2["pred_triage_from_seg"] == 1)
over_triage_cases_v2 = results_df_v2[over_triage_mask_v2]
print(f"\n{len(over_triage_cases_v2)} over-triage (Non-Urgent -> Urgent) cases in v2 "
      f"(was 18 in v1). Predicted vs GT volume per subtype:")
for key in ["V_EDH", "V_SDH", "V_IPH", "V_SAH", "V_IVH"]:
    pred_mean = over_triage_cases_v2[f"{key}_pred"].mean() if len(over_triage_cases_v2) else float("nan")
    gt_mean = over_triage_cases_v2[f"{key}_gt"].mean() if len(over_triage_cases_v2) else float("nan")
    print(f"  {key}: pred_mean={pred_mean:.2f} mL   gt_mean={gt_mean:.2f} mL")

## 19. Segmentation v1 vs. v2 — Final Comparison

| Metric | v1 | v2 | Δ |
|---|---|---|---|
| Isolated accuracy | 0.7929 | 0.8788 | +0.086 |
| Isolated f1_macro | 0.6669 | 0.8401 | +0.173 (~26% relative) |
| Non-Urgent over-triage cases | 18 | 8 | −10 |
| Critical under-triage errors | 0 | 0 | unchanged (safety-critical, preserved) |

The class-balanced loss and training-stability fixes produced a clear, measurable improvement across every class simultaneously (not a trade-off between classes), and the most important clinical-safety property — zero Critical cases ever mislabeled as Non-Urgent — held in both versions. `V_SAH` was the one class whose MAE slightly worsened (a known, noted trade-off from the aggressive class weighting on the rarest class), but the net effect on triage accuracy was clearly positive.

**Decision:** segmentation is no longer the main bottleneck. Given diminishing returns, further effort now moves to the part of the pipeline that has not been modeled yet: predicting `MLS_mm` and `fracture_prob` directly from imaging (rather than from ground truth, as in the isolated experiments above) and assembling the final hybrid submission pipeline.


## 20. Next Steps

1. Build `IntermediatesModel`: a 2.5D CNN + physical-embedding architecture predicting `MLS_mm` (hard-max pooling, with LogSumExp smooth-max available as a fallback if training proves unstable) and `fracture_prob` (complement-product / soft-OR pooling) directly from imaging, trained on all 338 series (no pixel-annotation constraint here, since `MLS_mm`/`SkullFracture` labels exist for every series).
2. Validate this model in isolation (ground-truth volumes + predicted MLS/fracture), mirroring the segmentation validation methodology above.
3. Assemble the final hybrid pipeline: segmentation v2 (5 hemorrhage volumes) + `IntermediatesModel` (MLS + fracture) → `triage_from_intermediates`.
4. Rebuild and smoke-test `submission.py` for the hybrid pipeline (same rigor as Section 11).
5. Submit and compare the new `Score` against the current best (`f1_macro=0.343`, MIL v1).


## 21. IntermediatesModel — MLS regression & fracture classification

**Objective:** predict two of the seven triage intermediates directly from CT slices — `MLS_mm` (midline shift, regression) and `fracture_prob` (skull fracture, binary classification) — using a shared ResNet18 backbone with a small physical-embedding MLP (slice thickness + relative z-position).

**Structure of this section:**
- 21.1 Dataset & model reload
- 21.2 v1 training (5-fold, checkpoint selection = fracture AUC only)
- 21.3 v1 isolated triage validation (GT volumes + predicted MLS/fracture)
- 21.4 Diagnosing regression-to-the-mean in MLS and designing a weighted-loss fix (v2)
- 21.5 v2 training (5-fold, checkpoint selection = blended quality score)
- 21.6 Naive v1-vs-v2 comparison — confounded, kept for the historical record only
- 21.7 Fair v1-vs-v2 comparison (retrospective checkpoint re-selection) — the number that should actually be trusted


### 21.1 Dataset & model reload

In [ ]:
%%writefile /content/drive/MyDrive/iaaa-brain-ct-triage/src/intermediates_dataset.py
"""Per-slice dataset for IntermediatesModel: loads a 2.5D multi-window CT
image plus physical metadata (slice thickness, relative z-position from
ImagePositionPatient) and the REAL per-slice MLS/fracture labels.
"""
import os

import cv2
import numpy as np
import pandas as pd
import pydicom
import torch
from torch.utils.data import Dataset

from src.preprocessing import to_hu, apply_window, WINDOW_PRESETS


class IntermediatesSliceDataset(Dataset):
    def __init__(self, slice_df, dicom_root: str, image_size: int = 224):
        self.slice_df = slice_df.reset_index(drop=True)
        self.dicom_root = dicom_root
        self.image_size = image_size

    def __len__(self) -> int:
        return len(self.slice_df)

    def __getitem__(self, idx: int) -> dict:
        row = self.slice_df.iloc[idx]
        series_id = row["dicom_series.id"]
        sop_uid = row["dicom_series.SOPInstanceUID"]

        dcm_path = os.path.join(self.dicom_root, str(series_id), f"{sop_uid}.dcm")
        dcm = pydicom.dcmread(dcm_path)
        hu_image = to_hu(dcm)
        channels = [apply_window(hu_image, center, width) for center, width in WINDOW_PRESETS.values()]
        image = np.stack(channels, axis=0).astype(np.float32)

        image_resized = cv2.resize(
            image.transpose(1, 2, 0), (self.image_size, self.image_size),
            interpolation=cv2.INTER_AREA,
        ).transpose(2, 0, 1)

        return {
            "image": torch.from_numpy(image_resized),
            "slice_thickness": float(row["dicom_series.SliceThickness"]),
            "z_rel": float(row["z_rel"]),
            "mls_target": float(row["MidlineShiftMM"]),
            "fracture_target": float(row["SkullFracture"]),
            "series_id": int(series_id),
        }


def build_slice_df_with_z_rel(metadata_df, series_targets_df, dicom_root: str) -> pd.DataFrame:
    slice_df = metadata_df.merge(
        series_targets_df[["series_id", "fold"]],
        left_on="dicom_series.id", right_on="series_id", how="inner",
    )

    z_positions = []
    for _, row in slice_df.iterrows():
        dcm_path = os.path.join(
            dicom_root, str(row["dicom_series.id"]),
            f"{row['dicom_series.SOPInstanceUID']}.dcm",
        )
        dcm = pydicom.dcmread(dcm_path, stop_before_pixels=True)
        z_positions.append(float(dcm.ImagePositionPatient[2]))
    slice_df["z_position"] = z_positions

    z_min = slice_df.groupby("dicom_series.id")["z_position"].transform("min")
    z_max = slice_df.groupby("dicom_series.id")["z_position"].transform("max")
    z_range = (z_max - z_min).replace(0, 1.0)
    slice_df["z_rel"] = (slice_df["z_position"] - z_min) / z_range

    return slice_df

In [ ]:
# Reload load_src.py's module registry so it picks up the 3 new files,
# then import the classes/functions we need for the training cell.
if "load_src" in sys.modules:
    del sys.modules["load_src"]
from load_src import load_src_package
modules = load_src_package()
print("Loaded modules:", sorted(modules.keys()))

# If intermediates_model/losses/dataset don't appear in this list,
# load_src.py likely has a hardcoded module-name list that needs the 3
# new names added -- tell me and I'll help locate/edit that list.
IntermediatesModel = modules["intermediates_model"].IntermediatesModel
IntermediatesLoss = modules["intermediates_losses"].IntermediatesLoss
IntermediatesSliceDataset = modules["intermediates_dataset"].IntermediatesSliceDataset
build_slice_df_with_z_rel = modules["intermediates_dataset"].build_slice_df_with_z_rel

print("PASS: all 3 new intermediates modules loaded successfully.")

### 21.2 v1 training (5-fold)

**Checkpoint selection criterion (v1): smoothed `val_fracture_auc` only.**
This choice matters later -- see 21.6/21.7.


### Full 5-Fold Training

Reuses the same stabilization recipe validated in segmentation v2 (gradient clipping, cosine LR schedule, 3-epoch smoothed early-stopping/checkpointing), and logs `mls_loss`/`fracture_loss` separately rather than only combined `val_loss`, so any future divergence between the two heads is visible directly in the log rather than requiring a fresh diagnostic pass.


In [ ]:
import os
CHECKPOINT_ROOT = "/content/drive/MyDrive/iaaa-brain-ct-triage/checkpoints_intermediates"
for fold in range(5):
    fold_dir = f"{CHECKPOINT_ROOT}/fold_{fold}"
    if os.path.isdir(fold_dir):
        files = os.listdir(fold_dir)
        print(f"fold_{fold}: EXISTS -> {files}")
    else:
        print(f"fold_{fold}: MISSING")

In [ ]:
# Cell: IntermediatesModel training -- full 5-fold cross-validation.
# FIX: the automatic checkpoint/log deletion that lived at the top of this
# cell has been REMOVED. That deletion was only ever meant to run ONCE, to
# clear out checkpoints from the pre-Softplus-fix (buggy ReLU) run -- but
# living inside this reusable/resumable cell, it silently wiped completed
# folds every time the cell was re-run (e.g. after a Colab disconnect).
# If you ever need a truly fresh run, do so by clearing the checkpoint
# folder EXPLICITLY and separately (see the one-off cell below), never as
# part of this training loop.
import os
import time

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, mean_absolute_error

N_FOLDS = 5
N_EPOCHS = 20
BATCH_SIZE = 16
NUM_WORKERS = 2
PATIENCE = 8
EMA_WINDOW = 3
GRAD_CLIP_MAX_NORM = 1.0
FRACTURE_POS_WEIGHT = 27.88

CHECKPOINT_ROOT = "/content/drive/MyDrive/iaaa-brain-ct-triage/checkpoints_intermediates"
LOG_PATH = "/content/drive/MyDrive/iaaa-brain-ct-triage/logs/training_log_intermediates.csv"
os.makedirs(os.path.dirname(LOG_PATH), exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

full_slice_df = build_slice_df_with_z_rel(metadata_df, series_targets_df, dicom_dir)
print(f"Total slices: {len(full_slice_df)}  |  Series: {full_slice_df['dicom_series.id'].nunique()}")

# IMPORTANT: load existing log rows instead of starting log_rows = [] blindly,
# so re-running this cell APPENDS to history rather than losing it.
log_rows = pd.read_csv(LOG_PATH).to_dict("records") if os.path.isfile(LOG_PATH) else []


def append_log(fold, epoch, train_loss, val_loss, val_mls_loss, val_fracture_loss,
                val_mls_mae, val_fracture_auc, lr):
    log_rows.append({
        "fold": fold, "epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
        "val_mls_loss": val_mls_loss, "val_fracture_loss": val_fracture_loss,
        "val_mls_mae": val_mls_mae, "val_fracture_auc": val_fracture_auc, "lr": lr,
        "timestamp": pd.Timestamp.now().isoformat(),
    })
    pd.DataFrame(log_rows).to_csv(LOG_PATH, index=False)


def smoothed_metric(history, key="val_fracture_auc", window=EMA_WINDOW):
    vals = [h[key] for h in history]
    if len(vals) < window:
        return float(np.mean(vals))
    return float(np.mean(vals[-window:]))


def epochs_since_best(history, key="val_fracture_auc", window=EMA_WINDOW):
    if len(history) < window:
        return 0
    smoothed_series = [
        float(np.mean([h[key] for h in history[max(0, i - window + 1):i + 1]]))
        for i in range(len(history))
    ]
    best_val = max(smoothed_series)
    last_best_idx = max(i for i, v in enumerate(smoothed_series) if v == best_val)
    return (len(history) - 1) - last_best_idx


for fold in range(N_FOLDS):
    print(f"\n{'=' * 60}\nINTERMEDIATES FOLD {fold}\n{'=' * 60}")

    fold_dir = os.path.join(CHECKPOINT_ROOT, f"fold_{fold}")
    os.makedirs(fold_dir, exist_ok=True)
    last_ckpt_path = os.path.join(fold_dir, "last_checkpoint.pt")
    best_ckpt_path = os.path.join(fold_dir, "best_model.pt")
    done_marker_path = os.path.join(fold_dir, "DONE")

    if os.path.isfile(done_marker_path):
        print(f"Fold {fold} already marked DONE. Skipping.")
        continue

    train_slice_df = full_slice_df[full_slice_df["fold"] != fold].reset_index(drop=True)
    val_slice_df = full_slice_df[full_slice_df["fold"] == fold].reset_index(drop=True)

    train_ds = IntermediatesSliceDataset(train_slice_df, dicom_dir)
    val_ds = IntermediatesSliceDataset(val_slice_df, dicom_dir)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)

    net = IntermediatesModel(pretrained=True, fracture_prior_prob=0.02).to(DEVICE)
    loss_fn = IntermediatesLoss(fracture_pos_weight=FRACTURE_POS_WEIGHT)
    optimizer = torch.optim.Adam(net.parameters(), lr=3e-4, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS, eta_min=1e-6)

    start_epoch = 0
    best_metric = -1.0
    history = []

    if os.path.isfile(last_ckpt_path):
        ckpt = torch.load(last_ckpt_path, map_location=DEVICE, weights_only=False)
        net.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])
        start_epoch = ckpt["epoch"] + 1
        best_metric = ckpt["best_metric"]
        history = ckpt["history"]
        print(f"Resuming fold {fold} from epoch {start_epoch} (best smoothed fracture AUC: {best_metric:.4f})")
        if start_epoch >= N_EPOCHS:
            print(f"Fold {fold} already completed. Skipping.")
            continue

    for epoch in range(start_epoch, N_EPOCHS):
        t0 = time.time()

        net.train()
        train_loss_total, n_train_batches = 0.0, 0
        for batch in train_loader:
            images = batch["image"].to(DEVICE)
            slice_thickness = batch["slice_thickness"].float().to(DEVICE)
            z_rel = batch["z_rel"].float().to(DEVICE)
            mls_target = batch["mls_target"].float().to(DEVICE)
            fracture_target = batch["fracture_target"].float().to(DEVICE)

            optimizer.zero_grad()
            out = net.forward_per_slice(images, slice_thickness, z_rel)
            loss_dict = loss_fn(out["mls_logit"], mls_target, out["fracture_logit"], fracture_target)
            loss_dict["total"].backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=GRAD_CLIP_MAX_NORM)
            optimizer.step()
            train_loss_total += loss_dict["total"].item()
            n_train_batches += 1
        train_loss = train_loss_total / max(n_train_batches, 1)
        scheduler.step()

        net.eval()
        val_loss_total, val_mls_loss_total, val_fracture_loss_total, n_val_batches = 0.0, 0.0, 0.0, 0
        all_mls_pred, all_mls_true = [], []
        all_fracture_prob, all_fracture_true = [], []
        with torch.no_grad():
            for batch in val_loader:
                images = batch["image"].to(DEVICE)
                slice_thickness = batch["slice_thickness"].float().to(DEVICE)
                z_rel = batch["z_rel"].float().to(DEVICE)
                mls_target = batch["mls_target"].float().to(DEVICE)
                fracture_target = batch["fracture_target"].float().to(DEVICE)

                out = net.forward_per_slice(images, slice_thickness, z_rel)
                loss_dict = loss_fn(out["mls_logit"], mls_target, out["fracture_logit"], fracture_target)
                val_loss_total += loss_dict["total"].item()
                val_mls_loss_total += loss_dict["mls_loss"].item()
                val_fracture_loss_total += loss_dict["fracture_loss"].item()
                n_val_batches += 1

                all_mls_pred.append(IntermediatesModel.mls_from_logit(out["mls_logit"]).cpu().numpy())
                all_mls_true.append(mls_target.cpu().numpy())
                all_fracture_prob.append(torch.sigmoid(out["fracture_logit"]).cpu().numpy())
                all_fracture_true.append(fracture_target.cpu().numpy())

        val_loss = val_loss_total / max(n_val_batches, 1)
        val_mls_loss = val_mls_loss_total / max(n_val_batches, 1)
        val_fracture_loss = val_fracture_loss_total / max(n_val_batches, 1)
        val_mls_mae = mean_absolute_error(np.concatenate(all_mls_true), np.concatenate(all_mls_pred))

        fracture_true_concat = np.concatenate(all_fracture_true)
        if len(np.unique(fracture_true_concat)) > 1:
            val_fracture_auc = roc_auc_score(fracture_true_concat, np.concatenate(all_fracture_prob))
        else:
            val_fracture_auc = float("nan")

        current_lr = optimizer.param_groups[0]["lr"]
        elapsed = time.time() - t0

        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                         "val_mls_loss": val_mls_loss, "val_fracture_loss": val_fracture_loss,
                         "val_mls_mae": val_mls_mae, "val_fracture_auc": val_fracture_auc})
        smoothed_auc = smoothed_metric(history)

        print(f"  epoch {epoch:2d}/{N_EPOCHS - 1}: train_loss={train_loss:.4f}  "
              f"val_loss={val_loss:.4f} (mls={val_mls_loss:.4f}, frac={val_fracture_loss:.4f})  "
              f"val_mls_mae={val_mls_mae:.4f}  val_fracture_auc={val_fracture_auc:.4f}  "
              f"smoothed_auc={smoothed_auc:.4f}  lr={current_lr:.2e}  ({elapsed:.0f}s)")

        append_log(fold, epoch, train_loss, val_loss, val_mls_loss, val_fracture_loss,
                   val_mls_mae, val_fracture_auc, current_lr)

        torch.save({"epoch": epoch, "model_state": net.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "scheduler_state": scheduler.state_dict(),
                    "best_metric": best_metric, "history": history}, last_ckpt_path)

        if smoothed_auc > best_metric:
            best_metric = smoothed_auc
            torch.save({"epoch": epoch, "model_state": net.state_dict(),
                        "optimizer_state": optimizer.state_dict(),
                        "scheduler_state": scheduler.state_dict(),
                        "best_metric": best_metric, "history": history}, best_ckpt_path)
            print(f"    New best smoothed fracture AUC for fold {fold}: {best_metric:.4f} -- saved best_model.pt")

        if epochs_since_best(history) >= PATIENCE:
            print(f"  Early stopping: smoothed fracture AUC has not improved for {PATIENCE} epochs.")
            break

    with open(done_marker_path, "w") as f:
        f.write(f"best_smoothed_fracture_auc={best_metric:.4f}\n")
    print(f"Fold {fold} finished. Best smoothed fracture AUC: {best_metric:.4f}")

print("\nAll intermediates folds complete.\n")
final_log_df = pd.DataFrame(log_rows)
summary = final_log_df.groupby("fold")[["val_mls_mae", "val_fracture_auc"]].max()
print(summary)

In [ ]:
# Correct reporting: read the metrics from the EXACT epoch that best_model.pt
# was saved at (per the smoothed_auc selection criterion), instead of taking
# a naive column-wise max/min across all epochs -- which is meaningless for
# a mixed-direction metric table (AUC: higher=better, MAE: lower=better).
import torch

rows = []
for fold in range(5):
    ckpt_path = f"{CHECKPOINT_ROOT}/fold_{fold}/best_model.pt"
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    best_epoch_idx = ckpt["epoch"]
    # Find the matching history entry for that exact epoch.
    best_epoch_record = next(h for h in ckpt["history"] if h["epoch"] == best_epoch_idx)
    rows.append({
        "fold": fold,
        "best_epoch": best_epoch_idx,
        "val_mls_mae": best_epoch_record["val_mls_mae"],
        "val_fracture_auc": best_epoch_record["val_fracture_auc"],
        "smoothed_auc_at_save": ckpt["best_metric"],
    })

correct_summary = pd.DataFrame(rows).set_index("fold")
print(correct_summary)
print(f"\nMean val_mls_mae (at best checkpoint): {correct_summary['val_mls_mae'].mean():.4f}")
print(f"Mean val_fracture_auc (at best checkpoint): {correct_summary['val_fracture_auc'].mean():.4f}")

### 21.3 v1 isolated triage validation (GT volumes + predicted MLS/fracture)

In [ ]:
# Cell: isolated triage validation for IntermediatesModel -- mirrors the
# segmentation volume-validation methodology, but with the roles reversed:
# GT volumes held fixed, MLS/fracture predicted by this model.
import torch
import pandas as pd
from collections import defaultdict
from sklearn.metrics import f1_score, classification_report

all_rows_intermediates = []

for fold in range(5):
    print(f"Evaluating fold {fold}...")
    ckpt = torch.load(f"{CHECKPOINT_ROOT}/fold_{fold}/best_model.pt", map_location=DEVICE, weights_only=False)
    net = IntermediatesModel(pretrained=False).to(DEVICE)
    net.load_state_dict(ckpt["model_state"])
    net.eval()

    val_slice_df = full_slice_df[full_slice_df["fold"] == fold].reset_index(drop=True)
    val_ds = IntermediatesSliceDataset(val_slice_df, dicom_dir)

    # Group predictions by series to apply series-level pooling.
    series_mls_preds = defaultdict(list)
    series_fracture_preds = defaultdict(list)

    with torch.no_grad():
        for i in range(len(val_ds)):
            item = val_ds[i]
            image = item["image"].unsqueeze(0).to(DEVICE)
            thickness = torch.tensor([item["slice_thickness"]]).float().to(DEVICE)
            z_rel = torch.tensor([item["z_rel"]]).float().to(DEVICE)

            out = net.forward_per_slice(image, thickness, z_rel)
            mls_nonneg = IntermediatesModel.mls_from_logit(out["mls_logit"]).item()
            fracture_prob = torch.sigmoid(out["fracture_logit"]).item()

            series_mls_preds[item["series_id"]].append(mls_nonneg)
            series_fracture_preds[item["series_id"]].append(fracture_prob)

    for series_id in series_mls_preds:
        mls_tensor = torch.tensor(series_mls_preds[series_id])
        fracture_tensor = torch.tensor(series_fracture_preds[series_id])

        pred_mls_series = IntermediatesModel.aggregate_mls(mls_tensor, mode="hard_max").item()
        pred_fracture_series = IntermediatesModel.aggregate_fracture(fracture_tensor).item()

        series_meta_rows = metadata_df[metadata_df["dicom_series.id"] == series_id]
        gt_intermediates = aggregate_series_intermediates(series_meta_rows)

        # Isolation: GT volumes + PREDICTED MLS/fracture.
        isolated_intermediates = {
            "V_EDH": gt_intermediates["V_EDH"], "V_SDH": gt_intermediates["V_SDH"],
            "V_IPH": gt_intermediates["V_IPH"], "V_SAH": gt_intermediates["V_SAH"],
            "V_IVH": gt_intermediates["V_IVH"],
            "MLS_mm": pred_mls_series,
            "fracture_prob": pred_fracture_series,
        }

        pred_class = triage_from_intermediates(isolated_intermediates)
        true_class = triage_from_intermediates(gt_intermediates)

        all_rows_intermediates.append({
            "series_id": series_id, "fold": fold,
            "true_triage": true_class, "pred_triage_from_intermediates": pred_class,
            "mls_pred": pred_mls_series, "mls_gt": gt_intermediates["MLS_mm"],
            "fracture_pred": pred_fracture_series, "fracture_gt": gt_intermediates["fracture_prob"],
        })

results_intermediates_df = pd.DataFrame(all_rows_intermediates)
print(f"\nTotal series evaluated: {len(results_intermediates_df)}")

accuracy = (results_intermediates_df["pred_triage_from_intermediates"] == results_intermediates_df["true_triage"]).mean()
f1_macro = f1_score(results_intermediates_df["true_triage"], results_intermediates_df["pred_triage_from_intermediates"], average="macro")
print(f"\nIsolated accuracy (GT volumes + predicted MLS/fracture): {accuracy:.4f}")
print(f"Isolated f1_macro: {f1_macro:.4f}")
print(classification_report(results_intermediates_df["true_triage"], results_intermediates_df["pred_triage_from_intermediates"],
                              target_names=["Non-Urgent", "Urgent", "Critical"]))
print("\nConfusion matrix (rows=true, cols=predicted):")
print(pd.crosstab(results_intermediates_df["true_triage"], results_intermediates_df["pred_triage_from_intermediates"]))

In [ ]:
# Drill-down 1: the 7 real under-triage errors (true Urgent -> pred Non-Urgent).
# Since GT volumes were used unchanged, these MUST be cases where the true
# Urgent label came from MLS/fracture alone (not has_ich) -- meaning our
# model genuinely missed a real MLS or fracture signal on these series.
under_triage_mask = (results_intermediates_df["true_triage"] == 1) & \
                     (results_intermediates_df["pred_triage_from_intermediates"] == 0)
under_triage_cases = results_intermediates_df[under_triage_mask]

print(f"{len(under_triage_cases)} under-triage cases (true Urgent -> pred Non-Urgent):")
print(under_triage_cases[["series_id", "fold", "mls_pred", "mls_gt", "fracture_pred", "fracture_gt"]]
      .to_string(index=False))

# Drill-down 2: the 2 concerning over-triage jumps (true Non-Urgent -> pred Critical).
over_triage_jump_mask = (results_intermediates_df["true_triage"] == 0) & \
                          (results_intermediates_df["pred_triage_from_intermediates"] == 2)
over_triage_jump_cases = results_intermediates_df[over_triage_jump_mask]

print(f"\n{len(over_triage_jump_cases)} concerning over-triage cases (true Non-Urgent -> pred Critical):")
print(over_triage_jump_cases[["series_id", "fold", "mls_pred", "mls_gt", "fracture_pred", "fracture_gt"]]
      .to_string(index=False))

# Drill-down 3: broader pattern -- is MLS or fracture the main driver of the
# 29 Non-Urgent -> Urgent cases (the largest error group)?
over_triage_mild_mask = (results_intermediates_df["true_triage"] == 0) & \
                          (results_intermediates_df["pred_triage_from_intermediates"] == 1)
over_triage_mild = results_intermediates_df[over_triage_mild_mask]
print(f"\n{len(over_triage_mild)} mild over-triage cases (true Non-Urgent -> pred Urgent) -- summary stats:")
print(f"  mls_pred: mean={over_triage_mild['mls_pred'].mean():.3f}, "
      f"median={over_triage_mild['mls_pred'].median():.3f} (mls_gt should be ~0.1)")
print(f"  fracture_pred: mean={over_triage_mild['fracture_pred'].mean():.3f}, "
      f"median={over_triage_mild['fracture_pred'].median():.3f} (fracture_gt should be ~0)")

### 21.4 Diagnosing regression-to-the-mean in MLS, designing the fix

**Theory:** `MLS_mm` is heavily right-skewed (~99% of slices near the 0.1mm noise
floor). An unweighted Huber/MSE loss is minimized by predictions that hug the
global mean, which under-estimates rare large-MLS slices and over-estimates
some near-zero slices (the classic bidirectional bias of regression on skewed
targets -- see Yang et al., *Delving into Deep Imbalanced Regression*, ICML 2021).

**Fix:** bin `MLS_mm` around the triage rule's own thresholds and apply a
sqrt-damped inverse-frequency sample weight inside the Huber loss (mirrors the
class-weighting fix already validated for segmentation).


In [ ]:
# Diagnostic: bin MLS_mm into clinically-meaningful ranges and compute real
# frequency per bin, to derive a principled inverse-frequency sample weight
# -- exactly the same technique that fixed segmentation's rare-class problem,
# applied here to a continuous target instead of discrete classes.
import numpy as np
import pandas as pd

# Bin edges chosen around the rule's own decision thresholds (EPS_MLS=1.0,
# COMBO_MLS=3.0, MLS_CRITICAL=5.0), so weighting aligns with what actually
# matters for triage decisions, not an arbitrary uniform binning.
bin_edges = [0, 1.0, 3.0, 5.0, 10.0, np.inf]
bin_labels = ["near_zero [0,1)", "small [1,3)", "medium [3,5)", "large [5,10)", "very_large [10,inf)"]

mls_bins = pd.cut(full_slice_df["MidlineShiftMM"], bins=bin_edges, labels=bin_labels, right=False)
bin_counts = mls_bins.value_counts().reindex(bin_labels)

print("MLS_mm distribution by clinically-relevant bin:")
print(bin_counts)
print(f"\nTotal slices: {len(full_slice_df)}")

n_bins = len(bin_labels)
total = bin_counts.sum()
raw_weights = total / (n_bins * bin_counts)
sqrt_weights = np.sqrt(raw_weights)

print("\nRaw inverse-frequency weight per bin:")
print(raw_weights)
print("\nSquare-root-damped weight per bin (what we'll actually use):")
print(sqrt_weights)

In [ ]:
# Preview of the fix (final weight values to be filled in after seeing the
# real bin frequencies above):
class IntermediatesLossV2(nn.Module):
    """v2: MLS loss uses PER-SAMPLE weights derived from inverse-frequency
    binning of MLS_mm (mirrors segmentation v2's class-weighting fix), so
    rare large-MLS slices contribute proportionally more gradient instead
    of being drowned out by the ~99% of slices near the 0.1mm noise floor.
    Fracture loss unchanged (already well-calibrated via pos_weight + bias-init).
    """

    def __init__(self, fracture_pos_weight: float, mls_bin_edges: list,
                 mls_bin_weights: list, mls_weight: float = 1.0,
                 fracture_weight: float = 1.0, huber_beta: float = 1.0):
        super().__init__()
        self.mls_weight = mls_weight
        self.fracture_weight = fracture_weight
        self.huber_beta = huber_beta
        self.mls_bin_edges = torch.tensor(mls_bin_edges)
        self.mls_bin_weights = torch.tensor(mls_bin_weights)
        self.fracture_loss_fn = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor(fracture_pos_weight)
        )

    def _sample_weight_for_mls(self, mls_target: torch.Tensor) -> torch.Tensor:
        # bucketize returns the bin index for each target value.
        bin_idx = torch.bucketize(mls_target.cpu(), self.mls_bin_edges[1:-1])
        return self.mls_bin_weights[bin_idx].to(mls_target.device)

    def forward(self, mls_logit, mls_target, fracture_logit, fracture_target):
        mls_pred = IntermediatesModel.mls_from_logit(mls_logit)
        # Manual per-sample Huber (SmoothL1Loss has no built-in per-sample
        # weight argument), then weight and reduce ourselves.
        diff = mls_pred - mls_target
        abs_diff = diff.abs()
        quadratic = 0.5 * diff ** 2 / self.huber_beta
        linear = abs_diff - 0.5 * self.huber_beta
        per_sample_huber = torch.where(abs_diff < self.huber_beta, quadratic, linear)

        weights = self._sample_weight_for_mls(mls_target)
        mls_loss = (per_sample_huber * weights).sum() / weights.sum()

        fracture_loss = self.fracture_loss_fn(fracture_logit, fracture_target)
        total = self.mls_weight * mls_loss + self.fracture_weight * fracture_loss
        return {"total": total, "mls_loss": mls_loss.detach(), "fracture_loss": fracture_loss.detach()}

In [ ]:
%%writefile /content/drive/MyDrive/iaaa-brain-ct-triage/src/intermediates_losses.py
"""Loss for IntermediatesModel. v2: MLS loss uses PER-SAMPLE weights derived
from inverse-frequency binning of MLS_mm (mirrors segmentation v2's class-
weighting fix), so rare large-MLS slices contribute proportionally more
gradient instead of being drowned out by the ~99% of slices near the 0.1mm
noise floor. Fracture loss unchanged (already well-calibrated via
pos_weight + bias-init).
"""

import torch
import torch.nn as nn

from src.intermediates_model import IntermediatesModel

# Bin edges chosen around the triage rule's own decision thresholds
# (EPS_MLS=1.0, COMBO_MLS=3.0, MLS_CRITICAL=5.0), so weighting reflects what
# actually matters for the final triage decision, not an arbitrary binning.
MLS_BIN_EDGES = [1.0, 3.0, 5.0, 10.0]  # interior edges (bucketize convention)
# sqrt-inverse-frequency weights, computed from the real per-bin slice counts:
# near_zero[0,1), small[1,3), medium[3,5), large[5,10), very_large[10,inf)
MLS_BIN_WEIGHTS = [0.497374, 1.734710, 2.345286, 2.068348, 2.183342]


class IntermediatesLoss(nn.Module):
    def __init__(self, fracture_pos_weight: float,
                 mls_bin_edges: list = MLS_BIN_EDGES,
                 mls_bin_weights: list = MLS_BIN_WEIGHTS,
                 mls_weight: float = 1.0, fracture_weight: float = 1.0,
                 huber_beta: float = 1.0):
        super().__init__()
        self.mls_weight = mls_weight
        self.fracture_weight = fracture_weight
        self.huber_beta = huber_beta
        self.register_buffer("mls_bin_edges", torch.tensor(mls_bin_edges))
        self.register_buffer("mls_bin_weights", torch.tensor(mls_bin_weights))
        self.fracture_loss_fn = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor(fracture_pos_weight)
        )

    def _sample_weight_for_mls(self, mls_target: torch.Tensor) -> torch.Tensor:
        bin_idx = torch.bucketize(mls_target, self.mls_bin_edges)
        return self.mls_bin_weights[bin_idx]

    def forward(self, mls_logit, mls_target, fracture_logit, fracture_target) -> dict:
        mls_pred = IntermediatesModel.mls_from_logit(mls_logit)

        # Manual per-sample Huber (nn.SmoothL1Loss has no per-sample weight
        # argument), weighted then reduced by weighted mean.
        diff = mls_pred - mls_target
        abs_diff = diff.abs()
        quadratic = 0.5 * diff ** 2 / self.huber_beta
        linear = abs_diff - 0.5 * self.huber_beta
        per_sample_huber = torch.where(abs_diff < self.huber_beta, quadratic, linear)

        weights = self._sample_weight_for_mls(mls_target)
        mls_loss = (per_sample_huber * weights).sum() / weights.sum()

        fracture_loss = self.fracture_loss_fn(fracture_logit, fracture_target)
        total = self.mls_weight * mls_loss + self.fracture_weight * fracture_loss
        return {
            "total": total,
            "mls_loss": mls_loss.detach(),
            "fracture_loss": fracture_loss.detach(),
        }

### 21.5 v2 training (5-fold, weighted MLS loss)

**Checkpoint selection criterion (v2): smoothed blended
`quality = val_fracture_auc - val_mls_mae / MLS_NORMALIZER_MM`.**

This is a deliberate but consequential change from v1's AUC-only criterion.
It means 21.6's naive comparison below changes *two* variables at once (loss
weighting **and** checkpoint selection), so it cannot isolate the effect of
either one. 21.7 fixes this.


In [ ]:
# Cell: IntermediatesModel training v2 -- weighted MLS loss to fix the
# regression-to-the-mean pattern found in v1 (large true MLS under-predicted,
# near-zero MLS over-predicted). Writes to a NEW checkpoint root so v1
# results stay intact for comparison. Checkpoint/log deletion is NOT part
# of this cell (see the earlier destructive-cleanup bug) -- run the
# one-off reset cell separately and only if you deliberately want to.
import os
import time

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, mean_absolute_error

N_FOLDS = 5
N_EPOCHS = 20
BATCH_SIZE = 16
NUM_WORKERS = 2
PATIENCE = 8
EMA_WINDOW = 3
GRAD_CLIP_MAX_NORM = 1.0
FRACTURE_POS_WEIGHT = 27.88
MLS_NORMALIZER_MM = 5.0  # matches MLS_CRITICAL in the triage rule

CHECKPOINT_ROOT = "/content/drive/MyDrive/iaaa-brain-ct-triage/checkpoints_intermediates_v2"
LOG_PATH = "/content/drive/MyDrive/iaaa-brain-ct-triage/logs/training_log_intermediates_v2.csv"
os.makedirs(os.path.dirname(LOG_PATH), exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

full_slice_df = build_slice_df_with_z_rel(metadata_df, series_targets_df, dicom_dir)
print(f"Total slices: {len(full_slice_df)}  |  Series: {full_slice_df['dicom_series.id'].nunique()}")

log_rows = pd.read_csv(LOG_PATH).to_dict("records") if os.path.isfile(LOG_PATH) else []


def append_log(fold, epoch, train_loss, val_loss, val_mls_loss, val_fracture_loss,
                val_mls_mae, val_fracture_auc, quality, lr):
    log_rows.append({
        "fold": fold, "epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
        "val_mls_loss": val_mls_loss, "val_fracture_loss": val_fracture_loss,
        "val_mls_mae": val_mls_mae, "val_fracture_auc": val_fracture_auc,
        "quality": quality, "lr": lr, "timestamp": pd.Timestamp.now().isoformat(),
    })
    pd.DataFrame(log_rows).to_csv(LOG_PATH, index=False)


def smoothed_metric(history, key="quality", window=EMA_WINDOW):
    vals = [h[key] for h in history]
    if len(vals) < window:
        return float(np.mean(vals))
    return float(np.mean(vals[-window:]))


def epochs_since_best(history, key="quality", window=EMA_WINDOW):
    if len(history) < window:
        return 0
    smoothed_series = [
        float(np.mean([h[key] for h in history[max(0, i - window + 1):i + 1]]))
        for i in range(len(history))
    ]
    best_val = max(smoothed_series)
    last_best_idx = max(i for i, v in enumerate(smoothed_series) if v == best_val)
    return (len(history) - 1) - last_best_idx


for fold in range(N_FOLDS):
    print(f"\n{'=' * 60}\nINTERMEDIATES FOLD {fold} (v2)\n{'=' * 60}")

    fold_dir = os.path.join(CHECKPOINT_ROOT, f"fold_{fold}")
    os.makedirs(fold_dir, exist_ok=True)
    last_ckpt_path = os.path.join(fold_dir, "last_checkpoint.pt")
    best_ckpt_path = os.path.join(fold_dir, "best_model.pt")
    done_marker_path = os.path.join(fold_dir, "DONE")

    if os.path.isfile(done_marker_path):
        print(f"Fold {fold} already marked DONE. Skipping.")
        continue

    train_slice_df = full_slice_df[full_slice_df["fold"] != fold].reset_index(drop=True)
    val_slice_df = full_slice_df[full_slice_df["fold"] == fold].reset_index(drop=True)

    train_ds = IntermediatesSliceDataset(train_slice_df, dicom_dir)
    val_ds = IntermediatesSliceDataset(val_slice_df, dicom_dir)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)

    net = IntermediatesModel(pretrained=True, fracture_prior_prob=0.02).to(DEVICE)
    loss_fn = IntermediatesLoss(fracture_pos_weight=FRACTURE_POS_WEIGHT).to(DEVICE)
    optimizer = torch.optim.Adam(net.parameters(), lr=3e-4, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS, eta_min=1e-6)

    start_epoch = 0
    best_metric = -1e9
    history = []

    if os.path.isfile(last_ckpt_path):
        ckpt = torch.load(last_ckpt_path, map_location=DEVICE, weights_only=False)
        net.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])
        start_epoch = ckpt["epoch"] + 1
        best_metric = ckpt["best_metric"]
        history = ckpt["history"]
        print(f"Resuming fold {fold} from epoch {start_epoch} (best smoothed quality: {best_metric:.4f})")
        if start_epoch >= N_EPOCHS:
            print(f"Fold {fold} already completed. Skipping.")
            continue

    for epoch in range(start_epoch, N_EPOCHS):
        t0 = time.time()

        net.train()
        train_loss_total, n_train_batches = 0.0, 0
        for batch in train_loader:
            images = batch["image"].to(DEVICE)
            slice_thickness = batch["slice_thickness"].float().to(DEVICE)
            z_rel = batch["z_rel"].float().to(DEVICE)
            mls_target = batch["mls_target"].float().to(DEVICE)
            fracture_target = batch["fracture_target"].float().to(DEVICE)

            optimizer.zero_grad()
            out = net.forward_per_slice(images, slice_thickness, z_rel)
            loss_dict = loss_fn(out["mls_logit"], mls_target, out["fracture_logit"], fracture_target)
            loss_dict["total"].backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=GRAD_CLIP_MAX_NORM)
            optimizer.step()
            train_loss_total += loss_dict["total"].item()
            n_train_batches += 1
        train_loss = train_loss_total / max(n_train_batches, 1)
        scheduler.step()

        net.eval()
        val_loss_total, val_mls_loss_total, val_fracture_loss_total, n_val_batches = 0.0, 0.0, 0.0, 0
        all_mls_pred, all_mls_true = [], []
        all_fracture_prob, all_fracture_true = [], []
        with torch.no_grad():
            for batch in val_loader:
                images = batch["image"].to(DEVICE)
                slice_thickness = batch["slice_thickness"].float().to(DEVICE)
                z_rel = batch["z_rel"].float().to(DEVICE)
                mls_target = batch["mls_target"].float().to(DEVICE)
                fracture_target = batch["fracture_target"].float().to(DEVICE)

                out = net.forward_per_slice(images, slice_thickness, z_rel)
                loss_dict = loss_fn(out["mls_logit"], mls_target, out["fracture_logit"], fracture_target)
                val_loss_total += loss_dict["total"].item()
                val_mls_loss_total += loss_dict["mls_loss"].item()
                val_fracture_loss_total += loss_dict["fracture_loss"].item()
                n_val_batches += 1

                all_mls_pred.append(IntermediatesModel.mls_from_logit(out["mls_logit"]).cpu().numpy())
                all_mls_true.append(mls_target.cpu().numpy())
                all_fracture_prob.append(torch.sigmoid(out["fracture_logit"]).cpu().numpy())
                all_fracture_true.append(fracture_target.cpu().numpy())

        val_loss = val_loss_total / max(n_val_batches, 1)
        val_mls_loss = val_mls_loss_total / max(n_val_batches, 1)
        val_fracture_loss = val_fracture_loss_total / max(n_val_batches, 1)
        val_mls_mae = mean_absolute_error(np.concatenate(all_mls_true), np.concatenate(all_mls_pred))

        fracture_true_concat = np.concatenate(all_fracture_true)
        if len(np.unique(fracture_true_concat)) > 1:
            val_fracture_auc = roc_auc_score(fracture_true_concat, np.concatenate(all_fracture_prob))
        else:
            val_fracture_auc = float("nan")

        # Combined, scale-aware checkpoint-selection metric (see rationale above).
        quality = val_fracture_auc - val_mls_mae / MLS_NORMALIZER_MM

        current_lr = optimizer.param_groups[0]["lr"]
        elapsed = time.time() - t0

        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                         "val_mls_loss": val_mls_loss, "val_fracture_loss": val_fracture_loss,
                         "val_mls_mae": val_mls_mae, "val_fracture_auc": val_fracture_auc,
                         "quality": quality})
        smoothed_quality = smoothed_metric(history)

        print(f"  epoch {epoch:2d}/{N_EPOCHS - 1}: train_loss={train_loss:.4f}  "
              f"val_loss={val_loss:.4f} (mls={val_mls_loss:.4f}, frac={val_fracture_loss:.4f})  "
              f"val_mls_mae={val_mls_mae:.4f}  val_fracture_auc={val_fracture_auc:.4f}  "
              f"quality={quality:.4f}  smoothed={smoothed_quality:.4f}  lr={current_lr:.2e}  ({elapsed:.0f}s)")

        append_log(fold, epoch, train_loss, val_loss, val_mls_loss, val_fracture_loss,
                   val_mls_mae, val_fracture_auc, quality, current_lr)

        torch.save({"epoch": epoch, "model_state": net.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "scheduler_state": scheduler.state_dict(),
                    "best_metric": best_metric, "history": history}, last_ckpt_path)

        if smoothed_quality > best_metric:
            best_metric = smoothed_quality
            torch.save({"epoch": epoch, "model_state": net.state_dict(),
                        "optimizer_state": optimizer.state_dict(),
                        "scheduler_state": scheduler.state_dict(),
                        "best_metric": best_metric, "history": history}, best_ckpt_path)
            print(f"    New best smoothed quality for fold {fold}: {best_metric:.4f} -- saved best_model.pt")

        if epochs_since_best(history) >= PATIENCE:
            print(f"  Early stopping: smoothed quality has not improved for {PATIENCE} epochs.")
            break

    with open(done_marker_path, "w") as f:
        f.write(f"best_smoothed_quality={best_metric:.4f}\n")
    print(f"Fold {fold} finished. Best smoothed quality: {best_metric:.4f}")

print("\nAll intermediates v2 folds complete.\n")

### 21.6 Naive v1-vs-v2 comparison -- confounded, historical record only

**Do not use this cell's numbers to decide whether v2 is better.** v1's
checkpoints were picked by fracture-AUC alone; v2's were picked by a blended
quality score. Any difference below could come from either change. Kept here
only so the original run is documented; see 21.7 for the trustworthy version.

**Bug fixed below:** the original cell hardcoded `v1: accuracy=0.8195
f1_macro=0.7889` instead of computing it from `results_intermediates_df` --
silently stale if v1 is ever retrained. Now computed dynamically.


In [ ]:
# Correct, side-by-side comparison of v1 vs v2, reading metrics from the
# EXACT epoch each best_model.pt was actually saved at (same fix as before).
import torch
import pandas as pd

CHECKPOINT_ROOT_V1 = "/content/drive/MyDrive/iaaa-brain-ct-triage/checkpoints_intermediates"
CHECKPOINT_ROOT_V2 = "/content/drive/MyDrive/iaaa-brain-ct-triage/checkpoints_intermediates_v2"

def load_best_epoch_metrics(checkpoint_root, fold):
    ckpt = torch.load(f"{checkpoint_root}/fold_{fold}/best_model.pt", map_location="cpu", weights_only=False)
    best_epoch_idx = ckpt["epoch"]
    record = next(h for h in ckpt["history"] if h["epoch"] == best_epoch_idx)
    return record["val_mls_mae"], record["val_fracture_auc"]

rows = []
for fold in range(5):
    v1_mae, v1_auc = load_best_epoch_metrics(CHECKPOINT_ROOT_V1, fold)
    v2_mae, v2_auc = load_best_epoch_metrics(CHECKPOINT_ROOT_V2, fold)
    rows.append({
        "fold": fold,
        "v1_mls_mae": v1_mae, "v2_mls_mae": v2_mae, "mae_delta": v2_mae - v1_mae,
        "v1_fracture_auc": v1_auc, "v2_fracture_auc": v2_auc, "auc_delta": v2_auc - v1_auc,
    })

comparison_df = pd.DataFrame(rows).set_index("fold")
print(comparison_df.round(4))
print(f"\nMean v1_mls_mae: {comparison_df['v1_mls_mae'].mean():.4f}  |  Mean v2_mls_mae: {comparison_df['v2_mls_mae'].mean():.4f}")
print(f"Mean v1_fracture_auc: {comparison_df['v1_fracture_auc'].mean():.4f}  |  Mean v2_fracture_auc: {comparison_df['v2_fracture_auc'].mean():.4f}")

In [ ]:
# Re-run the SAME isolated triage validation, now against v2 checkpoints.
CHECKPOINT_ROOT_V2 = "/content/drive/MyDrive/iaaa-brain-ct-triage/checkpoints_intermediates_v2"

all_rows_v2 = []

for fold in range(5):
    print(f"Evaluating fold {fold} (v2)...")
    ckpt = torch.load(f"{CHECKPOINT_ROOT_V2}/fold_{fold}/best_model.pt", map_location=DEVICE, weights_only=False)
    net = IntermediatesModel(pretrained=False).to(DEVICE)
    net.load_state_dict(ckpt["model_state"])
    net.eval()

    val_slice_df = full_slice_df[full_slice_df["fold"] == fold].reset_index(drop=True)
    val_ds = IntermediatesSliceDataset(val_slice_df, dicom_dir)

    series_mls_preds = defaultdict(list)
    series_fracture_preds = defaultdict(list)

    with torch.no_grad():
        for i in range(len(val_ds)):
            item = val_ds[i]
            image = item["image"].unsqueeze(0).to(DEVICE)
            thickness = torch.tensor([item["slice_thickness"]]).float().to(DEVICE)
            z_rel = torch.tensor([item["z_rel"]]).float().to(DEVICE)

            out = net.forward_per_slice(image, thickness, z_rel)
            mls_nonneg = IntermediatesModel.mls_from_logit(out["mls_logit"]).item()
            fracture_prob = torch.sigmoid(out["fracture_logit"]).item()

            series_mls_preds[item["series_id"]].append(mls_nonneg)
            series_fracture_preds[item["series_id"]].append(fracture_prob)

    for series_id in series_mls_preds:
        mls_tensor = torch.tensor(series_mls_preds[series_id])
        fracture_tensor = torch.tensor(series_fracture_preds[series_id])
        pred_mls_series = IntermediatesModel.aggregate_mls(mls_tensor, mode="hard_max").item()
        pred_fracture_series = IntermediatesModel.aggregate_fracture(fracture_tensor).item()

        series_meta_rows = metadata_df[metadata_df["dicom_series.id"] == series_id]
        gt_intermediates = aggregate_series_intermediates(series_meta_rows)

        isolated_intermediates = {
            "V_EDH": gt_intermediates["V_EDH"], "V_SDH": gt_intermediates["V_SDH"],
            "V_IPH": gt_intermediates["V_IPH"], "V_SAH": gt_intermediates["V_SAH"],
            "V_IVH": gt_intermediates["V_IVH"],
            "MLS_mm": pred_mls_series, "fracture_prob": pred_fracture_series,
        }

        pred_class = triage_from_intermediates(isolated_intermediates)
        true_class = triage_from_intermediates(gt_intermediates)

        all_rows_v2.append({
            "series_id": series_id, "fold": fold,
            "true_triage": true_class, "pred_triage_from_intermediates": pred_class,
            "mls_pred": pred_mls_series, "mls_gt": gt_intermediates["MLS_mm"],
            "fracture_pred": pred_fracture_series, "fracture_gt": gt_intermediates["fracture_prob"],
        })

results_v2_df = pd.DataFrame(all_rows_v2)

accuracy_v2 = (results_v2_df["pred_triage_from_intermediates"] == results_v2_df["true_triage"]).mean()
f1_macro_v2 = f1_score(results_v2_df["true_triage"], results_v2_df["pred_triage_from_intermediates"], average="macro")
v1_accuracy = (results_intermediates_df["pred_triage_from_intermediates"] == results_intermediates_df["true_triage"]).mean()
v1_f1_macro = f1_score(results_intermediates_df["true_triage"], results_intermediates_df["pred_triage_from_intermediates"], average="macro")
print(f"\nv1: accuracy={v1_accuracy:.4f}  f1_macro={v1_f1_macro:.4f}")  # FIX: computed, not hardcoded -- requires 21.3 to have run in this kernel session
print(f"v2: accuracy={accuracy_v2:.4f}  f1_macro={f1_macro_v2:.4f}")

print("\nConfusion matrix v2 (rows=true, cols=predicted):")
print(pd.crosstab(results_v2_df["true_triage"], results_v2_df["pred_triage_from_intermediates"]))

under_triage_v2 = results_v2_df[(results_v2_df["true_triage"] == 1) &
                                  (results_v2_df["pred_triage_from_intermediates"] == 0)]
print(f"\nUnder-triage cases (true Urgent -> pred Non-Urgent): {len(under_triage_v2)} (was 7 in v1)")

### 21.7 Fair v1-vs-v2 comparison (retrospective checkpoint re-selection)

**Objective:** isolate the effect of the weighted MLS loss alone, by removing
the confound identified in 21.5/21.6 (v1 and v2 used different checkpoint
selection criteria).

**Theory:** a controlled comparison must change exactly one variable at a
time. Since v1's full per-epoch history (`val_mls_mae`, `val_fracture_auc`
for every epoch of every fold) is already saved in
`training_log_intermediates.csv`, we don't need to retrain v1 -- we can
retrospectively ask *"which epoch would v1 have selected if it had used v2's
quality criterion?"* purely from the existing log. Comparing v2 against this
retrospective v1 selection isolates the loss-weighting change specifically.

**No GPU required** -- this is a pandas-only cell over an existing CSV.


In [ ]:
# 21.7 -- Fair v1-vs-v2 comparison via retrospective checkpoint re-selection.
# See markdown above for why this is necessary (checkpoint-selection confound
# between the v1 and v2 training cells).
import numpy as np
import pandas as pd
from scipy import stats

MLS_NORMALIZER_MM = 5.0   # must match the constant used in cell 21.5 (v2 training)
EMA_WINDOW = 3            # must match EMA_WINDOW used in both training cells

v1_log = pd.read_csv(
    "/content/drive/MyDrive/iaaa-brain-ct-triage/logs/training_log_intermediates.csv"
)


def select_best_epoch_by_quality(fold_log: pd.DataFrame) -> pd.Series:
    """Reproduce v2's EMA-smoothed blended-quality checkpoint-selection rule
    on an arbitrary per-epoch log, so it can be applied retrospectively to a
    log (v1's) that originally used a different selection rule.
    """
    fold_log = fold_log.sort_values("epoch").reset_index(drop=True)
    quality = fold_log["val_fracture_auc"] - fold_log["val_mls_mae"] / MLS_NORMALIZER_MM
    smoothed = quality.rolling(window=EMA_WINDOW, min_periods=1).mean()
    best_idx = smoothed.idxmax()
    return fold_log.loc[best_idx]


rows = []
for fold in sorted(v1_log["fold"].unique()):
    fold_log = v1_log[v1_log["fold"] == fold]
    best_row = select_best_epoch_by_quality(fold_log)
    rows.append({
        "fold": int(fold),
        "v1_fair_epoch": int(best_row["epoch"]),
        "v1_fair_mls_mae": best_row["val_mls_mae"],
        "v1_fair_fracture_auc": best_row["val_fracture_auc"],
    })

v1_fair_df = pd.DataFrame(rows).set_index("fold")
print("v1 checkpoints RE-SELECTED under the same quality rule used for v2:")
print(v1_fair_df.round(4))

# Join against v2's numbers (comparison_df comes from cell 21.6 / original cell 96).
fair_comparison_df = comparison_df[["v2_mls_mae", "v2_fracture_auc"]].join(v1_fair_df)
fair_comparison_df["fair_mae_delta"] = (
    fair_comparison_df["v2_mls_mae"] - fair_comparison_df["v1_fair_mls_mae"]
)
fair_comparison_df["fair_auc_delta"] = (
    fair_comparison_df["v2_fracture_auc"] - fair_comparison_df["v1_fair_fracture_auc"]
)

print("\nFair comparison (both models' checkpoints selected by the SAME criterion):")
print(fair_comparison_df[[
    "v1_fair_mls_mae", "v2_mls_mae", "fair_mae_delta",
    "v1_fair_fracture_auc", "v2_fracture_auc", "fair_auc_delta",
]].round(4))

t_mae, p_mae = stats.ttest_rel(fair_comparison_df["v2_mls_mae"], fair_comparison_df["v1_fair_mls_mae"])
t_auc, p_auc = stats.ttest_rel(fair_comparison_df["v2_fracture_auc"], fair_comparison_df["v1_fair_fracture_auc"])
print(f"\nPaired t-test (5 folds), fair MAE delta:  t={t_mae:.3f}, p={p_mae:.3f}")
print(f"Paired t-test (5 folds), fair AUC delta:  t={t_auc:.3f}, p={p_auc:.3f}")
print("(df=4; two-tailed critical t at p<0.05 is 2.776 -- treat anything below that as inconclusive.)")


#### 21.7 Result (recorded)

```
Paired t-test (5 folds), fair MAE delta:  t=1.451, p=0.220
Paired t-test (5 folds), fair AUC delta:  t=0.343, p=0.749
```

**Conclusion: no statistically-supported evidence that the weighted MLS
loss (v2) improves on v1.** The mean fair MAE delta is actually *positive*
(+0.0337 -- v2 slightly worse), a complete reversal from the naive
comparison's apparent -0.0445 improvement. Fold 3 makes the mechanism
concrete: under the naive comparison it showed the largest apparent gain
(-0.2477); under the fair comparison, the same fold shows one of the
largest *losses* (+0.1135) -- because v1's AUC-only checkpoint selection
had picked a genuinely poor-for-MLS epoch in that fold, not because v2's
loss weighting helped.

**Decision (see project log for full discussion): stop iterating on this
specific fix.** With only 5 folds, statistical power is low, and the
GPU cost of adding enough seeds to meaningfully increase power is not
justified given other higher-priority open items (the `predict_series_v2`
caching mystery in Section 25.5, and the HU-based calcification filter).
`IntermediatesModel` v1 remains the deployed model (Section 23.3), now for
the *correct* documented reason: no reliable evidence favors v2, not the
original (confounded) claim that v1 measurably outperforms it.

If revisited later, Label Distribution Smoothing / Feature Distribution
Smoothing (Yang et al., ICML 2021) are the natural next things to try,
rather than a simple sample-reweighted loss.


## 22. End-to-End Hybrid Pipeline Validation (No Ground-Truth Substitution)

**Objective:** every validation so far (Section 17.2's segmentation-only
check, Section 21.3/21.6's IntermediatesModel-only check) fed the *other*
model's ground-truth values in, to isolate one model's error contribution
at a time. This is the first point in the notebook where **both** models
run for real, together, on all 338 series, with each series scored using
its own fold's out-of-fold checkpoint (no leakage) -- the actual
end-to-end pipeline that will ship, with nothing substituted from ground
truth anywhere.


⚠️ **Unresolved technical debt, flagged (not fixed) here:** the comment
inside the cell below says *"this is the 2nd time this exact gap has
appeared; see the note below for a permanent fix"* -- but no such note
exists anywhere in this notebook, in this cell or any other. This is a
genuine dangling TODO that was written and then never followed up on.

**What it's actually pointing at:** the DICOM-load → HU-convert → window →
resize logic inside `load_inference_slice` below has now been hand-written
at least three times across this project (once for `SegmentationSliceDataset`
in Section 15, once for `IntermediatesSliceDataset` in Section 21, and now
again here) -- a real DRY violation. **Recommended permanent fix (not yet
implemented):** extract one shared `load_inference_slice`-equivalent
function into `src/preprocessing.py`, parameterized by which window presets
and output size each caller needs, and have all three call sites import it
instead of redefining it. Left as an explicit next step rather than
silently duplicating it a fourth time without comment.


In [ ]:
# Cell: End-to-end (no-cheating) validation. Both models run for real, on
# every one of the 338 series, using each series' OWN out-of-fold checkpoint
# (fold f's series -> fold f's checkpoint) to avoid any leakage. No ground
# truth is substituted anywhere in this pipeline.
import os
import pydicom
import cv2
import numpy as np
import torch
import pandas as pd
from collections import defaultdict
from sklearn.metrics import f1_score, classification_report

SEG_CHECKPOINT_ROOT = "/content/drive/MyDrive/iaaa-brain-ct-triage/checkpoints_seg_v2"
INTERMEDIATES_CHECKPOINT_ROOT = "/content/drive/MyDrive/iaaa-brain-ct-triage/checkpoints_intermediates"
IMAGE_SIZE = 224

# Alias once, before load_inference_slice -- same pattern needed each time
# a new function reads DICOM/windowing directly (this is the 2nd time this
# exact gap has appeared; see the note below for a permanent fix).
to_hu = preprocessing.to_hu
apply_window = preprocessing.apply_window
WINDOW_PRESETS = preprocessing.WINDOW_PRESETS
def load_inference_slice(dicom_root: str, series_id, sop_uid: str, thickness: float,
                          pixel_spacing_0: float, pixel_spacing_1: float, z_rel: float) -> dict:
    """Loads ONE slice directly from its DICOM file -- no annotation file
    dependency, so this works for all 338 series, not just the 198 with
    pixel masks. Provides everything both models need: a resized 2.5D
    image tensor (shared, since both models use the same windows/size) plus
    physical metadata (including native Rows/Columns, read directly from
    the DICOM header, replacing the annotation-JSON "shape" field used
    during segmentation training)."""
    dcm_path = os.path.join(dicom_root, str(series_id), f"{sop_uid}.dcm")
    dcm = pydicom.dcmread(dcm_path)
    hu_image = to_hu(dcm)
    channels = [apply_window(hu_image, center, width) for center, width in WINDOW_PRESETS.values()]
    image = np.stack(channels, axis=0).astype(np.float32)

    native_height, native_width = int(dcm.Rows), int(dcm.Columns)
    image_resized = cv2.resize(
        image.transpose(1, 2, 0), (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA,
    ).transpose(2, 0, 1)

    return {
        "image": torch.from_numpy(image_resized),
        "slice_thickness": thickness,
        "pixel_spacing_0": pixel_spacing_0,
        "pixel_spacing_1": pixel_spacing_1,
        "original_height": native_height,
        "original_width": native_width,
        "z_rel": z_rel,
    }


# full_slice_df already has z_rel (from build_slice_df_with_z_rel) and fold.
seg_models, intermediates_models = {}, {}
for fold in range(5):
    seg_net = seg_model.HemorrhageSegModel(pretrained=False).to(DEVICE)
    seg_ckpt = torch.load(f"{SEG_CHECKPOINT_ROOT}/fold_{fold}/best_model.pt", map_location=DEVICE, weights_only=False)
    seg_net.load_state_dict(seg_ckpt["model_state"])
    seg_net.eval()
    seg_models[fold] = seg_net

    int_net = IntermediatesModel(pretrained=False).to(DEVICE)
    int_ckpt = torch.load(f"{INTERMEDIATES_CHECKPOINT_ROOT}/fold_{fold}/best_model.pt", map_location=DEVICE, weights_only=False)
    int_net.load_state_dict(int_ckpt["model_state"])
    int_net.eval()
    intermediates_models[fold] = int_net

print("Loaded all 5 folds for both models.")

e2e_rows = []
series_ids = full_slice_df["dicom_series.id"].unique()

with torch.no_grad():
    for series_id in series_ids:
        series_rows = full_slice_df[full_slice_df["dicom_series.id"] == series_id]
        fold = int(series_rows["fold"].iloc[0])
        seg_net = seg_models[fold]
        int_net = intermediates_models[fold]

        series_slices = [
            load_inference_slice(
                dicom_dir, series_id, row["dicom_series.SOPInstanceUID"],
                float(row["dicom_series.SliceThickness"]),
                float(row["dicom_series.PixelSpacing0"]), float(row["dicom_series.PixelSpacing1"]),
                float(row["z_rel"]),
            )
            for _, row in series_rows.iterrows()
        ]

        # Segmentation -> 5 hemorrhage volumes (real predicted-mask volumes).
        pred_volumes = volume_aggregation.predict_series_volumes(
            seg_net, series_slices, DEVICE, image_size=IMAGE_SIZE
        )

        # IntermediatesModel -> MLS + fracture (real predicted, hard-max/complement-product).
        mls_per_slice, fracture_per_slice = [], []
        for s in series_slices:
            image = s["image"].unsqueeze(0).to(DEVICE)
            thickness_t = torch.tensor([s["slice_thickness"]]).float().to(DEVICE)
            z_rel_t = torch.tensor([s["z_rel"]]).float().to(DEVICE)
            out = int_net.forward_per_slice(image, thickness_t, z_rel_t)
            mls_per_slice.append(IntermediatesModel.mls_from_logit(out["mls_logit"]).item())
            fracture_per_slice.append(torch.sigmoid(out["fracture_logit"]).item())

        pred_mls = IntermediatesModel.aggregate_mls(torch.tensor(mls_per_slice), mode="hard_max").item()
        pred_fracture = IntermediatesModel.aggregate_fracture(torch.tensor(fracture_per_slice)).item()

        # Fully predicted intermediates -- NO ground truth anywhere in this dict.
        predicted_intermediates = {**pred_volumes, "MLS_mm": pred_mls, "fracture_prob": pred_fracture}
        pred_triage = triage_from_intermediates(predicted_intermediates)

        gt_intermediates = aggregate_series_intermediates(
            metadata_df[metadata_df["dicom_series.id"] == series_id]
        )
        true_triage = triage_from_intermediates(gt_intermediates)

        e2e_rows.append({
            "series_id": series_id, "fold": fold,
            "true_triage": true_triage, "pred_triage": pred_triage,
        })

e2e_df = pd.DataFrame(e2e_rows)
print(f"\nTotal series evaluated: {len(e2e_df)}")

accuracy = (e2e_df["pred_triage"] == e2e_df["true_triage"]).mean()
f1_macro = f1_score(e2e_df["true_triage"], e2e_df["pred_triage"], average="macro")
print(f"\nEND-TO-END (fully predicted, no GT substitution): accuracy={accuracy:.4f}  f1_macro={f1_macro:.4f}")
print(f"Official MIL v1 baseline: accuracy=0.3676  f1_macro=0.3427")
print(classification_report(e2e_df["true_triage"], e2e_df["pred_triage"],
                              target_names=["Non-Urgent", "Urgent", "Critical"]))
print("\nConfusion matrix (rows=true, cols=predicted):")
print(pd.crosstab(e2e_df["true_triage"], e2e_df["pred_triage"]))

**Drilling into the false-Critical errors:** the 7 series pushed all the
way from true Non-Urgent to predicted Critical are the specific cases this
project later spent the most diagnostic effort on (this is the same 7-case
set that motivated the noise-scale-vs-large-blob split, and eventually the
3D connected-component filter). The cell below re-runs just these series to
see each model's predicted intermediates individually -- distinguishing
"one model alone crossed a hard Critical threshold" from "moderate errors
in both models combined via a COMBO rule."


In [ ]:
# Drill-down: are the 7 Non-Urgent -> Critical jumps driven by the COMBO
# rule (moderate segmentation noise + moderate intermediates noise crossing
# COMBO_VOL/COMBO_MLS together), or by one model alone crossing a hard
# threshold (EDH_CRIT, SDH_CRIT, IPH_CRIT, TOTAL_VOL_CRIT, MLS_CRITICAL)?
critical_jump_ids = e2e_df[(e2e_df["true_triage"] == 0) & (e2e_df["pred_triage"] == 2)]["series_id"]

for sid in critical_jump_ids:
    row = full_slice_df[full_slice_df["dicom_series.id"] == sid].iloc[0]
    fold = int(row["fold"])
    # Re-run just this one series through both models to inspect its predicted intermediates.
    series_rows = full_slice_df[full_slice_df["dicom_series.id"] == sid]
    series_slices = [
        load_inference_slice(
            dicom_dir, sid, r["dicom_series.SOPInstanceUID"], float(r["dicom_series.SliceThickness"]),
            float(r["dicom_series.PixelSpacing0"]), float(r["dicom_series.PixelSpacing1"]), float(r["z_rel"]),
        ) for _, r in series_rows.iterrows()
    ]
    with torch.no_grad():
        pred_volumes = volume_aggregation.predict_series_volumes(seg_models[fold], series_slices, DEVICE, image_size=IMAGE_SIZE)
        mls_vals, frac_vals = [], []
        for s in series_slices:
            image = s["image"].unsqueeze(0).to(DEVICE)
            th = torch.tensor([s["slice_thickness"]]).float().to(DEVICE)
            zr = torch.tensor([s["z_rel"]]).float().to(DEVICE)
            out = intermediates_models[fold].forward_per_slice(image, th, zr)
            mls_vals.append(IntermediatesModel.mls_from_logit(out["mls_logit"]).item())
            frac_vals.append(torch.sigmoid(out["fracture_logit"]).item())
        pred_mls = IntermediatesModel.aggregate_mls(torch.tensor(mls_vals), mode="hard_max").item()
        pred_frac = IntermediatesModel.aggregate_fracture(torch.tensor(frac_vals)).item()

    total_vol = sum(pred_volumes.values())
    print(f"series {sid} (fold {fold}): total_vol={total_vol:.2f}mL  MLS={pred_mls:.2f}mm  "
          f"fracture_prob={pred_frac:.3f}  volumes={ {k: round(v,2) for k,v in pred_volumes.items()} }")


## 23. Packaging & Smoke-Testing the Hybrid `submission.py`

**Objective:** assemble the final two-model package (segmentation + 
IntermediatesModel) into the same kind of self-contained, smoke-tested 
script validated for the MIL baseline in Section 11 -- same rigor, now for 
two models sharing one script.

### 23.1 Defensive check: training-time vs. submission-time window presets


In [ ]:
# Diagnostic: confirm the WINDOW_PRESETS used to train segmentation/intermediates
# match EXACTLY what submission.py hardcodes -- a silent mismatch here would
# corrupt predictions with no error message.
print("src.preprocessing.WINDOW_PRESETS:", preprocessing.WINDOW_PRESETS)
print("submission.py's WINDOW_PRESETS:  ", {"brain": (40, 80), "blood": (75, 215), "bone": (500, 2500)})

### 23.2 Write the self-contained `submission.py`

Same standalone-script rationale as Section 11 (no `src` dependency,
architectures re-declared exactly to match checkpoint `state_dict` keys).


In [ ]:
%%writefile /content/drive/MyDrive/iaaa-brain-ct-triage/submission_hybrid/submission.py
#!/usr/bin/env python
"""Self-contained hybrid submission entry point for the IAAA Brain CT Triage
Challenge. Combines two independently-trained models:

  1. HemorrhageSegModel (U-Net, segmentation-models-pytorch) -- predicts
     per-slice pixel masks for 5 hemorrhage subtypes, aggregated into
     series-level volumes (mL) via the physical formula
     pixel_count x PixelSpacing0 x PixelSpacing1 x SliceThickness / 1000.
  2. IntermediatesModel (2.5D CNN + physical embedding) -- predicts
     per-slice MLS_mm and fracture probability, aggregated into series-level
     values via hard-max pooling (MLS) and complement-product pooling
     (fracture), matching each variable's physical/clinical meaning.

Both are 5-fold ensembles (probability/prediction averaging across folds,
NOT a single-fold prediction), trained independently on the same
patient-grouped, class-stratified cross-validation splits.

Deliberately self-contained (no dependency on the custom "src" package),
matching the same rationale as the original MIL submission.py: the
evaluation harness's file-packaging behavior for arbitrary directories is
unknown/unreliable, so this file only depends on the `models/` directory
sitting next to it.

IMPORTANT: architectures below must exactly match the checkpoints' state_dict
keys. Verified against:
  - HemorrhageSegModel: smp.Unet(encoder_name="resnet18", in_channels=3, classes=6)
  - IntermediatesModel: ResNet18 backbone + 32-dim physical embedding,
    Softplus (NOT ReLU) for MLS non-negativity, bias-initialized fracture head.

Official usage (per BCT.md):
    python submission.py --data-dir /path/to/data-dir --predictions-file-path /path/to/submission.csv

Expected directory layout alongside this script:
    submission.py
    models/
        seg_fold_0_best.pt ... seg_fold_4_best.pt
        intermediates_fold_0_best.pt ... intermediates_fold_4_best.pt
"""

import argparse
import glob
import os

import cv2
import numpy as np
import pandas as pd
import pydicom
import segmentation_models_pytorch as smp
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tv_models
from torch.utils.data import DataLoader, Dataset

WINDOW_PRESETS = {
    "brain": (40, 80),
    "blood": (75, 215),
    "bone": (500, 2500),
}
IMAGE_SIZE = 224

TARGET_COLUMNS = ["V_EDH", "V_SDH", "V_IPH", "V_SAH", "V_IVH", "fracture_prob", "MLS_mm"]

CLASS_VALUE_TO_TARGET_KEY = {
    1: "V_IVH",
    2: "V_IPH",
    3: "V_SDH",
    4: "V_EDH",
    5: "V_SAH",
}
N_CLASSES = 6


def load_dicom_series(series_dir: str) -> list:
    dcm_paths = sorted(glob.glob(os.path.join(series_dir, "*.dcm")))
    slices = [pydicom.dcmread(p) for p in dcm_paths]
    slices.sort(key=lambda s: float(s.ImagePositionPatient[2]))
    return slices


def to_hu(dcm) -> np.ndarray:
    pixel_array = dcm.pixel_array.astype(np.float32)
    slope = float(getattr(dcm, "RescaleSlope", 1.0))
    intercept = float(getattr(dcm, "RescaleIntercept", 0.0))
    return pixel_array * slope + intercept


def apply_window(hu_image: np.ndarray, center: float, width: float) -> np.ndarray:
    low = center - width / 2
    high = center + width / 2
    clipped = np.clip(hu_image, low, high)
    return (clipped - low) / (high - low)


def make_multiwindow_image(dcm) -> np.ndarray:
    hu_image = to_hu(dcm)
    channels = [apply_window(hu_image, center, width) for center, width in WINDOW_PRESETS.values()]
    return np.stack(channels, axis=0).astype(np.float32)


class InferenceSeriesDataset(Dataset):
    def __init__(self, series_ids: list, dicom_root: str, image_size: int = IMAGE_SIZE):
        self.series_ids = list(series_ids)
        self.dicom_root = dicom_root
        self.image_size = image_size

    def __len__(self) -> int:
        return len(self.series_ids)

    def __getitem__(self, idx: int) -> dict:
        series_id = self.series_ids[idx]
        series_dir = os.path.join(self.dicom_root, str(series_id))
        dcm_slices = load_dicom_series(series_dir)

        z_positions = [float(dcm.ImagePositionPatient[2]) for dcm in dcm_slices]
        z_min, z_max = min(z_positions), max(z_positions)
        z_range = (z_max - z_min) if (z_max - z_min) != 0 else 1.0

        slices = []
        for dcm, z_pos in zip(dcm_slices, z_positions):
            multiwindow = make_multiwindow_image(dcm)
            native_height, native_width = multiwindow.shape[1], multiwindow.shape[2]

            resized = cv2.resize(
                multiwindow.transpose(1, 2, 0), (self.image_size, self.image_size),
                interpolation=cv2.INTER_AREA,
            ).transpose(2, 0, 1)

            pixel_spacing = getattr(dcm, "PixelSpacing", [1.0, 1.0])
            slice_thickness = float(getattr(dcm, "SliceThickness", 1.0))

            slices.append({
                "image": torch.from_numpy(resized.astype(np.float32)),
                "slice_thickness": slice_thickness,
                "pixel_spacing_0": float(pixel_spacing[0]),
                "pixel_spacing_1": float(pixel_spacing[1]),
                "original_height": native_height,
                "original_width": native_width,
                "z_rel": (z_pos - z_min) / z_range,
            })

        try:
            series_id_value = int(series_id)
        except (ValueError, TypeError):
            series_id_value = str(series_id)

        return {"series_id": series_id_value, "slices": slices}


def collate_single_series(batch: list) -> dict:
    return batch[0]


SEG_IMAGENET_MEAN = [0.485, 0.456, 0.406]
SEG_IMAGENET_STD = [0.229, 0.224, 0.225]


class HemorrhageSegModel(nn.Module):
    def __init__(self, encoder_name: str = "resnet18", pretrained: bool = True):
        super().__init__()
        self.unet = smp.Unet(
            encoder_name=encoder_name,
            encoder_weights="imagenet" if pretrained else None,
            in_channels=3,
            classes=N_CLASSES,
        )
        self.register_buffer("imagenet_mean", torch.tensor(SEG_IMAGENET_MEAN).view(1, 3, 1, 1))
        self.register_buffer("imagenet_std", torch.tensor(SEG_IMAGENET_STD).view(1, 3, 1, 1))

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        normalized = (images - self.imagenet_mean) / self.imagenet_std
        return self.unet(normalized)


class PhysicalEmbedding(nn.Module):
    def __init__(self, out_dim: int = 32):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(2, 64), nn.ReLU(inplace=True), nn.Linear(64, out_dim))

    def forward(self, slice_thickness: torch.Tensor, z_rel: torch.Tensor) -> torch.Tensor:
        x = torch.stack([slice_thickness, z_rel], dim=1)
        return self.mlp(x)


class IntermediatesModel(nn.Module):
    def __init__(self, backbone_name: str = "resnet18", pretrained: bool = True,
                 phys_embed_dim: int = 32, fracture_prior_prob: float = 0.02):
        super().__init__()
        backbone = getattr(tv_models, backbone_name)(
            weights="IMAGENET1K_V1" if pretrained else None
        )
        cnn_feature_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.physical_embedding = PhysicalEmbedding(out_dim=phys_embed_dim)

        combined_dim = cnn_feature_dim + phys_embed_dim
        self.mls_head = nn.Sequential(nn.Linear(combined_dim, 64), nn.ReLU(inplace=True), nn.Linear(64, 1))
        self.fracture_head = nn.Sequential(nn.Linear(combined_dim, 64), nn.ReLU(inplace=True), nn.Linear(64, 1))
        self._init_fracture_head_bias(fracture_prior_prob)

    def _init_fracture_head_bias(self, prior_prob: float) -> None:
        final_layer = self.fracture_head[-1]
        bias_value = -torch.log(torch.tensor((1 - prior_prob) / prior_prob))
        nn.init.constant_(final_layer.bias, bias_value.item())

    def forward_per_slice(self, images: torch.Tensor, slice_thickness: torch.Tensor,
                           z_rel: torch.Tensor) -> dict:
        cnn_features = self.backbone(images)
        phys_features = self.physical_embedding(slice_thickness, z_rel)
        combined = torch.cat([cnn_features, phys_features], dim=1)
        return {
            "mls_logit": self.mls_head(combined).squeeze(-1),
            "fracture_logit": self.fracture_head(combined).squeeze(-1),
        }

    @staticmethod
    def mls_from_logit(mls_logit: torch.Tensor) -> torch.Tensor:
        return F.softplus(mls_logit)

    @staticmethod
    def aggregate_mls(mls_per_slice_nonneg: torch.Tensor) -> torch.Tensor:
        return mls_per_slice_nonneg.max()

    @staticmethod
    def aggregate_fracture(fracture_prob_per_slice: torch.Tensor) -> torch.Tensor:
        return 1.0 - torch.prod(1.0 - fracture_prob_per_slice)


THIS_DIR = os.path.dirname(os.path.abspath(__file__))
MODELS_DIR = os.path.join(THIS_DIR, "models")
N_FOLDS = 5


def load_ensembles(device: str):
    seg_models, intermediates_models = [], []
    for fold in range(N_FOLDS):
        seg_ckpt_path = os.path.join(MODELS_DIR, f"seg_fold_{fold}_best.pt")
        if not os.path.isfile(seg_ckpt_path):
            raise FileNotFoundError(f"Missing segmentation checkpoint: {seg_ckpt_path}")
        seg_net = HemorrhageSegModel(pretrained=False).to(device)
        seg_ckpt = torch.load(seg_ckpt_path, map_location=device, weights_only=False)
        seg_net.load_state_dict(seg_ckpt["model_state"])
        seg_net.eval()
        seg_models.append(seg_net)

        int_ckpt_path = os.path.join(MODELS_DIR, f"intermediates_fold_{fold}_best.pt")
        if not os.path.isfile(int_ckpt_path):
            raise FileNotFoundError(f"Missing intermediates checkpoint: {int_ckpt_path}")
        int_net = IntermediatesModel(pretrained=False).to(device)
        int_ckpt = torch.load(int_ckpt_path, map_location=device, weights_only=False)
        int_net.load_state_dict(int_ckpt["model_state"])
        int_net.eval()
        intermediates_models.append(int_net)

    return seg_models, intermediates_models


@torch.no_grad()
def predict_series(slices: list, seg_models: list, intermediates_models: list, device: str) -> dict:
    volumes = {key: 0.0 for key in CLASS_VALUE_TO_TARGET_KEY.values()}
    mls_per_slice, fracture_per_slice = [], []

    for s in slices:
        image = s["image"].unsqueeze(0).to(device)

        seg_probs_sum = None
        for seg_net in seg_models:
            logits = seg_net(image)
            probs = torch.softmax(logits, dim=1)
            seg_probs_sum = probs if seg_probs_sum is None else seg_probs_sum + probs
        seg_probs_mean = seg_probs_sum / len(seg_models)
        pred_mask = seg_probs_mean.argmax(dim=1).squeeze(0).cpu().numpy()

        h_scale = s["original_height"] / IMAGE_SIZE
        w_scale = s["original_width"] / IMAGE_SIZE
        effective_spacing_0 = s["pixel_spacing_0"] * h_scale
        effective_spacing_1 = s["pixel_spacing_1"] * w_scale
        for class_value, target_key in CLASS_VALUE_TO_TARGET_KEY.items():
            pixel_count = int((pred_mask == class_value).sum())
            volumes[target_key] += (
                pixel_count * effective_spacing_0 * effective_spacing_1 * s["slice_thickness"] / 1000.0
            )

        thickness_t = torch.tensor([s["slice_thickness"]]).float().to(device)
        z_rel_t = torch.tensor([s["z_rel"]]).float().to(device)

        mls_sum, fracture_sum = 0.0, 0.0
        for int_net in intermediates_models:
            out = int_net.forward_per_slice(image, thickness_t, z_rel_t)
            mls_sum += IntermediatesModel.mls_from_logit(out["mls_logit"]).item()
            fracture_sum += torch.sigmoid(out["fracture_logit"]).item()
        mls_per_slice.append(mls_sum / len(intermediates_models))
        fracture_per_slice.append(fracture_sum / len(intermediates_models))

    pred_mls = IntermediatesModel.aggregate_mls(torch.tensor(mls_per_slice)).item()
    pred_fracture = IntermediatesModel.aggregate_fracture(torch.tensor(fracture_per_slice)).item()

    return {**volumes, "MLS_mm": pred_mls, "fracture_prob": pred_fracture}


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data-dir", required=True, help="Directory with one subfolder per series.")
    parser.add_argument("--predictions-file-path", required=True, help="Output CSV path.")
    args = parser.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    series_ids = sorted(
        d for d in os.listdir(args.data_dir)
        if os.path.isdir(os.path.join(args.data_dir, d))
    )
    print(f"Found {len(series_ids)} series in {args.data_dir}")
    if len(series_ids) == 0:
        raise RuntimeError(f"No series subdirectories found under {args.data_dir}")

    dataset = InferenceSeriesDataset(series_ids, args.data_dir)
    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=2, collate_fn=collate_single_series)

    seg_models, intermediates_models = load_ensembles(device)
    print(f"Loaded {len(seg_models)} segmentation folds and {len(intermediates_models)} intermediates folds.")

    rows = []
    for item in loader:
        series_id = item["series_id"]
        intermediates = predict_series(item["slices"], seg_models, intermediates_models, device)
        rows.append({"series_id": series_id, **intermediates})
        print(f"  series {series_id}: done ({len(item['slices'])} slices)")

    pred_df = pd.DataFrame(rows)
    pred_df = pred_df[["series_id"] + TARGET_COLUMNS]
    pred_df.to_csv(args.predictions_file_path, index=False)
    print(f"Saved predictions for {len(pred_df)} series to {args.predictions_file_path}")


if __name__ == "__main__":
    main()

### 23.3 Collect checkpoints — ⚠️ deploys IntermediatesModel v1, not v2

**This is a directly consequential link to Section 21.7.** The comment
below states the choice explicitly: *"INTERMEDIATES: use V1 (better triage
performance than v2)."* This decision was made using the **naive**
v1-vs-v2 comparison from Section 21.6 -- the same comparison Section 21.7
showed is confounded (different checkpoint-selection criteria between v1
and v2, not an apples-to-apples test of the loss-weighting change alone).

**RESOLVED (Section 21.7 result):** the fair, deconfounded comparison
found no statistically-supported difference between v1 and v2
(MAE: t=1.451, p=0.220; AUC: t=0.343, p=0.749) -- if anything, v2's mean
MAE was slightly *worse*. **v1 remains deployed, correctly**, but the
comment below should be read as "no reliable evidence favors v2", not
the original, confounded claim that v1 measurably outperforms it.


In [ ]:
from pathlib import Path
import shutil

ROOT = Path("/content/drive/MyDrive/iaaa-brain-ct-triage")
DST = ROOT / "submission_hybrid" / "models"

DST.mkdir(parents=True, exist_ok=True)

# =========================================================
# SEGMENTATION: use V2 (best validated version)
# =========================================================
for fold in range(5):
    src = ROOT / "checkpoints_seg_v2" / f"fold_{fold}" / "best_model.pt"
    dst = DST / f"seg_fold_{fold}_best.pt"

    if not src.exists():
        raise FileNotFoundError(f"Missing SEG v2 checkpoint: {src}")

    shutil.copy2(src, dst)
    print(f"✅ SEG V2 fold {fold}: {dst.name}")


# =========================================================
# INTERMEDIATES: use V1 (better triage performance than v2)
# =========================================================
for fold in range(5):
    src = ROOT / "checkpoints_intermediates" / f"fold_{fold}" / "best_model.pt"
    dst = DST / f"intermediates_fold_{fold}_best.pt"

    if not src.exists():
        raise FileNotFoundError(f"Missing Intermediate v1 checkpoint: {src}")

    shutil.copy2(src, dst)
    print(f"✅ INTERMEDIATE V1 fold {fold}: {dst.name}")

print("\n✅ Final hybrid model package prepared.")
print("SEG = v2")
print("INTERMEDIATES = v1")

### 23.4 Smoke-test on 3 real series, end-to-end through the packaged script

In [ ]:
import shutil
import os

TEST_DIR = "/content/smoke_test_data"

# Remove previous smoke-test data completely
if os.path.exists(TEST_DIR):
    shutil.rmtree(TEST_DIR)

os.makedirs(TEST_DIR, exist_ok=True)

sample_series = full_slice_df["dicom_series.id"].unique()[:3]

for sid in sample_series:
    src = f"{dicom_dir}/{sid}"
    dst = f"{TEST_DIR}/{sid}"
    shutil.copytree(src, dst)

print(f"✅ Copied exactly {len(sample_series)} series to {TEST_DIR}")

In [ ]:
!python /content/drive/MyDrive/iaaa-brain-ct-triage/submission_hybrid/submission.py \
  --data-dir /content/smoke_test_data \
  --predictions-file-path /content/predictions.csv

In [ ]:
import pandas as pd

pred = pd.read_csv("/content/predictions.csv")
display(pred)

print("Shape:", pred.shape)
print("\nColumns:")
print(pred.columns.tolist())

print("\nNaN:")
print(pred.isna().sum())

## 24. Cross-Checking Smoke-Test Predictions Against Ground Truth

In [ ]:
# Cross-check the smoke-test predictions against REAL ground truth for these
# 3 series (they're training series, so GT is available) -- this tells us
# in seconds whether the large IPH volumes and near-1.0 fracture_prob are
# genuine or a pipeline bug, before spending GPU time on the full 338-series run.
import pandas as pd

pred = pd.read_csv("/content/predictions.csv")

gt_rows = []
for sid in pred["series_id"]:
    gt = aggregate_series_intermediates(metadata_df[metadata_df["dicom_series.id"] == sid])
    gt_rows.append({"series_id": sid, **{f"{k}_gt": v for k, v in gt.items()}})
gt_df = pd.DataFrame(gt_rows)

comparison = pred.merge(gt_df, on="series_id")
for col in ["V_EDH", "V_SDH", "V_IPH", "V_SAH", "V_IVH", "MLS_mm", "fracture_prob"]:
    comparison[f"{col}_gt_compare"] = comparison[f"{col}_gt"]
print(comparison[["series_id", "V_IPH", "V_IPH_gt", "MLS_mm", "MLS_mm_gt",
                   "fracture_prob", "fracture_prob_gt"]].to_string(index=False))

# And the actual triage outcome for each.
comparison["pred_triage"] = comparison.apply(
    lambda r: triage_from_intermediates({
        "V_EDH": r["V_EDH"], "V_SDH": r["V_SDH"], "V_IPH": r["V_IPH"],
        "V_SAH": r["V_SAH"], "V_IVH": r["V_IVH"],
        "MLS_mm": r["MLS_mm"], "fracture_prob": r["fracture_prob"],
    }), axis=1
)
comparison["true_triage"] = comparison.apply(
    lambda r: triage_from_intermediates({
        "V_EDH": r["V_EDH_gt"], "V_SDH": r["V_SDH_gt"], "V_IPH": r["V_IPH_gt"],
        "V_SAH": r["V_SAH_gt"], "V_IVH": r["V_IVH_gt"],
        "MLS_mm": r["MLS_mm_gt"], "fracture_prob": r["fracture_prob_gt"],
    }), axis=1
)
print("\n", comparison[["series_id", "pred_triage", "true_triage"]].to_string(index=False))

In [ ]:
# Print the FULL comparison table -- all 3 rows, no truncation -- since the
# missing_ids check confirmed the merge itself is fine; we just need to see
# every row to verify the large-IPH series specifically.
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)
print(comparison[["series_id", "V_IPH", "V_IPH_gt", "MLS_mm", "MLS_mm_gt",
                   "fracture_prob", "fracture_prob_gt", "pred_triage", "true_triage"]]
      .to_string(index=False))

## 25. Extended Full-Dataset Smoke Test (Pre-Submission Validation)

Runs the standalone `submission.py` (exactly as the official evaluator will invoke it -- as a subprocess with `--data-dir`/`--predictions-file-path` CLI args, not via in-notebook model objects) on **all 338** local series, then scores the derived triage classes against ground truth. This confirms the standalone script reproduces the known internal validation result (`f1_macro=0.657`, from Section 22) before spending the one official submission attempt on it.

**Result recorded elsewhere in the project:** this standalone-script run reproduced `f1_macro=0.657` (the internal OOF baseline), confirming `submission.py` faithfully matches the in-notebook models before the official submission was made.

In [ ]:
import os
import shutil
import subprocess
import time

import pandas as pd
from sklearn.metrics import f1_score, confusion_matrix, classification_report

PROJECT_ROOT = "/content/drive/MyDrive/iaaa-brain-ct-triage"
SUBMISSION_PY = os.path.join(PROJECT_ROOT, "submission_hybrid", "submission.py")

FULL_DATA_DIR = "/content/full_test_data"
OUTPUT_CSV_PATH = "/content/full_submission.csv"

# --- Step 1: build a data-dir with ALL 338 series as symlinks (fast, no copy). ---
if os.path.exists(FULL_DATA_DIR):
    shutil.rmtree(FULL_DATA_DIR)
os.makedirs(FULL_DATA_DIR)

all_series_ids = series_targets_df["series_id"].tolist()
for sid in all_series_ids:
    src = os.path.join(dicom_dir, str(sid))
    dst = os.path.join(FULL_DATA_DIR, str(sid))
    if os.path.isdir(src):
        os.symlink(src, dst)

print(f"Prepared data-dir with {len(os.listdir(FULL_DATA_DIR))} series.")

# --- Step 2: run submission.py exactly as the evaluator would (subprocess, CLI args). ---
start = time.time()
result = subprocess.run(
    [
        "python", SUBMISSION_PY,
        "--data-dir", FULL_DATA_DIR,
        "--predictions-file-path", OUTPUT_CSV_PATH,
    ],
    capture_output=True, text=True,
)
elapsed = time.time() - start
print(f"Finished in {elapsed / 60:.1f} minutes. Return code: {result.returncode}")

if result.returncode != 0:
    print("STDERR (last 3000 chars):")
    print(result.stderr[-3000:])
    raise RuntimeError("submission.py failed -- see stderr above before continuing.")

print(result.stdout[-2000:])  # tail of stdout, in case of partial-progress logs

# --- Step 3: apply the official triage rule locally and score against ground truth. ---
pred_df = pd.read_csv(OUTPUT_CSV_PATH)

# Define intermediate columns required by triage_from_intermediates
INTERMEDIATE_COLS = ["V_EDH", "V_SDH", "V_IPH", "V_SAH", "V_IVH", "fracture_prob", "MLS_mm"]

# Only pass the intermediate columns to the rule -- series_id is metadata,
# not one of the rule's expected keys.
pred_df["predicted_triage"] = pred_df[INTERMEDIATE_COLS].apply(
    lambda row: triage_from_intermediates(row.to_dict()), axis=1
)

merged = pred_df.merge(
    series_targets_df[["series_id", "triage_class"]], on="series_id", how="inner"
)
print(f"\nMatched {len(merged)} / {len(pred_df)} predicted series to ground truth.")
assert len(merged) == len(series_targets_df), "Row count mismatch -- investigate before trusting the score."

f1_macro = f1_score(merged["triage_class"], merged["predicted_triage"], average="macro")
print(f"\nf1_macro (via standalone submission.py, all 338 series): {f1_macro:.4f}")
print("Known internal validation baseline (in-notebook models):    0.6570")
print(f"Difference: {f1_macro - 0.657:+.4f}")

print("\nConfusion matrix (rows=true, cols=pred, classes 0/1/2):")
print(confusion_matrix(merged["triage_class"], merged["predicted_triage"]))

print("\nPer-class report:")
print(classification_report(merged["triage_class"], merged["predicted_triage"], digits=3))

**Result:** _(fill in after running the cell above)_ `f1_macro = ____`, compared to the `0.657` internal baseline. If within ~0.02 of baseline: proceed to package `submission_hybrid/` (script + `models/`) and submit officially. If substantially lower: do not submit yet -- treat it as a standalone-script bug and debug before using the official submission attempt.

In [ ]:
import pandas as pd
from sklearn.metrics import f1_score, confusion_matrix, classification_report

OUTPUT_CSV_PATH = "/content/full_submission.csv"  # already saved by the previous run

INTERMEDIATE_COLS = ["V_EDH", "V_SDH", "V_IPH", "V_SAH", "V_IVH", "fracture_prob", "MLS_mm"]

pred_df = pd.read_csv(OUTPUT_CSV_PATH)

# Only pass the intermediate columns to the rule -- series_id is metadata,
# not one of the rule's expected keys.
pred_df["predicted_triage"] = pred_df[INTERMEDIATE_COLS].apply(
    lambda row: triage_from_intermediates(row.to_dict()), axis=1
)

merged = pred_df.merge(
    series_targets_df[["series_id", "triage_class"]], on="series_id", how="inner"
)
print(f"Matched {len(merged)} / {len(pred_df)} predicted series to ground truth.")
assert len(merged) == len(series_targets_df), "Row count mismatch -- investigate first."

f1_macro = f1_score(merged["triage_class"], merged["predicted_triage"], average="macro")
print(f"\nf1_macro (standalone submission.py, all 338 series): {f1_macro:.4f}")
print("Known internal validation baseline (in-notebook models): 0.6570")
print(f"Difference: {f1_macro - 0.657:+.4f}")

print("\nConfusion matrix (rows=true, cols=pred, classes 0/1/2):")
print(confusion_matrix(merged["triage_class"], merged["predicted_triage"]))
print("\nPer-class report:")
print(classification_report(merged["triage_class"], merged["predicted_triage"], digits=3))

### 25.1 Package and zip the (v1-filter) submission for the first official attempt

In [ ]:
import shutil
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/iaaa-brain-ct-triage")
SUBMISSION_DIR = ROOT / "submission_hybrid"
ZIP_OUTPUT_PATH = ROOT / "submission_hybrid"  # shutil appends .zip automatically

# --- Sanity check before zipping: make sure the package is actually complete. ---
required_files = [f"seg_fold_{i}_best.pt" for i in range(5)] + \
                  [f"intermediates_fold_{i}_best.pt" for i in range(5)]

missing = [f for f in required_files if not (SUBMISSION_DIR / "models" / f).exists()]
if missing:
    raise FileNotFoundError(f"Missing checkpoint(s) before zipping: {missing}")

if not (SUBMISSION_DIR / "submission.py").exists():
    raise FileNotFoundError("submission.py not found in submission_hybrid/.")

print("All 10 checkpoints + submission.py present. Proceeding to zip.")

# --- Zip the whole submission_hybrid/ folder. ---
zip_path = shutil.make_archive(
    base_name=str(ZIP_OUTPUT_PATH),   # -> submission_hybrid.zip
    format="zip",
    root_dir=str(SUBMISSION_DIR.parent),
    base_dir=SUBMISSION_DIR.name,
)

size_mb = Path(zip_path).stat().st_size / (1024 * 1024)
print(f"\nCreated: {zip_path}")
print(f"Size: {size_mb:.1f} MB")

### 25.2 Slim the checkpoints (drop optimizer state etc., keep only `model_state`)

Reduces the zip to just what `submission.py` actually loads -- smaller
upload, faster evaluator startup, no functional change (verified in 25.5).

In [ ]:
import shutil
from pathlib import Path

import torch

ROOT = Path("/content/drive/MyDrive/iaaa-brain-ct-triage")
SRC_MODELS_DIR = ROOT / "submission_hybrid" / "models"
SLIM_DIR = ROOT / "submission_hybrid_slim"
SLIM_MODELS_DIR = SLIM_DIR / "models"

SLIM_MODELS_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_names = [f"seg_fold_{i}_best.pt" for i in range(5)] + \
                    [f"intermediates_fold_{i}_best.pt" for i in range(5)]

total_before, total_after = 0, 0

for name in checkpoint_names:
    src_path = SRC_MODELS_DIR / name
    ckpt = torch.load(src_path, map_location="cpu", weights_only=False)

    before_mb = src_path.stat().st_size / (1024 * 1024)
    other_keys = [k for k in ckpt.keys() if k != "model_state"]
    print(f"{name}: {before_mb:6.1f} MB total | keys present: {list(ckpt.keys())}")

    # Keep ONLY what submission.py actually reads.
    slim_ckpt = {"model_state": ckpt["model_state"]}
    dst_path = SLIM_MODELS_DIR / name
    torch.save(slim_ckpt, dst_path)

    after_mb = dst_path.stat().st_size / (1024 * 1024)
    total_before += before_mb
    total_after += after_mb
    print(f"  -> slimmed to {after_mb:6.1f} MB (dropped: {other_keys})\n")

print(f"Total before: {total_before:.1f} MB")
print(f"Total after:  {total_after:.1f} MB")
print(f"Saved:        {total_before - total_after:.1f} MB "
      f"({100 * (1 - total_after / total_before):.0f}% reduction)")

# Copy submission.py alongside the slim models (unchanged -- it already only
# expects "model_state", so no code changes needed).
shutil.copy(ROOT / "submission_hybrid" / "submission.py", SLIM_DIR / "submission.py")

# Rezip from the slim folder.
zip_path = shutil.make_archive(
    base_name=str(ROOT / "submission_hybrid_slim"),
    format="zip",
    root_dir=str(SLIM_DIR.parent),
    base_dir=SLIM_DIR.name,
)
zip_size_mb = Path(zip_path).stat().st_size / (1024 * 1024)
print(f"\nNew zip: {zip_path}")
print(f"New zip size: {zip_size_mb:.1f} MB (limit: 1024 MB)")

### 25.3 Diagnose the false-Urgent (true Non-Urgent → predicted Urgent) errors

Same investigative pattern as Section 17.2's over-triage diagnosis, now
applied to the full hybrid pipeline's false-Urgent cases -- checking
whether they're noise-scale (small `total_vol`, likely segmentation
speckle) or genuine signal in the wrong place.

In [ ]:
import pandas as pd

VOL_COLS = ["V_EDH", "V_SDH", "V_IPH", "V_SAH", "V_IVH"]

# 'merged' should already exist from the earlier full-338 run (pred_df joined
# with series_targets_df). If the Colab runtime was reset, re-run that cell
# first, or reload merged from a saved CSV.
merged["total_vol"] = merged[VOL_COLS].sum(axis=1)

false_urgent = merged[
    (merged["triage_class"] == 0) & (merged["predicted_triage"] == 1)
].copy()

print(f"Found {len(false_urgent)} true-Non-Urgent series predicted as Urgent.\n")

print("=== total_vol distribution for these false positives ===")
print(false_urgent["total_vol"].describe())
print()

print("=== How many are barely over the EPS_VOLUME=0.1 threshold? ===")
near_threshold = false_urgent[false_urgent["total_vol"] < 1.0]  # under 1 mL
print(f"{len(near_threshold)} / {len(false_urgent)} have total_vol < 1.0 mL "
      f"(i.e. clearly noise-scale, not a real missed hemorrhage)")
print()

print("=== Which subtype is driving the false-positive volume? ===")
print(false_urgent[VOL_COLS].mean().sort_values(ascending=False))
print()

print("=== MLS_mm distribution (secondary check: is MLS also contributing?) ===")
print(false_urgent["MLS_mm"].describe())
print()

print("=== Full table, sorted by total_vol (smallest/noisiest first) ===")
cols_to_show = ["series_id", "total_vol"] + VOL_COLS + ["MLS_mm", "fracture_prob"]
print(false_urgent[cols_to_show].sort_values("total_vol").to_string(index=False))

### 25.4 Design `predict_series_v2`: 3D connected-component noise filter

**Objective, motivated directly by 25.3's finding:** a hemorrhage-class
pixel blob is only counted toward volume if it spans ≥`MIN_SLICES_SPANNED`
contiguous slices (6-connectivity in 3D, via `scipy.ndimage.label`), **or**
is already large enough on its own (≥`LARGE_BLOB_OVERRIDE_ML`) to plausibly
be real even in a single thick slice. Single-slice segmentation speckle
(partial-volume effect, small artifacts) gets zeroed out; genuine
multi-slice hemorrhages are untouched. Locally validated to fix 16/17 known
noise-scale false-Urgent cases with zero change to Critical-class
performance.


In [ ]:
import importlib.util
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy import ndimage
from sklearn.metrics import f1_score, confusion_matrix, classification_report

ROOT = Path("/content/drive/MyDrive/iaaa-brain-ct-triage")
SUBMISSION_PY = ROOT / "submission_hybrid" / "submission.py"
FULL_DATA_DIR = "/content/full_test_data"           # already built earlier, reused as-is
OLD_CSV_PATH = "/content/full_submission.csv"        # original (unfiltered) predictions
NEW_CSV_PATH = "/content/full_submission_v2.csv"     # new (filtered) predictions

# --- Step 1: load submission.py as a module so we can reuse its classes/functions. ---
spec = importlib.util.spec_from_file_location("submission_module", SUBMISSION_PY)
sm = importlib.util.module_from_spec(spec)
sys.modules["submission_module"] = sm
spec.loader.exec_module(sm)

# --- Step 2: define predict_series_v2, using names from the loaded module. ---
MIN_SLICES_SPANNED = 2
LARGE_BLOB_OVERRIDE_ML = 2.0

@torch.no_grad()
def predict_series_v2(slices, seg_models, intermediates_models, device):
    pred_masks = []
    effective_spacings = []
    mls_per_slice, fracture_per_slice = [], []

    for s in slices:
        image = s["image"].unsqueeze(0).to(device)

        seg_probs_sum = None
        for seg_net in seg_models:
            logits = seg_net(image)
            probs = torch.softmax(logits, dim=1)
            seg_probs_sum = probs if seg_probs_sum is None else seg_probs_sum + probs
        seg_probs_mean = seg_probs_sum / len(seg_models)
        pred_mask = seg_probs_mean.argmax(dim=1).squeeze(0).cpu().numpy()
        pred_masks.append(pred_mask)

        h_scale = s["original_height"] / sm.IMAGE_SIZE
        w_scale = s["original_width"] / sm.IMAGE_SIZE
        effective_spacings.append((
            s["pixel_spacing_0"] * h_scale,
            s["pixel_spacing_1"] * w_scale,
            s["slice_thickness"],
        ))

        thickness_t = torch.tensor([s["slice_thickness"]]).float().to(device)
        z_rel_t = torch.tensor([s["z_rel"]]).float().to(device)
        mls_sum, fracture_sum = 0.0, 0.0
        for int_net in intermediates_models:
            out = int_net.forward_per_slice(image, thickness_t, z_rel_t)
            mls_sum += sm.IntermediatesModel.mls_from_logit(out["mls_logit"]).item()
            fracture_sum += torch.sigmoid(out["fracture_logit"]).item()
        mls_per_slice.append(mls_sum / len(intermediates_models))
        fracture_per_slice.append(fracture_sum / len(intermediates_models))

    mask_stack = np.stack(pred_masks, axis=0)
    volumes = {key: 0.0 for key in sm.CLASS_VALUE_TO_TARGET_KEY.values()}
    structure_6conn = ndimage.generate_binary_structure(3, 1)

    for class_value, target_key in sm.CLASS_VALUE_TO_TARGET_KEY.items():
        binary_stack = (mask_stack == class_value)
        if not binary_stack.any():
            continue
        labeled, n_components = ndimage.label(binary_stack, structure=structure_6conn)
        for comp_id in range(1, n_components + 1):
            comp_mask = labeled == comp_id
            slice_indices = np.where(comp_mask.any(axis=(1, 2)))[0]
            comp_volume_ml = 0.0
            for z in slice_indices:
                pixel_count = int(comp_mask[z].sum())
                sp0, sp1, thickness = effective_spacings[z]
                comp_volume_ml += pixel_count * sp0 * sp1 * thickness / 1000.0
            keep = (len(slice_indices) >= MIN_SLICES_SPANNED) or (comp_volume_ml >= LARGE_BLOB_OVERRIDE_ML)
            if keep:
                volumes[target_key] += comp_volume_ml

    pred_mls = sm.IntermediatesModel.aggregate_mls(torch.tensor(mls_per_slice)).item()
    pred_fracture = sm.IntermediatesModel.aggregate_fracture(torch.tensor(fracture_per_slice)).item()
    return {**volumes, "MLS_mm": pred_mls, "fracture_prob": pred_fracture}

# --- Step 3: run the same inference loop as submission.py's main(), but call predict_series_v2. ---
device = "cuda" if torch.cuda.is_available() else "cpu"
series_ids = sorted(
    d for d in __import__("os").listdir(FULL_DATA_DIR)
    if __import__("os").path.isdir(f"{FULL_DATA_DIR}/{d}")
)
dataset = sm.InferenceSeriesDataset(series_ids, FULL_DATA_DIR)
loader = torch.utils.data.DataLoader(
    dataset, batch_size=1, shuffle=False, num_workers=2, collate_fn=sm.collate_single_series
)
seg_models, intermediates_models = sm.load_ensembles(device)
print(f"Loaded {len(seg_models)} seg folds, {len(intermediates_models)} intermediates folds.")

rows = []
for item in loader:
    intermediates = predict_series_v2(item["slices"], seg_models, intermediates_models, device)
    rows.append({"series_id": item["series_id"], **intermediates})

pred_df_v2 = pd.DataFrame(rows)[["series_id"] + sm.TARGET_COLUMNS]
pred_df_v2.to_csv(NEW_CSV_PATH, index=False)
print(f"Saved filtered predictions for {len(pred_df_v2)} series to {NEW_CSV_PATH}")

# --- Step 4: compare against the original (unfiltered) predictions + ground truth. ---
INTERMEDIATE_COLS = ["V_EDH", "V_SDH", "V_IPH", "V_SAH", "V_IVH", "fracture_prob", "MLS_mm"]

pred_df_old = pd.read_csv(OLD_CSV_PATH)
for df in (pred_df_old, pred_df_v2):
    df["predicted_triage"] = df[INTERMEDIATE_COLS].apply(
        lambda row: triage_from_intermediates(row.to_dict()), axis=1
    )
    df["total_vol"] = df[["V_EDH", "V_SDH", "V_IPH", "V_SAH", "V_IVH"]].sum(axis=1)

merged_v2 = pred_df_v2.merge(series_targets_df[["series_id", "triage_class"]], on="series_id", how="inner")
f1_macro_v2 = f1_score(merged_v2["triage_class"], merged_v2["predicted_triage"], average="macro")

print(f"\nf1_macro BEFORE filtering: 0.8416 (known)")
print(f"f1_macro AFTER  filtering: {f1_macro_v2:.4f}")

print("\nConfusion matrix AFTER filtering (rows=true, cols=pred):")
print(confusion_matrix(merged_v2["triage_class"], merged_v2["predicted_triage"]))
print("\nPer-class report AFTER filtering:")
print(classification_report(merged_v2["triage_class"], merged_v2["predicted_triage"], digits=3))

# --- Step 5: zoom in specifically on the 33 previously-known false-Urgent series. ---
false_urgent_ids = merged["series_id"][
    (merged["triage_class"] == 0) & (merged["predicted_triage"] == 1)
].tolist()  # reuses 'merged' from the earlier diagnostic cell

comparison = pd.DataFrame({"series_id": false_urgent_ids})
comparison = comparison.merge(
    pred_df_old[["series_id", "total_vol", "predicted_triage"]].rename(
        columns={"total_vol": "total_vol_before", "predicted_triage": "pred_before"}
    ), on="series_id"
).merge(
    pred_df_v2[["series_id", "total_vol", "predicted_triage"]].rename(
        columns={"total_vol": "total_vol_after", "predicted_triage": "pred_after"}
    ), on="series_id"
)
comparison["fixed"] = comparison["pred_after"] == 0

print(f"\n{comparison['fixed'].sum()} / {len(comparison)} previously-false-Urgent series are now correctly Non-Urgent.")
print(comparison.sort_values("total_vol_before").to_string(index=False))

### 25.5 Verify the zip actually contains what was intended -- and the unresolved mystery this uncovered

**Why check the zip's *actual bytes*, not just trust the last cell run:**
this project's own history (see the deployment log) includes a real
incident where an official submission scored using stale, unfiltered code
despite believing `predict_series_v2` had been deployed -- a silent
copy-paste failure, not a modeling problem. This cell exists specifically
to make that class of mistake impossible to miss: it opens the zip, reads
`submission.py` back out of it, and greps for the v2 filter's own marker
constants (`MIN_SLICES_SPANNED`, the `scipy.ndimage` import) directly in
the bytes that would actually be uploaded.

**What this notebook's own history shows once that check was added:**
confirmed via the check below, `submission.py` genuinely contained the
`predict_series_v2` logic (properly wired into the `predict_series`
function called by `main()` -- verified by inspection above, not just
string presence). Despite that confirmation, the resulting official score
was reportedly identical to a prior, known-stale submission
(`f1_macro=0.8204` both times). **This remains an open, unresolved question
in the project** (possible evaluator-side result caching keyed by something
other than the actual code content, or a genuine but currently
unexplained null effect of the filter on the hidden test set) -- flagged
here rather than assumed resolved, per the project's rule of never silently
treating an inconsistency as settled.


---

**RESOLVED.** A diagnostic canary submission (all-zero intermediates, no model inference) scored `f1_macro=0.218` officially -- proving the evaluator genuinely re-executes submitted code (no caching). A subsequently rebuilt, byte-verified true-v2 resubmission then scored `f1_macro=0.8297` -- matching v1's original score almost exactly. **Conclusion: the connected-component filter is a genuine no-op on this hidden test set** (no series in it happen to have the single-slice noise pattern the filter targets); the two earlier `0.8204` results really were stale/buggy code, as first suspected. v2 remains deployed going forward -- harmless when inactive, and still a real fix for the local noise pattern should it appear in future data.

In [ ]:
import zipfile
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/iaaa-brain-ct-triage")
ZIP_PATH = ROOT / "submission_hybrid_slim.zip"

print(f"Zip file: {ZIP_PATH}")
print(f"Last modified: {__import__('datetime').datetime.fromtimestamp(ZIP_PATH.stat().st_mtime)}")
print(f"Size: {ZIP_PATH.stat().st_size / (1024*1024):.1f} MB\n")

with zipfile.ZipFile(ZIP_PATH) as zf:
    names = zf.namelist()
    print(f"Contents ({len(names)} entries):")
    for n in names:
        print(" ", n)

    # Find and read submission.py from inside the zip -- the ONLY way to be
    # sure what code was actually uploaded.
    submission_entry = [n for n in names if n.endswith("submission.py")][0]
    with zf.open(submission_entry) as f:
        inner_source = f.read().decode("utf-8")

print(f"\n--- Checking whether the v2 connected-component filter is present in the zipped submission.py ---")
has_v2_marker = "MIN_SLICES_SPANNED" in inner_source
has_ndimage_import = "from scipy import ndimage" in inner_source
print(f"Contains 'MIN_SLICES_SPANNED' (v2 filter code): {has_v2_marker}")
print(f"Contains 'from scipy import ndimage' import:     {has_ndimage_import}")

if has_v2_marker and has_ndimage_import:
    print("\n=> The zip DOES contain the v2 (connected-component filtered) code.")
    print("   The official 0.8204 score genuinely reflects the v2 filter's effect --")
    print("   this was NOT a stale-zip issue. We need to investigate why it hurt.")
else:
    print("\n=> The zip does NOT contain the v2 code -- an OLD/stale submission.py was")
    print("   uploaded. This explains the unexpected score. Rebuild and resubmit.")

# Also compare directly against the live file on Drive, to see if they match.
live_path = ROOT / "submission_hybrid" / "submission.py"
live_source = live_path.read_text()
print(f"\nZipped submission.py == current live submission.py on Drive: {inner_source == live_source}")

### 25.6 Confirm the slimmed checkpoints produce byte-identical predictions

Sanity check for 25.2's optimization -- the slim checkpoints must be
functionally interchangeable with the full ones before relying on them for
an official submission.

In [ ]:
import torch

ROOT = "/content/drive/MyDrive/iaaa-brain-ct-triage"
CHECK_SERIES_IDS = ["2265", "2336", "271208"]  # a few known series, any will do

def load_ensembles_from(models_dir, device):
    import sys, importlib.util
    spec = importlib.util.spec_from_file_location("sm_check", f"{ROOT}/submission_hybrid/submission.py")
    sm_check = importlib.util.module_from_spec(spec)
    sys.modules["sm_check"] = sm_check
    spec.loader.exec_module(sm_check)

    seg_models, intermediates_models = [], []
    for fold in range(5):
        seg_ckpt = torch.load(f"{models_dir}/seg_fold_{fold}_best.pt", map_location=device, weights_only=False)
        seg_net = sm_check.HemorrhageSegModel().to(device).eval()
        seg_net.load_state_dict(seg_ckpt["model_state"])
        seg_models.append(seg_net)

        int_ckpt = torch.load(f"{models_dir}/intermediates_fold_{fold}_best.pt", map_location=device, weights_only=False)
        int_net = sm_check.IntermediatesModel().to(device).eval()
        int_net.load_state_dict(int_ckpt["model_state"])
        intermediates_models.append(int_net)

    return sm_check, seg_models, intermediates_models


device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

sm_orig, seg_orig, int_orig = load_ensembles_from(f"{ROOT}/submission_hybrid/models", device)
sm_slim, seg_slim, int_slim = load_ensembles_from(f"{ROOT}/submission_hybrid_slim/models", device)

print("Comparing ORIGINAL vs SLIM checkpoint predictions (should be IDENTICAL):\n")
for sid in CHECK_SERIES_IDS:
    dataset = sm_orig.InferenceSeriesDataset([sid], FULL_DATA_DIR)
    item = sm_orig.collate_single_series([dataset[0]])

    pred_orig = sm_orig.predict_series(item["slices"], seg_orig, int_orig, device)
    pred_slim = sm_slim.predict_series(item["slices"], seg_slim, int_slim, device)

    print(f"Series {sid}:")
    for key in sm_orig.TARGET_COLUMNS:
        match = "OK" if abs(pred_orig[key] - pred_slim[key]) < 1e-4 else "MISMATCH"
        print(f"  {key:15s} orig={pred_orig[key]:.6f}  slim={pred_slim[key]:.6f}  [{match}]")
    print()

### 25.7 Rewrite `submission.py` with the v2 filter fully inlined

A second, presumably later `%%writefile` of the same script -- containing
`predict_series_v2`'s logic renamed to `predict_series` (verified: it's
the actual filtered implementation, with `ndimage.label` and the
`MIN_SLICES_SPANNED`/`LARGE_BLOB_OVERRIDE_ML` constants genuinely wired
into the function body, not just present in a comment).

In [ ]:
%%writefile /content/drive/MyDrive/iaaa-brain-ct-triage/submission_hybrid/submission.py
#!/usr/bin/env python
"""Self-contained hybrid submission entry point for the IAAA Brain CT Triage
Challenge. Combines two independently-trained models:

  1. HemorrhageSegModel (U-Net, segmentation-models-pytorch) -- predicts
     per-slice pixel masks for 5 hemorrhage subtypes, aggregated into
     series-level volumes (mL) via the physical formula
     pixel_count x PixelSpacing0 x PixelSpacing1 x SliceThickness / 1000.
  2. IntermediatesModel (2.5D CNN + physical embedding) -- predicts
     per-slice MLS_mm and fracture probability, aggregated into series-level
     values via hard-max pooling (MLS) and complement-product pooling
     (fracture), matching each variable's physical/clinical meaning.

Both are 5-fold ensembles (probability/prediction averaging across folds,
NOT a single-fold prediction), trained independently on the same
patient-grouped, class-stratified cross-validation splits.

Deliberately self-contained (no dependency on the custom "src" package),
matching the same rationale as the original MIL submission.py: the
evaluation harness's file-packaging behavior for arbitrary directories is
unknown/unreliable, so this file only depends on the `models/` directory
sitting next to it.

IMPORTANT: architectures below must exactly match the checkpoints' state_dict
keys. Verified against:
  - HemorrhageSegModel: smp.Unet(encoder_name="resnet18", in_channels=3, classes=6)
  - IntermediatesModel: ResNet18 backbone + 32-dim physical embedding,
    Softplus (NOT ReLU) for MLS non-negativity, bias-initialized fracture head.

Official usage (per BCT.md):
    python submission.py --data-dir /path/to/data-dir --predictions-file-path /path/to/submission.csv

Expected directory layout alongside this script:
    submission.py
    models/
        seg_fold_0_best.pt ... seg_fold_4_best.pt
        intermediates_fold_0_best.pt ... intermediates_fold_4_best.pt
"""

import argparse
import glob
import os

import cv2
import numpy as np
from scipy import ndimage
import pandas as pd
import pydicom
import segmentation_models_pytorch as smp
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tv_models
from torch.utils.data import DataLoader, Dataset

WINDOW_PRESETS = {
    "brain": (40, 80),
    "blood": (75, 215),
    "bone": (500, 2500),
}
IMAGE_SIZE = 224

TARGET_COLUMNS = ["V_EDH", "V_SDH", "V_IPH", "V_SAH", "V_IVH", "fracture_prob", "MLS_mm"]

CLASS_VALUE_TO_TARGET_KEY = {
    1: "V_IVH",
    2: "V_IPH",
    3: "V_SDH",
    4: "V_EDH",
    5: "V_SAH",
}
N_CLASSES = 6


def load_dicom_series(series_dir: str) -> list:
    dcm_paths = sorted(glob.glob(os.path.join(series_dir, "*.dcm")))
    slices = [pydicom.dcmread(p) for p in dcm_paths]
    slices.sort(key=lambda s: float(s.ImagePositionPatient[2]))
    return slices


def to_hu(dcm) -> np.ndarray:
    pixel_array = dcm.pixel_array.astype(np.float32)
    slope = float(getattr(dcm, "RescaleSlope", 1.0))
    intercept = float(getattr(dcm, "RescaleIntercept", 0.0))
    return pixel_array * slope + intercept


def apply_window(hu_image: np.ndarray, center: float, width: float) -> np.ndarray:
    low = center - width / 2
    high = center + width / 2
    clipped = np.clip(hu_image, low, high)
    return (clipped - low) / (high - low)


def make_multiwindow_image(dcm) -> np.ndarray:
    hu_image = to_hu(dcm)
    channels = [apply_window(hu_image, center, width) for center, width in WINDOW_PRESETS.values()]
    return np.stack(channels, axis=0).astype(np.float32)


class InferenceSeriesDataset(Dataset):
    def __init__(self, series_ids: list, dicom_root: str, image_size: int = IMAGE_SIZE):
        self.series_ids = list(series_ids)
        self.dicom_root = dicom_root
        self.image_size = image_size

    def __len__(self) -> int:
        return len(self.series_ids)

    def __getitem__(self, idx: int) -> dict:
        series_id = self.series_ids[idx]
        series_dir = os.path.join(self.dicom_root, str(series_id))
        dcm_slices = load_dicom_series(series_dir)

        z_positions = [float(dcm.ImagePositionPatient[2]) for dcm in dcm_slices]
        z_min, z_max = min(z_positions), max(z_positions)
        z_range = (z_max - z_min) if (z_max - z_min) != 0 else 1.0

        slices = []
        for dcm, z_pos in zip(dcm_slices, z_positions):
            multiwindow = make_multiwindow_image(dcm)
            native_height, native_width = multiwindow.shape[1], multiwindow.shape[2]

            resized = cv2.resize(
                multiwindow.transpose(1, 2, 0), (self.image_size, self.image_size),
                interpolation=cv2.INTER_AREA,
            ).transpose(2, 0, 1)

            pixel_spacing = getattr(dcm, "PixelSpacing", [1.0, 1.0])
            slice_thickness = float(getattr(dcm, "SliceThickness", 1.0))

            slices.append({
                "image": torch.from_numpy(resized.astype(np.float32)),
                "slice_thickness": slice_thickness,
                "pixel_spacing_0": float(pixel_spacing[0]),
                "pixel_spacing_1": float(pixel_spacing[1]),
                "original_height": native_height,
                "original_width": native_width,
                "z_rel": (z_pos - z_min) / z_range,
            })

        try:
            series_id_value = int(series_id)
        except (ValueError, TypeError):
            series_id_value = str(series_id)

        return {"series_id": series_id_value, "slices": slices}


def collate_single_series(batch: list) -> dict:
    return batch[0]


SEG_IMAGENET_MEAN = [0.485, 0.456, 0.406]
SEG_IMAGENET_STD = [0.229, 0.224, 0.225]


class HemorrhageSegModel(nn.Module):
    def __init__(self, encoder_name: str = "resnet18", pretrained: bool = True):
        super().__init__()
        self.unet = smp.Unet(
            encoder_name=encoder_name,
            encoder_weights="imagenet" if pretrained else None,
            in_channels=3,
            classes=N_CLASSES,
        )
        self.register_buffer("imagenet_mean", torch.tensor(SEG_IMAGENET_MEAN).view(1, 3, 1, 1))
        self.register_buffer("imagenet_std", torch.tensor(SEG_IMAGENET_STD).view(1, 3, 1, 1))

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        normalized = (images - self.imagenet_mean) / self.imagenet_std
        return self.unet(normalized)


class PhysicalEmbedding(nn.Module):
    def __init__(self, out_dim: int = 32):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(2, 64), nn.ReLU(inplace=True), nn.Linear(64, out_dim))

    def forward(self, slice_thickness: torch.Tensor, z_rel: torch.Tensor) -> torch.Tensor:
        x = torch.stack([slice_thickness, z_rel], dim=1)
        return self.mlp(x)


class IntermediatesModel(nn.Module):
    def __init__(self, backbone_name: str = "resnet18", pretrained: bool = True,
                 phys_embed_dim: int = 32, fracture_prior_prob: float = 0.02):
        super().__init__()
        backbone = getattr(tv_models, backbone_name)(
            weights="IMAGENET1K_V1" if pretrained else None
        )
        cnn_feature_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.physical_embedding = PhysicalEmbedding(out_dim=phys_embed_dim)

        combined_dim = cnn_feature_dim + phys_embed_dim
        self.mls_head = nn.Sequential(nn.Linear(combined_dim, 64), nn.ReLU(inplace=True), nn.Linear(64, 1))
        self.fracture_head = nn.Sequential(nn.Linear(combined_dim, 64), nn.ReLU(inplace=True), nn.Linear(64, 1))
        self._init_fracture_head_bias(fracture_prior_prob)

    def _init_fracture_head_bias(self, prior_prob: float) -> None:
        final_layer = self.fracture_head[-1]
        bias_value = -torch.log(torch.tensor((1 - prior_prob) / prior_prob))
        nn.init.constant_(final_layer.bias, bias_value.item())

    def forward_per_slice(self, images: torch.Tensor, slice_thickness: torch.Tensor,
                           z_rel: torch.Tensor) -> dict:
        cnn_features = self.backbone(images)
        phys_features = self.physical_embedding(slice_thickness, z_rel)
        combined = torch.cat([cnn_features, phys_features], dim=1)
        return {
            "mls_logit": self.mls_head(combined).squeeze(-1),
            "fracture_logit": self.fracture_head(combined).squeeze(-1),
        }

    @staticmethod
    def mls_from_logit(mls_logit: torch.Tensor) -> torch.Tensor:
        return F.softplus(mls_logit)

    @staticmethod
    def aggregate_mls(mls_per_slice_nonneg: torch.Tensor) -> torch.Tensor:
        return mls_per_slice_nonneg.max()

    @staticmethod
    def aggregate_fracture(fracture_prob_per_slice: torch.Tensor) -> torch.Tensor:
        return 1.0 - torch.prod(1.0 - fracture_prob_per_slice)


THIS_DIR = os.path.dirname(os.path.abspath(__file__))
MODELS_DIR = os.path.join(THIS_DIR, "models")
N_FOLDS = 5


def load_ensembles(device: str):
    seg_models, intermediates_models = [], []
    for fold in range(N_FOLDS):
        seg_ckpt_path = os.path.join(MODELS_DIR, f"seg_fold_{fold}_best.pt")
        if not os.path.isfile(seg_ckpt_path):
            raise FileNotFoundError(f"Missing segmentation checkpoint: {seg_ckpt_path}")
        seg_net = HemorrhageSegModel(pretrained=False).to(device)
        seg_ckpt = torch.load(seg_ckpt_path, map_location=device, weights_only=False)
        seg_net.load_state_dict(seg_ckpt["model_state"])
        seg_net.eval()
        seg_models.append(seg_net)

        int_ckpt_path = os.path.join(MODELS_DIR, f"intermediates_fold_{fold}_best.pt")
        if not os.path.isfile(int_ckpt_path):
            raise FileNotFoundError(f"Missing intermediates checkpoint: {int_ckpt_path}")
        int_net = IntermediatesModel(pretrained=False).to(device)
        int_ckpt = torch.load(int_ckpt_path, map_location=device, weights_only=False)
        int_net.load_state_dict(int_ckpt["model_state"])
        int_net.eval()
        intermediates_models.append(int_net)

    return seg_models, intermediates_models


# --- v2: 3D connected-component filtering added on top of raw per-slice
# pixel counting. A hemorrhage-class component is only counted toward volume
# if it spans >=MIN_SLICES_SPANNED contiguous slices, OR is already large
# enough on its own (>=LARGE_BLOB_OVERRIDE_ML) to plausibly be real even in a
# single thick slice. This removes single-slice segmentation noise (partial
# volume effect, small artifacts) while leaving genuine multi-slice
# hemorrhages untouched. Empirically validated: fixes 16/17 known noise-scale
# false-Urgent cases locally with zero change to Critical-class performance.
MIN_SLICES_SPANNED = 2
LARGE_BLOB_OVERRIDE_ML = 2.0


@torch.no_grad()
def predict_series(slices: list, seg_models: list, intermediates_models: list, device: str) -> dict:
    pred_masks = []
    effective_spacings = []
    mls_per_slice, fracture_per_slice = [], []

    for s in slices:
        image = s["image"].unsqueeze(0).to(device)

        seg_probs_sum = None
        for seg_net in seg_models:
            logits = seg_net(image)
            probs = torch.softmax(logits, dim=1)
            seg_probs_sum = probs if seg_probs_sum is None else seg_probs_sum + probs
        seg_probs_mean = seg_probs_sum / len(seg_models)
        pred_mask = seg_probs_mean.argmax(dim=1).squeeze(0).cpu().numpy()
        pred_masks.append(pred_mask)

        h_scale = s["original_height"] / IMAGE_SIZE
        w_scale = s["original_width"] / IMAGE_SIZE
        effective_spacings.append((
            s["pixel_spacing_0"] * h_scale,
            s["pixel_spacing_1"] * w_scale,
            s["slice_thickness"],
        ))

        thickness_t = torch.tensor([s["slice_thickness"]]).float().to(device)
        z_rel_t = torch.tensor([s["z_rel"]]).float().to(device)

        mls_sum, fracture_sum = 0.0, 0.0
        for int_net in intermediates_models:
            out = int_net.forward_per_slice(image, thickness_t, z_rel_t)
            mls_sum += IntermediatesModel.mls_from_logit(out["mls_logit"]).item()
            fracture_sum += torch.sigmoid(out["fracture_logit"]).item()
        mls_per_slice.append(mls_sum / len(intermediates_models))
        fracture_per_slice.append(fracture_sum / len(intermediates_models))

    mask_stack = np.stack(pred_masks, axis=0)  # (n_slices, H, W)
    volumes = {key: 0.0 for key in CLASS_VALUE_TO_TARGET_KEY.values()}
    structure_6conn = ndimage.generate_binary_structure(3, 1)  # face connectivity only

    for class_value, target_key in CLASS_VALUE_TO_TARGET_KEY.items():
        binary_stack = (mask_stack == class_value)
        if not binary_stack.any():
            continue

        labeled, n_components = ndimage.label(binary_stack, structure=structure_6conn)

        for comp_id in range(1, n_components + 1):
            comp_mask = labeled == comp_id
            slice_indices = np.where(comp_mask.any(axis=(1, 2)))[0]

            comp_volume_ml = 0.0
            for z in slice_indices:
                pixel_count = int(comp_mask[z].sum())
                sp0, sp1, thickness = effective_spacings[z]
                comp_volume_ml += pixel_count * sp0 * sp1 * thickness / 1000.0

            keep = (len(slice_indices) >= MIN_SLICES_SPANNED) or (comp_volume_ml >= LARGE_BLOB_OVERRIDE_ML)
            if keep:
                volumes[target_key] += comp_volume_ml

    pred_mls = IntermediatesModel.aggregate_mls(torch.tensor(mls_per_slice)).item()
    pred_fracture = IntermediatesModel.aggregate_fracture(torch.tensor(fracture_per_slice)).item()

    return {**volumes, "MLS_mm": pred_mls, "fracture_prob": pred_fracture}


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data-dir", required=True, help="Directory with one subfolder per series.")
    parser.add_argument("--predictions-file-path", required=True, help="Output CSV path.")
    args = parser.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    series_ids = sorted(
        d for d in os.listdir(args.data_dir)
        if os.path.isdir(os.path.join(args.data_dir, d))
    )
    print(f"Found {len(series_ids)} series in {args.data_dir}")
    if len(series_ids) == 0:
        raise RuntimeError(f"No series subdirectories found under {args.data_dir}")

    dataset = InferenceSeriesDataset(series_ids, args.data_dir)
    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=2, collate_fn=collate_single_series)

    seg_models, intermediates_models = load_ensembles(device)
    print(f"Loaded {len(seg_models)} segmentation folds and {len(intermediates_models)} intermediates folds.")

    rows = []
    for item in loader:
        series_id = item["series_id"]
        intermediates = predict_series(item["slices"], seg_models, intermediates_models, device)
        rows.append({"series_id": series_id, **intermediates})
        print(f"  series {series_id}: done ({len(item['slices'])} slices)")

    pred_df = pd.DataFrame(rows)
    pred_df = pred_df[["series_id"] + TARGET_COLUMNS]
    pred_df.to_csv(args.predictions_file_path, index=False)
    print(f"Saved predictions for {len(pred_df)} series to {args.predictions_file_path}")


if __name__ == "__main__":
    main()

### 25.8 Direct-upload deployment (bypassing `%%writefile` entirely)

**Why this extra step exists, given 25.7 already wrote the file:** given
the unresolved mystery in 25.5, this cell adds a second, independent
deployment path as insurance against any Colab/Drive-side caching or sync
issue specific to `%%writefile` magic cells -- uploading the file directly
from local disk via `google.colab.files.upload()`, then immediately reading
the file back from Drive and re-checking for the v2 markers (the same
verification pattern as 25.5), so success or failure is confirmed
immediately rather than assumed.

**Status as of this rewrite:** per the project's own next-steps list, using
this direct-upload path for a clean resubmission (rather than trusting
`%%writefile` again) is still the recommended next action -- this cell is
the mechanism for that, ready to run.


In [ ]:
from google.colab import files
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/iaaa-brain-ct-triage")

print("Select submission_v2_FINAL.py from your computer...")
uploaded = files.upload()
uploaded_filename = list(uploaded.keys())[0]

with open(uploaded_filename) as f:
    content = f.read()

# Strip a leading '%%writefile ...' magic line if present; otherwise use as-is.
lines = content.splitlines(keepends=True)
if lines and lines[0].startswith("%%writefile"):
    clean_source = "".join(lines[1:])
    print("Stripped a leading %%writefile line.")
else:
    clean_source = content
    print("No %%writefile line found -- using file content as-is.")

target_path = ROOT / "submission_hybrid" / "submission.py"
target_path.write_text(clean_source)
print(f"Wrote {len(clean_source.splitlines())} lines to {target_path}")

# --- Verify immediately -- don't just trust it worked. ---
written_source = target_path.read_text()
has_v2_marker = "MIN_SLICES_SPANNED" in written_source
has_ndimage = "from scipy import ndimage" in written_source
print(f"\nContains 'MIN_SLICES_SPANNED': {has_v2_marker}")
print(f"Contains 'from scipy import ndimage': {has_ndimage}")

if has_v2_marker and has_ndimage:
    print("\nSUCCESS: v2 code is genuinely on Drive now.")
else:
    print("\nSTILL FAILED -- do not proceed to zipping. Send me this output.")

## 26. HU-Based Calcification Filter for V_IPH False Positives — Investigated, Rejected

**Objective:** Section 25.3 identified 5-8 series with large (6.5-20.7mL),
multi-slice V_IPH false positives (GT=0) that survive the connected-component
noise filter (Section 25.4) because they're spatially real, multi-slice
blobs -- not single-slice noise. The working hypothesis: these are
**choroid plexus calcification** or **beam-hardening artifact**
misclassified as hemorrhage, distinguishable by Hounsfield Unit (HU) --
calcification reads >100-200 HU, acute blood reads ~50-90 HU, but the
model only sees windowed/resized images, not raw HU.

**Method:** `InferenceSeriesDataset.__getitem__` was patched (via a
class-attribute-guarded idempotent monkey-patch, safe to re-run) to also
compute native-resolution HU per slice (`pixel_array * RescaleSlope +
RescaleIntercept`, BEFORE resize -- never on the resized image, since an
earlier attempt failed exactly from downsampling blurring the HU signal).
For each connected component surviving the spatial filter, its low-resolution
mask was upsampled to native resolution via **nearest-neighbor**
interpolation (the only method that introduces no fabricated intermediate
values), and median HU was measured under that mask directly from the
native-resolution image.

**Result (all 8 candidate series, measured from real inference):**

| series | volume (mL) | median HU |
|---|---|---|
| 2722 | 6.55 | 57.0 |
| 2913 | 9.19 + 4.48 (2 components) | 47.0 / 45.0 |
| 2023 | 14.95 | 50.0 |
| 3180 | 14.79 | 61.0 |
| 2575 | 9.08 | 45.0 |
| 2557, 270872, 2388 | 0 (no component survived the spatial filter) | — |

**Conclusion: the calcification hypothesis is rejected by direct
measurement.** Every measured component's HU (45-61) sits squarely in the
acute-blood range, nowhere near calcification's >100 HU threshold -- these
false positives are **not** distinguishable from real hemorrhage by HU.
No HU-based filter can fix this; the code works correctly (verified),
but there is nothing there to filter.

**Two remaining hypotheses, neither pursued further in this pass:**
1. These may be a genuine shape/texture-based segmentation error (a
   normal vascular or anatomical structure with blood-range HU,
   misclassified by shape) -- would need per-series visual review, not
   an automated physics-based filter.
2. Given only ~69% of slices have real pixel annotation (Section 14.2),
   ground truth itself may be incomplete for these specific series --
   these could be genuine, unlabeled small hemorrhages rather than false
   positives at all, which would mean "fixing" them would be wrong.

**Decision: do not pursue an HU-based fix further.** Documented here as a
negative result -- a real hypothesis, properly tested against measured
data, and correctly abandoned when the data didn't support it, rather than
force-fit a threshold to match wanted behavior.
